# ARES-X · Rover autónomo de exploración planetaria
**IA híbrida: RNA + lógica difusa + algoritmo genético, con supervisor de seguridad.**

Simulación ficticia con parámetros astronómicos de referencia. Todo valor operacional es un *supuesto de simulación*.

**Cómo usar:** `Entorno de ejecución → Ejecutar todas`. CPU gratuita, sin GPU ni claves. El panel de control (sección 5)
ocupa una sola pantalla; cada botón ejecuta el motor Python de este cuaderno.

Flujo por paso: sensores con ruido → preprocesamiento → RNA (probabilidades) → difuso (velocidad/cautela) →
AG (rutas, solo ante objetivo/bloqueo/evento grave) → supervisor de seguridad → acción → física → bitácora.

In [ ]:
# 0. Dependencias (todas gratuitas y preinstaladas en Colab)
import importlib, subprocess, sys
for pkg in ('numpy', 'pandas', 'sklearn', 'matplotlib'):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn' if pkg == 'sklearn' else pkg], check=True)
import os
for folder in ('ares_x/src/ares', 'ares_x/web', 'ares_x/tests'):
    os.makedirs(folder, exist_ok=True)
print('Python', sys.version.split()[0])

## 1. Código modular
Clases `PlanetProfile`, `World`, `SensorSuite`, `NeuralPerception`, `FuzzyController`, `GeneticPlanner`, `SafetySupervisor`, `MissionLogger`, `HybridMission` y `Dashboard`, escritas como módulos dentro del cuaderno.

In [ ]:
%%writefile ares_x/src/ares/__init__.py
"""ARES-X deterministic physical simulation, independent of AI modules."""
from .planet import PlanetProfile
from .world import World
from .sensors import SensorSuite
from .events import Event, EVENTS
from .logger import MissionLogger
from .state_machine import Mission, MissionState, StateMachine

__all__ = ['PlanetProfile', 'World', 'SensorSuite', 'Event', 'EVENTS', 'MissionLogger', 'Mission', 'MissionState', 'StateMachine']


In [ ]:
%%writefile ares_x/src/ares/planet.py
"""Planetary reference profiles and explicit vehicle compatibility."""
from dataclasses import dataclass
import unicodedata

ASSUMPTION = 'supuesto de simulación'


def normalize(value):
    return ''.join(c for c in unicodedata.normalize('NFD', value.lower()) if unicodedata.category(c) != 'Mn')


@dataclass(frozen=True)
class PlanetProfile:
    name: str
    rocky: bool
    gravity: float
    temperature: float
    pressure: float
    oxygen: float
    hazards: tuple
    mode: str = 'rover'
    operational_label: str = ASSUMPTION

    @classmethod
    def for_body(cls, name='Marte', mode='rover'):
        key = normalize(name)
        if key not in PROFILES:
            raise ValueError('Planeta desconocido')
        mode_key = normalize(mode)
        modes = {'rover': 'rover', 'ruedas': 'rover', 'sonda': 'probe', 'probe': 'probe'}
        if mode_key not in modes:
            raise ValueError('Modo inválido: use rover o sonda')
        profile = cls(*PROFILES[key], mode=modes[mode_key])
        profile.validate()
        return profile

    def validate(self):
        if self.mode not in ('rover', 'probe'):
            raise ValueError('Modo inválido: use rover o sonda')
        if not self.rocky and self.mode == 'rover':
            raise ValueError(f'{self.name} no tiene suelo sólido accesible y estable para ruedas; seleccione modo sonda atmosférica virtual.')
        return self


PROFILES = {
    'mercurio': ('Mercurio', True, 3.7, 167., 0., 0., ('radiación', 'contraste térmico')),
    'venus': ('Venus', True, 8.87, 464., 9200000., 0., ('corrosión', 'presión', 'temperatura')),
    'tierra': ('Tierra', True, 9.81, 15., 101325., 20.95, ('sismo', 'lluvia')),
    'marte': ('Marte', True, 3.71, -63., 610., .13, ('polvo', 'barranco', 'noche')),
    'jupiter': ('Júpiter', False, 24.79, -110., 100000., 0., ('turbulencia', 'presión')),
    'saturno': ('Saturno', False, 10.44, -140., 100000., 0., ('viento', 'enlace')),
    'urano': ('Urano', False, 8.69, -195., 100000., 0., ('frío', 'energía')),
    'neptuno': ('Neptuno', False, 11.15, -200., 100000., 0., ('viento', 'latencia')),
}


In [ ]:
%%writefile ares_x/src/ares/world.py
"""Seeded two-dimensional physical terrain; coordinates are (row, column)."""
import numpy as np
from .planet import ASSUMPTION


class World:
    operational_label = ASSUMPTION

    def __init__(self, width=16, height=12, seed=0, difficulty='medium', science_count=5):
        if not all(isinstance(v, int) and not isinstance(v, bool) and v > 0 for v in (width, height)):
            raise ValueError('Las dimensiones deben ser enteros positivos')
        if difficulty not in ('low', 'medium', 'high'):
            raise ValueError('Dificultad inválida: low, medium o high')
        if not isinstance(science_count, int) or science_count < 0:
            raise ValueError('Cantidad de sitios inválida')
        self.width, self.height, self.seed, self.difficulty = width, height, seed, difficulty
        self.rng = np.random.default_rng(seed)
        self.elevation = self.rng.uniform(0, 3, (height, width))
        self.elevation[0, 0] = 0
        cells = [(r, c) for r in range(height) for c in range(width) if (r, c) != (0, 0)]
        self.rng.shuffle(cells)
        count = int(len(cells) * {'low': .05, 'medium': .12, 'high': .22}[difficulty])
        self.obstacles = set(cells[:count])
        free = cells[count:]
        self.science_sites = set(free[:science_count])
        self.resources = set(free[science_count:science_count + 3])
        self.refuges = {(0, 0)}
        self.cliffs = set()
        self.visited_science = set()

    def in_bounds(self, position):
        try:
            r, c = position
            return isinstance(r, (int, np.integer)) and isinstance(c, (int, np.integer)) and 0 <= r < self.height and 0 <= c < self.width
        except (TypeError, ValueError):
            return False

    def add_obstacle(self, position):
        if not self.in_bounds(position):
            raise ValueError('Obstáculo fuera del mapa')
        self.obstacles.add(tuple(position))

    def traversable(self, position, mode='rover'):
        return self.in_bounds(position) and tuple(position) not in self.obstacles and (mode == 'probe' or tuple(position) not in self.cliffs)

    @property
    def science_progress(self):
        return len(self.visited_science & self.science_sites) / len(self.science_sites) if self.science_sites else 1.0

    def refuge_distance(self, position, mode='rover'):
        """Shortest currently feasible return distance, without promising rescue."""
        from collections import deque
        queue = deque([(tuple(position), 0)])
        seen = {tuple(position)}
        while queue:
            cell, distance = queue.popleft()
            if cell in self.refuges:
                return distance
            for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                neighbor = (cell[0] + dr, cell[1] + dc)
                if neighbor not in seen and self.traversable(neighbor, mode):
                    seen.add(neighbor)
                    queue.append((neighbor, distance + 1))
        return float('inf')


In [ ]:
%%writefile ares_x/src/ares/sensors.py
"""Imperfect measurements, with explicit critical-reading validation."""
import math
import numpy as np
from .planet import ASSUMPTION


class SensorSuite:
    operational_label = ASSUMPTION
    ranges = {'distance': (0, 20), 'slope': (0, 45), 'battery': (0, 100),
              'elevation_delta': (-1e6, 1e6), 'temperature_external': (-273.15, 1e4)}

    def __init__(self, seed=0, noise=.05):
        if not math.isfinite(noise) or noise < 0:
            raise ValueError('Ruido inválido')
        self.rng = np.random.default_rng(seed)
        self.noise = noise
        self.failures = {}
        self.link_available = True

    @staticmethod
    def validate(readings):
        for key in ('distance',):
            value = readings.get(key)
            if not isinstance(value, (int, float, np.number)) or not math.isfinite(value) or not 0 < value <= 20:
                return False
        for key in ('elevation_delta', 'temperature_external', 'temperature_internal'):
            if key in readings:
                value = readings[key]
                if not isinstance(value, (int, float, np.number)) or not math.isfinite(value):
                    return False
        for key, bounds in SensorSuite.ranges.items():
            if key in readings:
                value = readings[key]
                if not isinstance(value, (int, float, np.number)) or not math.isfinite(value) or not bounds[0] <= value <= bounds[1]:
                    return False
        return True

    def fail(self, sensor='distance', failure='nan'):
        if failure not in ('nan', 'saturation', 'missing'):
            raise ValueError('Tipo de falla desconocido')
        self.failures[sensor] = failure

    def observe(self, world, position, target, battery, profile, daylight=True, visibility=1.):
        valid = world.in_bounds(target)
        delta = float(world.elevation[target] - world.elevation[position]) if valid else 0.
        blocked = not world.traversable(target, profile.mode)
        readings = {
            'distance': 0. if blocked else float(np.clip(10 + self.rng.normal(0, self.noise), .001, 20)),
            'elevation_delta': delta,
            'cliff': valid and (tuple(target) in world.cliffs or delta < -5),
            'slope': float(min(45, abs(delta) * 3)),
            'battery': battery,
            'temperature_external': profile.temperature + float(self.rng.normal(0, self.noise)),
            'temperature_internal': 20., 'radiation': .1, 'pressure': profile.pressure,
            'oxygen_ambient': profile.oxygen, 'oxygen_stored': 100., 'co2': 95. if profile.name == 'Marte' else .04,
            'so2': 0., 'visibility': visibility, 'particles': 1 - visibility,
            'solar_power': 10. * visibility if daylight else 0., 'position': tuple(position),
            'heading': 0., 'vibration': 0., 'texture': float(self.rng.random()),
            'link_available': self.link_available,
        }
        for key, failure in self.failures.items():
            readings[key] = {'nan': float('nan'), 'saturation': float('inf'), 'missing': None}[failure]
        return readings


In [ ]:
%%writefile ares_x/src/ares/events.py
"""Explicit, ordered event injection; narrative signals are not scientific proof."""
from dataclasses import dataclass, field
from .planet import normalize

EVENTS = ('barranco', 'bateria_baja', 'dia', 'noche', 'sismo', 'meteorito', 'tormenta_polvo', 'corrosion', 'biosenal_anomala', 'alienigena', 'falla_sensor', 'perdida_enlace', 'viento', 'ola_calor')
ALIASES = {'low_battery': 'bateria_baja', 'day': 'dia', 'night': 'noche', 'quake': 'sismo', 'dust_storm': 'tormenta_polvo', 'sensor_failure': 'falla_sensor', 'link_loss': 'perdida_enlace', 'wind': 'viento', 'heat_wave': 'ola_calor'}


@dataclass
class Event:
    kind: str
    data: dict = field(default_factory=dict)

    def __post_init__(self):
        self.kind = normalize(self.kind).replace(' ', '_')
        self.kind = ALIASES.get(self.kind, self.kind)
        if self.kind not in EVENTS:
            raise ValueError('Evento desconocido')

    def apply(self, mission):
        world = mission.world
        position = tuple(self.data.get('position', mission.position))
        if self.kind in ('barranco', 'meteorito', 'alienigena'):
            if not world.in_bounds(position):
                raise ValueError('Evento fuera del mapa')
            if self.kind == 'barranco':
                world.cliffs.add(position)
                world.elevation[position] -= 20
            else:
                world.add_obstacle(position)
        elif self.kind == 'bateria_baja':
            mission.battery = min(mission.battery, 10.)
        elif self.kind in ('dia', 'noche'):
            mission.daylight = self.kind == 'dia'
        elif self.kind == 'sismo':
            mission.wait_steps = max(mission.wait_steps, 1)
        elif self.kind == 'tormenta_polvo':
            mission.visibility = .2
        elif self.kind == 'corrosion':
            mission.health = max(0., mission.health - 20.)
        elif self.kind == 'falla_sensor':
            mission.sensors.fail(self.data.get('sensor', 'distance'), self.data.get('failure', 'nan'))
        elif self.kind == 'perdida_enlace':
            mission.sensors.link_available = False
        elif self.kind == 'viento' and mission.profile.mode == 'probe':
            mission.wait_steps = max(mission.wait_steps, 1)
        return {'kind': self.kind, 'message': {
            'biosenal_anomala': 'Bioseñal anómala simulada; no confirma vida.',
            'alienigena': 'Evento narrativo completamente ficticio; evitar objeto.',
            'viento': 'Ráfaga de viento simulada: mayor consumo durante 5 pasos.',
            'ola_calor': 'Ola de calor simulada: +40 °C exterior, +35 °C en componentes y +20 % de consumo durante 12 pasos.',
        }.get(self.kind, 'Evento aplicado: ' + self.kind)}


In [ ]:
%%writefile ares_x/src/ares/logger.py
"""Independent step snapshots with portable JSON-valued CSV columns."""
import copy
import csv
import io
import json
import math
from pathlib import Path


def clean(value):
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, dict):
        return {str(k): clean(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [clean(v) for v in value]
    return value


class MissionLogger:
    def __init__(self):
        self.trace = []

    def record(self, **record):
        snapshot = clean(copy.deepcopy(record))
        self.trace.append(snapshot)
        return copy.deepcopy(snapshot)

    def export_csv(self, path=None):
        output = io.StringIO(newline='')
        fields = list(dict.fromkeys(key for row in self.trace for key in row))
        writer = csv.DictWriter(output, fieldnames=fields)
        if fields:
            writer.writeheader()
            for row in self.trace:
                writer.writerow({k: json.dumps(v, ensure_ascii=False, sort_keys=True) if isinstance(v, (list, dict)) else v for k, v in row.items()})
        result = output.getvalue()
        if path is not None:
            Path(path).write_text(result, encoding='utf-8')
        return result


In [ ]:
%%writefile ares_x/src/ares/state_machine.py
"""Physical-only mission loop. No AI policy is implied by this controller."""
from collections import deque
from enum import Enum
import math
from .planet import PlanetProfile, ASSUMPTION
from .sensors import SensorSuite
from .logger import MissionLogger
from .events import Event


class MissionState(str, Enum):
    CONFIGURACION = 'CONFIGURACION'
    EXPLORACION = 'EXPLORACION'
    ALERTA = 'ALERTA'
    REPLANIFICACION = 'REPLANIFICACION'
    REFUGIO = 'REFUGIO'
    MODO_SEGURO = 'MODO_SEGURO'
    EXITO = 'EXITO'
    FRACASO = 'FRACASO'


class Mission:
    operational_label = ASSUMPTION

    def __init__(self, world, profile=None, battery=100., step_duration=1., sensors=None, logger=None, thermal_limit=500.):
        if not math.isfinite(battery) or not 0 <= battery <= 100:
            raise ValueError('Batería inválida: debe estar entre 0 y 100')
        if not math.isfinite(step_duration) or step_duration <= 0:
            raise ValueError('Duración de paso inválida')
        if not math.isfinite(thermal_limit) or thermal_limit <= -273.15:
            raise ValueError('Límite térmico inválido')
        self.thermal_limit = thermal_limit
        self.world = world
        self.profile = (profile or PlanetProfile.for_body()).validate()
        self.battery, self.step_duration = float(battery), float(step_duration)
        self.sensors = sensors or SensorSuite(world.seed)
        self.logger = logger or MissionLogger()
        self.position = (0, 0)
        self.state = MissionState.CONFIGURACION
        self.health = 100.
        self.daylight, self.visibility = True, 1.
        self.wait_steps = 0
        self.elapsed = 0.
        self.pending_events = []
        self.paused = False
        self.path = [self.position]

    @property
    def science_progress(self):
        return self.world.science_progress

    def refuge_distance(self, start):
        """Physical reachability only; this is not an AI route planner."""
        queue = deque([(start, 0)])
        seen = {start}
        while queue:
            position, distance = queue.popleft()
            if position in self.world.refuges:
                return distance
            for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):
                neighbor = (position[0] + dr, position[1] + dc)
                if neighbor in seen or not self.world.traversable(neighbor, self.profile.mode):
                    continue
                if self.profile.mode == 'rover' and self.world.elevation[neighbor] - self.world.elevation[position] < -5:
                    continue
                seen.add(neighbor)
                queue.append((neighbor, distance + 1))
        return math.inf

    def inject_event(self, kind, **data):
        event = Event(kind, data)
        self.pending_events.append(event.apply(self))

    def step(self, target=None):
        if self.paused or self.state in (MissionState.EXITO, MissionState.FRACASO):
            return None
        target = self.position if target is None else tuple(target)
        readings = self.sensors.observe(self.world, self.position, target, self.battery, self.profile, self.daylight, self.visibility)
        action, reason = 'detener', 'ESPERA'
        if self.battery <= 0 or self.health <= 0:
            self.state, reason = MissionState.FRACASO, 'RECURSOS_AGOTADOS'
        elif readings['temperature_external'] is not None and math.isfinite(readings['temperature_external']) and readings['temperature_external'] > self.thermal_limit:
            self.state, reason = MissionState.MODO_SEGURO, 'LIMITE_TERMICO'
        elif self.health < 30:
            self.state, reason = MissionState.MODO_SEGURO, 'DANO_CRITICO'
        elif readings['cliff'] and self.profile.mode == 'rover':
            if self.world.in_bounds(target):
                self.world.cliffs.add(target)
            self.state, reason = MissionState.REPLANIFICACION, 'BARRANCO_CONFIRMADO'
        elif not self.world.traversable(target, self.profile.mode):
            self.state, reason = MissionState.REPLANIFICACION, 'RUTA_BLOQUEADA'
        elif not self.sensors.validate(readings):
            self.state, reason = MissionState.MODO_SEGURO, 'SENSOR_CRITICO_INVALIDO'
        elif self.wait_steps:
            self.wait_steps -= 1
            self.state, reason = MissionState.ALERTA, 'ESPERAR_REMEDIR'
        elif abs(target[0] - self.position[0]) + abs(target[1] - self.position[1]) > 1:
            self.state, reason = MissionState.REPLANIFICACION, 'MOVIMIENTO_NO_CONTIGUO'
        elif target != self.position:
            cost = self.step_duration * (1 + (1-self.visibility))
            return_distance = self.refuge_distance(self.position)
            target_distance = self.refuge_distance(target)
            if cost > self.battery:
                self.state, reason = MissionState.MODO_SEGURO, 'ENERGIA_INSUFICIENTE'
            elif self.battery <= 10 and self.position not in self.world.refuges and (return_distance * cost > self.battery or target_distance >= return_distance):
                self.state, reason = MissionState.MODO_SEGURO, 'REFUGIO_INALCANZABLE'
                if return_distance * cost <= self.battery:
                    self.state, reason = MissionState.REFUGIO, 'RECURSOS_CRITICOS'
            else:
                self.battery = max(0., self.battery - cost)
                self.position = target
                self.path.append(target)
                self.world.visited_science.update({target} & self.world.science_sites)
                self.state, action, reason = MissionState.EXPLORACION, 'avanzar', 'MOVIMIENTO_VALIDO'
                if self.battery == 0:
                    self.state, reason = MissionState.FRACASO, 'RECURSOS_AGOTADOS'
        else:
            self.state = MissionState.EXPLORACION
        if self.science_progress == 1 and self.position in self.world.refuges and self.state == MissionState.EXPLORACION:
            self.state, reason = MissionState.EXITO, 'OBJETIVOS_Y_RETORNO_COMPLETADOS'
        self.elapsed += self.step_duration
        events, self.pending_events = self.pending_events, []
        return self.logger.record(step=len(self.logger.trace), elapsed=self.elapsed, state=self.state.value,
                                  position=self.position, battery=self.battery, health=self.health,
                                  sensors=readings, events=events, action=action, reason=reason,
                                  science_progress=self.science_progress, seed=self.world.seed,
                                  operational_label=ASSUMPTION)


StateMachine = Mission


In [ ]:
%%writefile ares_x/src/ares/neural.py
"""Inspectable CPU perception trained on noisy, map-separated synthetic data.

Ground truth is a simulator property: cliff membership or a physical drop below
-5 m takes precedence over obstacle membership. Neither membership nor the
deterministic edge flag is passed to the MLP. Observations are noisy proxies;
confidence is not a safety guarantee and never replaces the edge-sensor veto.
"""
from collections.abc import Mapping
import warnings

import numpy as np
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

from .planet import ASSUMPTION, PlanetProfile
from .sensors import SensorSuite
from .world import World


FEATURES = ('distance', 'slope', 'elevation_delta', 'texture', 'visibility', 'vibration')
LABELS = ('libre', 'obstaculo', 'barranco')
BOUNDS = ((.000001, 20.), (0., 45.), (-1e6, 1e6), (0., 1.), (0., 1.), (0., 20.))


def _integer(value, minimum, message):
    if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < minimum:
        raise ValueError(message)
    return int(value)


def _matrix(values):
    try:
        array = np.asarray(values, dtype=float)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError('Entrada de sensores inválida: se requieren números finitos') from error
    if array.ndim != 2 or array.shape[1] != len(FEATURES) or not len(array) or not np.isfinite(array).all():
        raise ValueError('Entrada de sensores inválida: se requieren seis valores finitos')
    lower, upper = np.asarray(BOUNDS).T
    if np.any(array < lower) or np.any(array > upper):
        raise ValueError('Entrada de sensores fuera de rango físico; distancia cero requiere frenado')
    return array


def generate_training_data(seed=0, map_count=18, samples_per_map=120):
    """Generate map-grouped observations with simulator-owned terrain labels.

    Distance, slope, elevation and texture include observation noise and map
    biases. Cliffs are added as synthetic simulator terrain (not sensor labels).
    Distances below sensor resolution are represented as a small positive
    return here; a literal zero from the live sensor is separately invalid.
    """
    seed = _integer(seed, 0, 'Semilla inválida')
    map_count = _integer(map_count, 5, 'Se requieren al menos cinco mapas')
    samples_per_map = _integer(samples_per_map, 10, 'Se requieren al menos diez muestras por mapa')
    rng = np.random.default_rng(seed)
    profile = PlanetProfile.for_body('Marte')
    rows, labels, groups = [], [], []
    for group in range(map_count):
        map_seed = int(rng.integers(0, 2**31 - 1))
        world = World(width=12, height=10, seed=map_seed, difficulty=('low', 'medium', 'high')[group % 3])
        sensors = SensorSuite(seed=map_seed, noise=.12)
        cells = [(r, c) for r in range(world.height) for c in range(world.width) if (r, c) != (0, 0)]
        cliff_indices = rng.choice(len(cells), size=12, replace=False)
        world.cliffs.update(cells[index] for index in cliff_indices)
        for cell in world.cliffs:
            world.elevation[cell] = rng.uniform(-12, -7)
        edges = [(position, (position[0] + dr, position[1] + dc))
                 for position in cells for dr, dc in ((0, 1), (0, -1), (1, 0), (-1, 0))
                 if world.in_bounds((position[0] + dr, position[1] + dc))]
        distance_bias = rng.normal(0, .3)
        for index in rng.integers(0, len(edges), size=samples_per_map):
            position, target = edges[index]
            delta = float(world.elevation[target] - world.elevation[position])
            label = ('barranco' if target in world.cliffs or delta < -5 else
                     'obstaculo' if target in world.obstacles else 'libre')
            observation = sensors.observe(world, position, target, 80., profile,
                                          visibility=float(rng.uniform(.35, 1.)))
            observation['distance'] = float(np.clip(observation['distance'] + distance_bias + rng.normal(0, .8), .01, 20))
            observation['elevation_delta'] = delta + rng.normal(0, 1.3)
            observation['slope'] = float(np.clip(observation['slope'] + rng.normal(0, 2.), 0, 45))
            observation['texture'] = float(np.clip(.25 + .3 * (target in world.obstacles) + rng.normal(0, .22), 0, 1))
            observation['vibration'] = float(np.clip(abs(delta) * .15 + rng.normal(0, .15), 0, 20))
            rows.append([observation[name] for name in FEATURES])
            labels.append(label)
            groups.append(group)
    return {'X': np.asarray(rows), 'y': np.asarray(labels), 'groups': np.asarray(groups),
            'operational_label': ASSUMPTION}


def _calibration_error(truth, probabilities, predictions):
    """Ten-bin expected calibration error, measured only on the named split."""
    confidence = probabilities.max(axis=1)
    correct = truth == predictions
    bins = np.minimum((confidence * 10).astype(int), 9)
    return float(sum(np.mean(bins == index) * abs(correct[bins == index].mean() - confidence[bins == index].mean())
                     for index in range(10) if np.any(bins == index)))


class NeuralPerception:
    """MLP (32, 16), training-only scaling, and transparent held-out metrics.

    ``train`` returns validation/test metrics and split provenance. Validation
    is reserved for inspection, not mixed into fitting. ``predict`` accepts a
    sensor mapping or a row ordered by FEATURES; malformed critical input
    raises ValueError so the caller must brake/degrade rather than infer blind.
    """
    operational_label = ASSUMPTION

    def __init__(self, seed=0, confidence_threshold=.65):
        self.seed = _integer(seed, 0, 'Semilla inválida')
        try:
            threshold = float(confidence_threshold)
        except (TypeError, ValueError, OverflowError) as error:
            raise ValueError('Umbral de confianza inválido') from error
        if not np.isfinite(threshold) or not 0 < threshold <= 1:
            raise ValueError('Umbral de confianza inválido')
        self.confidence_threshold = threshold
        self.model = None
        self.scaler = None

    def train(self, dataset=None, map_count=18, samples_per_map=120, max_iter=200):
        max_iter = _integer(max_iter, 1, 'Cantidad de iteraciones inválida')
        data = generate_training_data(self.seed, map_count, samples_per_map) if dataset is None else dataset
        try:
            rows = _matrix(data['X'])
            truth = np.asarray(data['y'], dtype=str)
            groups = np.asarray(data['groups'])
        except (KeyError, TypeError, ValueError) as error:
            raise ValueError('Entrada de datos de entrenamiento inválida') from error
        if (truth.shape != (len(rows),) or groups.shape != (len(rows),)
                or set(truth) != set(LABELS) or groups.dtype.kind not in 'iu'):
            raise ValueError('Los datos requieren tres clases y grupos de mapa enteros')
        unique = np.unique(groups)
        if len(unique) < 5:
            raise ValueError('Se requieren al menos cinco mapas independientes')
        ordered = np.random.default_rng(self.seed).permutation(unique)
        train_end, valid_end = int(.6 * len(ordered)), int(.8 * len(ordered))
        map_splits = dict(zip(('train', 'validation', 'test'), np.split(ordered, (train_end, valid_end))))
        indices = {key: np.flatnonzero(np.isin(groups, maps)) for key, maps in map_splits.items()}
        if set(truth[indices['train']]) != set(LABELS):
            raise ValueError('Los mapas de entrenamiento deben contener las tres clases')
        scaler = StandardScaler().fit(rows[indices['train']])
        model = MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu',
                              early_stopping=False, random_state=self.seed,
                              max_iter=max_iter, learning_rate_init=.003)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            model.fit(scaler.transform(rows[indices['train']]), truth[indices['train']])
        majority = max(LABELS, key=lambda label: np.sum(truth[indices['train']] == label))
        report = {'operational_label': ASSUMPTION,
                  'split_maps': {key: maps.tolist() for key, maps in map_splits.items()},
                  'training_samples': len(indices['train']),
                  'converged': not any(issubclass(item.category, ConvergenceWarning) for item in caught),
                  'iterations': int(model.n_iter_), 'majority_label': majority,
                  'confidence_note': 'Probabilidad del modelo; no es garantía de seguridad'}
        for name in ('validation', 'test'):
            selection = indices[name]
            expected = truth[selection]
            probabilities = model.predict_proba(scaler.transform(rows[selection]))
            predicted = model.classes_[probabilities.argmax(axis=1)]
            report[name] = {
                'samples': len(selection), 'labels': list(LABELS),
                'accuracy': float(accuracy_score(expected, predicted)),
                'majority_baseline': float(np.mean(expected == majority)),
                'confusion_matrix': confusion_matrix(expected, predicted, labels=LABELS).tolist(),
                'macro_f1': float(f1_score(expected, predicted, labels=LABELS, average='macro', zero_division=0)),
                'barranco_recall': float(recall_score(expected, predicted, labels=['barranco'], average='macro', zero_division=0)),
                'calibration_error': _calibration_error(expected, probabilities, predicted),
            }
        # RF-11: per-iteration replay of the same training (same seed and data) to
        # expose loss and accuracy curves; partial_fit runs one epoch per call.
        probe = MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu', early_stopping=False,
                              random_state=self.seed, learning_rate_init=.003)
        train_x, train_y = scaler.transform(rows[indices['train']]), truth[indices['train']]
        valid_x, valid_y = scaler.transform(rows[indices['validation']]), truth[indices['validation']]
        history = []
        for iteration in range(1, int(model.n_iter_) + 1):
            probe.partial_fit(train_x, train_y, classes=model.classes_)
            history.append({'iteration': iteration, 'loss': float(probe.loss_),
                            'train_accuracy': float(accuracy_score(train_y, probe.predict(train_x))),
                            'validation_accuracy': float(accuracy_score(valid_y, probe.predict(valid_x)))})
        report['history'] = history
        report['loss_curve'] = [float(v) for v in model.loss_curve_]
        self.model, self.scaler = model, scaler
        self.split_indices_, self.report_ = indices, report
        return report

    def predict(self, readings):
        if self.model is None:
            raise ValueError('La red debe entrenarse antes de predecir')
        if isinstance(readings, Mapping):
            try:
                values = [readings[key] for key in FEATURES]
            except KeyError as error:
                raise ValueError(f'Entrada de sensor incompleta: falta {error.args[0]}') from error
        else:
            values = readings
        row = _matrix([values])
        normalized = self.scaler.transform(row)
        probabilities = dict(zip(self.model.classes_, self.model.predict_proba(normalized)[0].tolist()))
        probabilities = {label: float(probabilities[label]) for label in LABELS}
        activation = normalized
        summaries = []
        for weights, bias in zip(self.model.coefs_[:-1], self.model.intercepts_[:-1]):
            activation = np.maximum(0., activation @ weights + bias)
            summaries.append({'neurons': activation.shape[1], 'mean': float(activation.mean()),
                              'max': float(activation.max()), 'min': float(activation.min()),
                              'active_fraction': float(np.mean(activation > 0))})
        label = max(probabilities, key=probabilities.get)
        confidence = probabilities[label]
        return {'label': label, 'probabilities': probabilities, 'confidence': confidence,
                'low_confidence': confidence < self.confidence_threshold,
                'raw_input': row[0].tolist(), 'normalized_input': normalized[0].tolist(),
                'feature_names': list(FEATURES), 'activation_summaries': summaries,
                'risk': .7 * probabilities['obstaculo'] + probabilities['barranco'],
                'operational_label': ASSUMPTION}


In [ ]:
%%writefile ares_x/src/ares/fuzzy.py
"""Inspectable Mamdani controller; all operational thresholds are assumptions."""
import math

import numpy as np

from .planet import ASSUMPTION


def triangular(value, a, b, c):
    """Triangular membership, including degenerate shoulder endpoints."""
    if value < a or value > c:
        return 0.0
    if value == b:
        return 1.0
    return float((value - a) / (b - a) if value < b else (c - value) / (c - b))


def trapezoidal(value, a, b, c, d):
    """Trapezoidal membership with closed left/right shoulders."""
    if value < a or value > d:
        return 0.0
    if b <= value <= c:
        return 1.0
    return float((value - a) / (b - a) if value < b else (d - value) / (d - c))


class FuzzyController:
    """Speed is advisory, not a physical veto; H3 supplies hard safety limits.

    Invalid input or zero aggregate area produces speed 0 and caution 1.
    Temperature uses hypothetical hardware-relative thresholds, not planetary
    engineering limits. Min implements AND, max implements union of rules.
    """

    operational_label = ASSUMPTION
    membership_sets = {
        'distance': {'cerca': ('trap', 0, 0, 2, 6),
                     'media': ('tri', 2, 9, 15), 'lejos': ('trap', 10, 15, 20, 20)},
        'battery': {'critica': ('trap', 0, 0, 8, 18),
                    'baja': ('tri', 10, 30, 55), 'alta': ('trap', 40, 65, 100, 100)},
        'slope': {'plana': ('trap', 0, 0, 4, 12),
                  'media': ('tri', 5, 20, 32), 'fuerte': ('trap', 25, 35, 45, 45)},
        'temperature': {'fria': ('trap', -273.15, -273.15, -60, 0),
                        'templada': ('tri', -30, 20, 70),
                        'caliente': ('trap', 40, 90, 1000, 1000)},
    }
    output_sets = {
        'parar': ('trap', 0, 0, .01, .06),
        'lenta': ('tri', .03, .18, .35),
        'moderada': ('tri', .25, .5, .75),
        'rapida': ('trap', .6, .85, 1, 1),
    }
    # Each tuple is an AND antecedent and its speed consequent.
    rule_specs = (
        ('Barranco o riesgo cercano: parar.', (('distance', 'cerca'),), 'parar'),
        ('Batería crítica: avance lento.', (('battery', 'critica'),), 'lenta'),
        ('Pendiente fuerte y batería alta: avance lento.', (('slope', 'fuerte'), ('battery', 'alta')), 'lenta'),
        ('Pendiente media y distancia media: avance lento.', (('slope', 'media'), ('distance', 'media')), 'lenta'),
        ('Batería alta, terreno plano y riesgo lejano: rápido.', (('battery', 'alta'), ('slope', 'plana'), ('distance', 'lejos')), 'rapida'),
        ('Batería baja y distancia media: lento.', (('battery', 'baja'), ('distance', 'media')), 'lenta'),
        ('Distancia media y terreno plano: moderado.', (('distance', 'media'), ('slope', 'plana')), 'moderada'),
        ('Temperatura caliente: parar y evaluar protección.', (('temperature', 'caliente'),), 'parar'),
        ('Temperatura fría: lento para proteger equipos.', (('temperature', 'fria'),), 'lenta'),
        ('Batería alta, distancia media y temperatura templada: moderado.', (('battery', 'alta'), ('distance', 'media'), ('temperature', 'templada')), 'moderada'),
        ('Batería baja y riesgo lejano: moderado para conservar energía.', (('battery', 'baja'), ('distance', 'lejos')), 'moderada'),
        ('Riesgo lejano y pendiente media: moderado.', (('distance', 'lejos'), ('slope', 'media')), 'moderada'),
    )

    def __init__(self, rules=None):
        self.rules = [
            {'id': index + 1, 'description': description,
             'antecedents': antecedents, 'consequent': consequent}
            for index, (description, antecedents, consequent) in enumerate(self.rule_specs)
        ] if rules is None else list(rules)
        self.universe = np.linspace(0, 1, 201)

    @staticmethod
    def membership(value, specification):
        kind, *points = specification
        return triangular(value, *points) if kind == 'tri' else trapezoidal(value, *points)

    def infer(self, distance, battery, slope, temperature=20):
        values = dict(distance=distance, battery=battery, slope=slope, temperature=temperature)
        limits = {'distance': (0, 20), 'battery': (0, 100),
                  'slope': (0, 45), 'temperature': (-273.15, 1000)}
        aggregate = np.zeros_like(self.universe)
        result = {'speed': 0.0, 'caution': 1.0, 'centroid': 0.0,
                  'membership_degrees': {}, 'activated_rules': [],
                  'universe': self.universe.tolist(), 'aggregated_curve': aggregate.tolist(),
                  'reason': 'ENTRADA_INVALIDA', 'operational_label': ASSUMPTION}
        if any(not isinstance(value, (int, float, np.number)) or not math.isfinite(value)
               or not limits[name][0] <= value <= limits[name][1]
               for name, value in values.items()):
            return result
        degrees = {name: {label: self.membership(values[name], spec)
                          for label, spec in sets.items()}
                   for name, sets in self.membership_sets.items()}
        result['membership_degrees'] = degrees
        for rule in self.rules:
            strength = min(degrees[name][label] for name, label in rule['antecedents'])
            if strength > 0:
                curve = np.array([self.membership(x, self.output_sets[rule['consequent']])
                                  for x in self.universe])
                aggregate = np.maximum(aggregate, np.minimum(strength, curve))
                result['activated_rules'].append({**rule, 'strength': strength})
        integrate = getattr(np, 'trapezoid', None) or np.trapz  # numpy < 2 compatibility
        area = integrate(aggregate, self.universe)
        centroid = float(integrate(self.universe * aggregate, self.universe) / area) if area > 0 else 0.0
        result.update(speed=centroid, centroid=centroid, caution=1.0 - centroid,
                      aggregated_curve=aggregate.tolist(),
                      reason='INFERENCIA_MAMDANI' if area > 0 else 'SIN_REGLAS_ACTIVAS')
        return result


In [ ]:
%%writefile ares_x/src/ares/genetic.py
"""Seeded waypoint evolution over a known map, never a future-event oracle."""
from collections import deque
from math import isfinite
import random

from .planet import ASSUMPTION


class GeneticPlanner:
    """Science-visit ordering and maze feasibility with conservative safe return.

    All operational defaults and objective weights are simulation assumptions.
    Positive science competes with normalized distance, energy, risk and time.
    Every infeasible chromosome receives a penalty above any possible reward.
    """

    operational_label = ASSUMPTION
    default_weights = dict(science=3., distance=1., energy=1., predicted_risk=2., time=1.)

    def __init__(self, population_size=30, generations=30, waypoints=4, seed=0,
                 mutation_rate=.25, crossover_rate=.85, elitism=2,
                 tournament_size=3, weights=None, early_stop=True):
        for name, value, minimum in [('population_size', population_size, 30),
                                     ('generations', generations, 30),
                                     ('waypoints', waypoints, 1),
                                     ('elitism', elitism, 1),
                                     ('tournament_size', tournament_size, 2)]:
            if not isinstance(value, int) or isinstance(value, bool) or value < minimum:
                raise ValueError(f'{name}: valor inválido; mínimo {minimum}')
        if elitism >= population_size or tournament_size > population_size:
            raise ValueError('Elitismo o torneo incompatible con la población')
        if any(not isinstance(v, (int, float)) or not isfinite(v) or not 0 <= v <= 1
               for v in (mutation_rate, crossover_rate)):
            raise ValueError('Las tasas deben estar entre 0 y 1')
        self.weights = self.default_weights.copy()
        if weights is not None:
            if set(weights) - set(self.weights):
                raise ValueError('Componente de fitness desconocido')
            self.weights.update(weights)
        if any(not isinstance(v, (int, float)) or not isfinite(v) or v < 0
               for v in self.weights.values()) or not any(self.weights.values()):
            raise ValueError('Pesos inválidos: deben ser finitos, no negativos y no todos cero')
        self.population_size, self.generations, self.waypoints = population_size, generations, waypoints
        self.seed, self.mutation_rate, self.crossover_rate = seed, mutation_rate, crossover_rate
        self.elitism, self.tournament_size, self.early_stop = elitism, tournament_size, early_stop
        self.rng = random.Random(seed)
        self.history = []

    @staticmethod
    def _free(world):
        return [(r, c) for r in range(world.height) for c in range(world.width)
                if world.traversable((r, c))]

    def repair(self, chromosome, world):
        """Clamp invalid coordinates and replace blocked genes with nearest free cells."""
        free = self._free(world)
        if not free:
            return [(0, 0)] * self.waypoints
        genes = list(chromosome)[:self.waypoints]
        genes += [free[0]] * (self.waypoints - len(genes))
        repaired = []
        for gene in genes:
            try:
                row, col = gene
                row = min(world.height - 1, max(0, round(row))) if isfinite(row) else 0
                col = min(world.width - 1, max(0, round(col))) if isfinite(col) else 0
            except (TypeError, ValueError, OverflowError):
                row, col = free[0]
            point = (row, col)
            if not world.traversable(point):
                point = min(free, key=lambda p: (abs(p[0] - row) + abs(p[1] - col), p))
            repaired.append(point)
        return repaired

    def mutate(self, chromosome, world):
        """Mutate positions or scientific visit order, then repair every gene."""
        genes = self.repair(chromosome, world)
        free = self._free(world)
        if free:
            for i in range(self.waypoints):
                if self.rng.random() < self.mutation_rate:
                    genes[i] = self.rng.choice(free)
            if self.waypoints > 1 and self.rng.random() < self.mutation_rate:
                a, b = self.rng.sample(range(self.waypoints), 2)
                genes[a], genes[b] = genes[b], genes[a]
        return self.repair(genes, world)

    @staticmethod
    def _path(world, start, goal):
        if not world.traversable(start) or not world.traversable(goal):
            return None
        queue, previous = deque([start]), {start: None}
        while queue:
            cell = queue.popleft()
            if cell == goal:
                route = []
                while cell is not None:
                    route.append(cell)
                    cell = previous[cell]
                return route[::-1]
            for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):
                neighbor = (cell[0] + dr, cell[1] + dc)
                if (neighbor not in previous and world.traversable(neighbor)
                        and world.elevation[neighbor] - world.elevation[cell] >= -5):
                    # Match the deterministic cliff rule used by SensorSuite.
                    previous[neighbor] = cell
                    queue.append(neighbor)
        return None

    def _return_distance(self, world, position):
        """Return paths obey the same directed cliff edges as outbound travel."""
        distances = []
        for refuge in sorted(world.refuges):
            path = self._path(world, position, refuge)
            if path is not None:
                distances.append(len(path) - 1)
        return min(distances, default=float('inf'))

    @staticmethod
    def _risk_at(estimated_risk, cell):
        if estimated_risk is None:
            return 0.
        try:
            value = estimated_risk.get(cell, 0.) if hasattr(estimated_risk, 'get') else estimated_risk[cell]
            value = float(value)
        except (TypeError, ValueError, IndexError, KeyError):
            raise ValueError('Mapa de riesgo estimado inválido') from None
        if not isfinite(value) or not 0 <= value <= 1:
            raise ValueError('Riesgo estimado fuera de [0, 1]')
        return value

    def evaluate(self, chromosome, world, start, goal, battery=100, estimated_risk=None,
                 temperature=20, thermal_limit=80, energy_per_step=1, time_per_step=1):
        """Decode a chromosome; normalized costs use fixed map-wide scales.

        Battery reserves cover the route AND a shortest known return from its end.
        Thermal feasibility is a supplied current hardware estimate, not weather.
        """
        numeric = (battery, temperature, thermal_limit, energy_per_step, time_per_step)
        if any(not isinstance(v, (int, float)) or not isfinite(v) for v in numeric):
            raise ValueError('Parámetros de planificación no finitos')
        if battery < 0 or energy_per_step <= 0 or time_per_step <= 0:
            raise ValueError('Batería o costo de paso inválido')
        genes = self.repair(chromosome, world)
        repaired = list(chromosome) != genes
        violations, route = [], []
        if not world.in_bounds(start) or not world.in_bounds(goal):
            violations.append('bounds')
        elif not world.traversable(start) or not world.traversable(goal):
            violations.append('collision_or_cliff')
        else:
            route = [tuple(start)]
            for index, point in enumerate(genes + [tuple(goal)]):
                segment = self._path(world, route[-1], point)
                if segment is None:
                    if index < self.waypoints:
                        # A disconnected optional waypoint cannot invalidate a reachable goal.
                        genes[index] = route[-1]
                        repaired = True
                        continue
                    violations.append('unreachable')
                    break
                route.extend(segment[1:])
        if temperature > thermal_limit:
            violations.append('thermal')
        distance = max(0, len(route) - 1)
        return_distance = self._return_distance(world, route[-1]) if route else float('inf')
        energy_required = (distance + return_distance) * energy_per_step
        if energy_required > battery or (distance > 0 and battery == 0):
            violations.append('battery')
        remaining_sites = world.science_sites - world.visited_science
        science_order = []
        for cell in route:
            if cell in remaining_sites and cell not in science_order:
                science_order.append(cell)
        max_distance = max(1, (world.width * world.height - 1) * (self.waypoints + 1))
        components = {
            'science': len(science_order) / len(remaining_sites) if remaining_sites else 0.,
            'distance': min(1., distance / max_distance),
            'energy': min(1., distance * energy_per_step / 100.),
            'predicted_risk': sum(self._risk_at(estimated_risk, p) for p in route) / max(1, len(route)),
            'time': min(1., distance * time_per_step / max_distance),
        }
        penalty = (1 + sum(self.weights.values())) * len(violations)
        fitness = self.weights['science'] * components['science'] - sum(
            self.weights[k] * v for k, v in components.items() if k != 'science') - penalty
        return dict(chromosome=genes, route=route, feasible=not violations, fitness=fitness,
                    components=components, penalty=penalty, violations=violations,
                    science_order=science_order, energy_required=energy_required,
                    distance=distance, repaired=repaired)

    def plan(self, world, start, goal, battery=100, estimated_risk=None, **kwargs):
        """Return at least three candidates plus inspectable evolutionary history.

        A planning run resets its seed for identical inputs/configuration. Early
        stopping can end before the configured >=30-generation budget, after ten
        stagnant generations; the global best feasible route is always retained.
        """
        self.rng = random.Random(self.seed)
        free = self._free(world)
        fallback = tuple(start) if world.in_bounds(start) else (0, 0)
        population = [[fallback] * self.waypoints]
        sites = sorted(world.science_sites - world.visited_science)
        for _ in range(self.population_size - 1):
            pool = sites if sites and self.rng.random() < .5 else free
            population.append([self.rng.choice(pool) if pool else fallback for _ in range(self.waypoints)])
        self.history, archive = [], {}
        stale, best_score, best_feasible = 0, float('-inf'), None
        counters = dict(tournaments=0, crossovers=0, mutations=0)
        for generation in range(self.generations + 1):
            scored = [self.evaluate(g, world, start, goal, battery, estimated_risk, **kwargs) for g in population]
            scored.sort(key=lambda c: (c['feasible'], c['fitness']), reverse=True)
            for candidate in scored:
                key = tuple(candidate['route'])
                if key not in archive or candidate['fitness'] > archive[key]['fitness']:
                    archive[key] = candidate
                if candidate['feasible'] and (best_feasible is None or candidate['fitness'] > best_feasible['fitness']):
                    best_feasible = candidate
            score = scored[0]['fitness']
            stale = 0 if score > best_score else stale + 1
            best_score = max(best_score, score)
            self.history.append(dict(generation=generation, chromosomes=[c['chromosome'][:] for c in scored],
                                     best_chromosome=scored[0]['chromosome'][:], best_fitness=score,
                                     best_feasible_fitness=best_feasible['fitness'] if best_feasible else None,
                                     stale_generations=stale, **counters))
            if generation == self.generations or (self.early_stop and stale >= 10):
                break
            population = [c['chromosome'][:] for c in scored[:self.elitism]]
            counters = dict(tournaments=0, crossovers=0, mutations=0)
            while len(population) < self.population_size:
                parents = []
                for _ in range(2):
                    contestants = self.rng.sample(scored, self.tournament_size)
                    parents.append(max(contestants, key=lambda c: (c['feasible'], c['fitness']))['chromosome'])
                    counters['tournaments'] += 1
                child = parents[0][:]
                if self.waypoints > 1 and self.rng.random() < self.crossover_rate:
                    split = self.rng.randrange(1, self.waypoints)
                    child = parents[0][:split] + parents[1][split:]
                    counters['crossovers'] += 1
                mutated = self.mutate(child, world)
                counters['mutations'] += sum(a != b for a, b in zip(child, mutated))
                population.append(mutated)
        candidates = sorted(archive.values(), key=lambda c: (c['feasible'], c['fitness']), reverse=True)[:3]
        while len(candidates) < 3:
            candidates.append(scored[len(candidates) % len(scored)])
        feasible = best_feasible is not None
        return dict(route=best_feasible['route'] if feasible else [], feasible=feasible,
                    fitness=best_feasible['fitness'] if feasible else scored[0]['fitness'],
                    candidates=candidates, history=self.history, generations_run=generation,
                    stopped_early=generation < self.generations,
                    message='Ruta factible (supuesto de simulación)' if feasible else
                    'Sin solución: retornar solo si es factible; de lo contrario, esperar en modo seguro.')


def optimize_visit_order(leg, closing, seed=0, population_size=40, generations=60,
                         crossover_rate=.9, mutation_rate=.3, elitism=2, tournament_size=3):
    """Permutation GA (travelling-salesman family): best order to visit n stops.

    ``leg[i][j]`` is the cost from node i to node j, node 0 being the start and
    nodes 1..n the stops; ``closing[i]`` is the cost of ending the tour at node i
    (the safe return). Returns the order as stop indices 0..n-1, its cost and the
    best cost per generation. Seeded, so the same inputs give the same order.
    """
    stops = len(leg) - 1
    if stops <= 0:
        return dict(order=[], cost=0, history=[0])

    def cost(order):
        total, node = 0, 0
        for stop in order:
            total += leg[node][stop + 1]
            node = stop + 1
        return total + closing[node]

    rng = random.Random(seed)
    base = list(range(stops))
    nearest, node, left = [], 0, set(base)       # nearest-neighbour seed individual
    while left:
        nxt = min(left, key=lambda s: (leg[node][s + 1], s))
        nearest.append(nxt)
        left.discard(nxt)
        node = nxt + 1
    population = [nearest, base[:]]
    while len(population) < population_size:
        population.append(rng.sample(base, stops))
    if stops < 3:                                # every permutation fits in the population
        population = [nearest] + [list(p) for p in ([base, base[::-1]] if stops == 2 else [base])]
    history = []
    for _ in range(generations):
        population.sort(key=lambda o: (cost(o), o))
        history.append(cost(population[0]))
        if stops < 3:
            break
        children = [o[:] for o in population[:elitism]]
        while len(children) < population_size:
            a = min(rng.sample(population, tournament_size), key=cost)
            b = min(rng.sample(population, tournament_size), key=cost)
            child = a[:]
            if rng.random() < crossover_rate:    # order crossover (OX)
                i, j = sorted(rng.sample(range(stops + 1), 2))
                kept = a[i:j]
                rest = [s for s in b if s not in kept]
                child = rest[:i] + kept + rest[i:]
            if rng.random() < mutation_rate:     # swap mutation
                i, j = rng.sample(range(stops), 2)
                child[i], child[j] = child[j], child[i]
            if rng.random() < mutation_rate:     # segment reversal (2-opt move)
                i, j = sorted(rng.sample(range(stops + 1), 2))
                child[i:j] = child[i:j][::-1]
            children.append(child)
        population = children
    population.sort(key=lambda o: (cost(o), o))
    best = population[0]
    history.append(cost(best))
    return dict(order=best, cost=cost(best), history=history)


In [ ]:
%%writefile ares_x/src/ares/supervisor.py
"""Hybrid safety supervisor: physical prohibitions first, then AI proposals.

Every decision carries a reproducible reason code. Rules are evaluated in a
fixed priority order, so simultaneous events always resolve the same way.
"""
from dataclasses import dataclass, field

from .planet import ASSUMPTION

# Ordered by priority (lower index wins). Codes are stable for audits/tests.
REASONS = {
    'R00_RECURSOS_AGOTADOS': 'Batería o integridad agotadas: misión terminada.',
    'R01_LIMITE_TERMICO': 'Temperatura fuera de la tolerancia del hardware: se aísla el sistema.',
    'R02_SENSORES_NAVEGACION_FALLIDOS': 'Fallan todos los sensores de navegación: no se avanza a ciegas.',
    'R03_DANO_CRITICO': 'Integridad mecánica crítica: modo seguro.',
    'R04_VETO_BARRANCO_CONFIRMADO': 'El sensor de borde confirma un barranco en la siguiente celda: se detiene y replanifica.',
    'R05_RNA_PELIGRO': 'La RNA predice peligro en la siguiente celda con confianza suficiente: se detiene y replanifica.',
    'R06_REFUGIO_INALCANZABLE': 'La energía no alcanza para llegar a ningún refugio: se declara inalcanzable y se espera en modo seguro.',
    'R07_RETORNO_ENERGIA': 'La energía restante solo alcanza para volver: se prioriza el refugio.',
    'R08_ESPERA_SISMO': 'Sismo detectado: frenar, esperar y volver a medir.',
    'R09_RECARGA': 'En refugio con batería incompleta: se recarga.',
    'R10_SIN_RUTA': 'No existe ruta factible: se informa sin solución y se espera.',
    'R11_REPLANIFICAR': 'Ruta bloqueada o evento grave: se ejecuta el AG.',
    'R12_DIFUSO_PARAR': 'El controlador difuso recomienda velocidad casi nula: se detiene.',
    'R13_AVANCE_DEGRADADO': 'Sensor degradado o RNA con baja confianza: avance conservador fusionado con sensor determinista.',
    'R14_AVANCE': 'Ruta segura y sensores válidos: se avanza.',
    'R15_MISION_COMPLETA': 'Objetivos alcanzables completados y retorno seguro.',
    'R16_ESPERA_VIENTO': 'Ráfaga de viento en modo sonda: se mantiene la posición y se corrige la deriva.',
}
PRIORITY = list(REASONS)


@dataclass
class Decision:
    action: str            # avanzar | detener | recargar | volver | replanificar
    reason: str
    state: str
    speed_cap: float = 1.0
    vetoes: list = field(default_factory=list)
    permitted_by: list = field(default_factory=list)

    @property
    def explanation(self):
        return REASONS[self.reason]

    def to_dict(self):
        return {'action': self.action, 'reason': self.reason, 'state': self.state,
                'explanation': self.explanation, 'speed_cap': self.speed_cap,
                'vetoes': list(self.vetoes), 'permitted_by': list(self.permitted_by),
                'operational_label': ASSUMPTION}


class SafetySupervisor:
    """Arbitrates RNA, fuzzy and GA recommendations under hard constraints.

    ``context`` keys: battery, health (required); temperature_internal,
    thermal_limit, nav_valid (sensor -> bool), edge_cliff, rna (prediction or
    None), rna_used, fuzzy_speed, has_route, needs_replan, quake_wait,
    at_refuge, energy_to_refuge, recharge_target, returning, mission_complete.
    """

    operational_label = ASSUMPTION

    def __init__(self, confidence_threshold=.65, hazard_threshold=.5, return_margin=1.25,
                 stop_speed=.08, degraded_speed=.3):
        self.confidence_threshold = confidence_threshold
        self.hazard_threshold = hazard_threshold
        self.return_margin = return_margin
        self.stop_speed = stop_speed
        self.degraded_speed = degraded_speed

    def decide(self, ctx):
        vetoes, permits = [], []
        battery, health = ctx['battery'], ctx['health']
        if battery <= 0 or health <= 0:
            return Decision('detener', 'R00_RECURSOS_AGOTADOS', 'FRACASO')
        if ctx.get('temperature_internal', 20) > ctx.get('thermal_limit', 1e9):
            return Decision('detener', 'R01_LIMITE_TERMICO', 'MODO_SEGURO')
        nav = ctx.get('nav_valid', {})
        if nav and not any(nav.values()):
            return Decision('detener', 'R02_SENSORES_NAVEGACION_FALLIDOS', 'MODO_SEGURO')
        if health < 30:
            return Decision('detener', 'R03_DANO_CRITICO', 'MODO_SEGURO')
        if ctx.get('mission_complete'):
            return Decision('detener', 'R15_MISION_COMPLETA', 'EXITO')
        # Energy feasibility is a physical constraint and precedes AI proposals.
        energy_to_refuge = ctx.get('energy_to_refuge', 0.)
        if not ctx.get('at_refuge') and (energy_to_refuge == float('inf') or energy_to_refuge > battery):
            return Decision('detener', 'R06_REFUGIO_INALCANZABLE', 'MODO_SEGURO')
        if ctx.get('edge_cliff'):
            vetoes.append('Barranco confirmado por sensor de borde')
            return Decision('replanificar', 'R04_VETO_BARRANCO_CONFIRMADO', 'REPLANIFICACION', 0., vetoes)
        rna = ctx.get('rna') if ctx.get('rna_used', True) else None
        low_confidence = False
        if rna is not None:
            hazard = 1 - rna['probabilities']['libre']
            if rna['confidence'] >= self.confidence_threshold and rna['label'] != 'libre':
                vetoes.append(f"RNA: {rna['label']} p={rna['confidence']:.2f}")
                return Decision('replanificar', 'R05_RNA_PELIGRO', 'ALERTA', 0., vetoes)
            low_confidence = rna['low_confidence'] or hazard >= self.hazard_threshold
            permits.append(f"RNA: libre p={rna['probabilities']['libre']:.2f}")
        if ctx.get('quake_wait', 0) > 0:
            return Decision('detener', 'R08_ESPERA_SISMO', 'ALERTA', 0., vetoes, permits)
        if ctx.get('wind_wait', 0) > 0:
            return Decision('detener', 'R16_ESPERA_VIENTO', 'ALERTA', 0., vetoes, permits)
        if ctx.get('at_refuge') and battery < ctx.get('recharge_target', 0):
            return Decision('recargar', 'R09_RECARGA', 'REFUGIO', 0., vetoes, permits)
        if not ctx.get('at_refuge') and not ctx.get('returning') and energy_to_refuge * self.return_margin >= battery:
            permits.append('Energía justa para el retorno')
            return Decision('volver', 'R07_RETORNO_ENERGIA', 'REFUGIO', self.degraded_speed, vetoes, permits)
        if ctx.get('needs_replan'):
            return Decision('replanificar', 'R11_REPLANIFICAR', 'REPLANIFICACION', 0., vetoes, permits)
        if not ctx.get('has_route'):
            return Decision('detener', 'R10_SIN_RUTA', 'ALERTA', 0., vetoes, permits)
        speed = ctx.get('fuzzy_speed')
        if speed is not None and speed < self.stop_speed:
            return Decision('detener', 'R12_DIFUSO_PARAR', 'ALERTA', 0., vetoes, permits)
        if low_confidence or (nav and not all(nav.values())):
            permits.append('Avance limitado por degradación')
            return Decision('avanzar', 'R13_AVANCE_DEGRADADO', 'ALERTA', self.degraded_speed, vetoes, permits)
        state = 'REFUGIO' if ctx.get('returning') else 'EXPLORACION'
        return Decision('avanzar', 'R14_AVANCE', state, 1.0, vetoes, permits)

    def filter_routes(self, candidates, known_cliffs, known_obstacles):
        """Reject GA candidates crossing confirmed hazards; document each veto."""
        accepted, rejected = [], []
        for index, candidate in enumerate(candidates):
            route = candidate.get('route') or []
            bad = [cell for cell in route if cell in known_cliffs or cell in known_obstacles]
            if bad:
                rejected.append({'index': index, 'reason': 'cruza peligro confirmado', 'cells': [list(c) for c in bad[:3]]})
            elif not candidate.get('feasible'):
                rejected.append({'index': index, 'reason': 'infactible: ' + ', '.join(candidate.get('violations', [])), 'cells': []})
            elif not route:
                rejected.append({'index': index, 'reason': 'ruta vacía', 'cells': []})
            else:
                accepted.append(index)
        return accepted, rejected


In [ ]:
%%writefile ares_x/src/ares/mission.py
"""Hybrid mission loop.

observar -> validar_sensores -> predecir_riesgo_rna -> inferir_control_difuso ->
replanificar_ag_si_corresponde -> aplicar_restricciones -> ejecutar_accion ->
actualizar_mundo -> registrar_traza

The rover is not omniscient: it plans over a *belief* map that misses hidden
cliffs and part of the obstacles. Those are revealed through sensors (RNA
camera/LIDAR lookahead, deterministic edge sensor) or, worst case, by contact.
"""
from collections import deque
import math
import time

import numpy as np

from .events import Event
from .fuzzy import FuzzyController
from .genetic import GeneticPlanner, optimize_visit_order
from .logger import MissionLogger, clean
from .neural import NeuralPerception
from .planet import ASSUMPTION, PROFILES, PlanetProfile, normalize
from .sensors import SensorSuite
from .supervisor import SafetySupervisor
from .world import World

POLICIES = {
    'hibrido': dict(rna=True, fuzzy=True, ga=True, fixed_risk=False, label='Híbrido completo'),
    'sin_rna': dict(rna=False, fuzzy=True, ga=True, fixed_risk=False, label='Ablación: sin RNA'),
    'sin_difuso': dict(rna=True, fuzzy=False, ga=True, fixed_risk=False, label='Ablación: sin difuso'),
    'sin_ag': dict(rna=True, fuzzy=True, ga=False, fixed_risk=False, label='Ablación: sin AG'),
    'control': dict(rna=False, fuzzy=False, ga=False, fixed_risk=False, label='Control: ruta más corta sin riesgos'),
    'ag_puro': dict(rna=False, fuzzy=False, ga=True, fixed_risk=True, label='AG puro con riesgos fijos'),
}
FIXED_SPEED = .5          # speed used when the fuzzy controller is disabled
PLAN_SPEED = .85          # speed assumed when budgeting energy (typical cruise; avoids optimistic plans)
RISK_PRIOR = .1           # belief risk of a never-observed cell (supuesto)
FIXED_RISK = .15          # constant estimated risk for the 'ag_puro' baseline
DAY_LENGTH = 30           # steps per half day/night cycle (supuesto)
MAX_STEPS = 400
INSTRUMENT_BATTERY = 5.   # battery recovered with each instrument cache (supuesto)
# Event effects (all values are simulation assumptions).
MAX_DESTINATIONS = 8
DRONE_AHEAD, DRONE_ENERGY, DRONE_RESERVE = 4, .08, 15.  # cells scouted, % battery per flight, minimum battery
HEAT_STEPS, HEAT_EXTERNAL, HEAT_INTERNAL, HEAT_ENERGY = 12, 40., 35., 1.2
WIND_STEPS, WIND_ENERGY, WIND_SPEED = 5, 1.3, 22.
LINK_STEPS = 10
QUAKE_DAMAGE, METEORITE_DAMAGE, BIOSIGNAL_ENERGY, VIBRATION_LEVEL = 3., 5., 1., 2.5
# Events the rover perceives at once: the route is recomputed when they happen (F9).
# Hidden hazards (cliff, alien, manual obstacle) are discovered later by the sensors.
PERCEIVABLE_EVENTS = ('meteorito', 'sismo', 'viento', 'bateria_baja', 'ola_calor', 'tormenta_polvo')
WAIT_REASONS = ('R02_SENSORES_NAVEGACION_FALLIDOS', 'R08_ESPERA_SISMO', 'R12_DIFUSO_PARAR', 'R16_ESPERA_VIENTO')
EVENT_LABELS = {
    'barranco': 'Barranco', 'bateria_baja': 'Batería baja', 'dia': 'Día', 'noche': 'Noche',
    'sismo': 'Sismo', 'meteorito': 'Meteorito', 'tormenta_polvo': 'Tormenta de polvo',
    'corrosion': 'Corrosión', 'biosenal_anomala': 'Señal biológica anómala',
    'alienigena': 'Objeto/ser alienígena (ficción)', 'falla_sensor': 'Falla de sensor',
    'perdida_enlace': 'Pérdida de enlace', 'viento': 'Ráfaga de viento', 'ola_calor': 'Ola de calor',
}
MAX_EVENT_RATE = .10
# F3 random events: base weights, raised by each planet's own hazards (simulation assumption).
RANDOM_EVENT_WEIGHTS = {'sismo': 1., 'meteorito': 1., 'viento': 1., 'bateria_baja': .5, 'falla_sensor': .5,
                        'perdida_enlace': .5, 'biosenal_anomala': .3, 'barranco': 1.}
PLANET_EVENT_BONUS = {
    'mercurio': {'meteorito': 2., 'ola_calor': 2.}, 'venus': {'corrosion': 3., 'ola_calor': 2.},
    'tierra': {'sismo': 2.}, 'marte': {'tormenta_polvo': 3., 'barranco': 1.},
    'jupiter': {'viento': 2.}, 'saturno': {'viento': 2., 'perdida_enlace': 1.},
    'urano': {'bateria_baja': 1.}, 'neptuno': {'viento': 2., 'perdida_enlace': 1.},
}

_NN_CACHE = {}


def shared_perception(seed=0):
    """Train the MLP once per process (offline phase) and reuse it online."""
    if seed not in _NN_CACHE:
        started = time.perf_counter()
        perception = NeuralPerception(seed=seed)
        perception.train()
        perception.train_seconds_ = time.perf_counter() - started
        _NN_CACHE[seed] = perception
    return _NN_CACHE[seed]


def _finite(value):
    return isinstance(value, (int, float, np.number)) and not isinstance(value, bool) and math.isfinite(value)


def validate_config(config):
    """Return a normalized config or raise ValueError with a readable message."""
    cfg = dict(planet='Marte', mode='rover', seed=42, difficulty='medium', battery=100.,
               policy='hibrido', step_duration=1., science_count=5, thermal_limit=70.,
               weights=None, hidden_cliffs=4, ga_population=30, ga_generations=30,
               ga_crossover=.85, ga_mutation=.25, drone=False, event_rate=0.)
    cfg.update({k: v for k, v in (config or {}).items() if v is not None})
    if normalize(str(cfg['planet'])) not in PROFILES:
        raise ValueError(f"Planeta desconocido: {cfg['planet']}")
    if normalize(str(cfg['mode'])) not in ('rover', 'ruedas', 'sonda', 'probe'):
        raise ValueError('Modo inválido: use rover (ruedas) o sonda')
    for key, low, high in (('battery', 0, 100), ('step_duration', .01, 100), ('thermal_limit', -100, 1000)):
        try:
            cfg[key] = float(cfg[key])
        except (TypeError, ValueError):
            raise ValueError(f'{key}: debe ser un número') from None
        if not math.isfinite(cfg[key]) or not low <= cfg[key] <= high:
            raise ValueError(f'{key}: valor imposible ({cfg[key]}); rango permitido {low}–{high}')
    try:
        cfg['event_rate'] = float(cfg['event_rate'])
    except (TypeError, ValueError):
        raise ValueError('event_rate: debe ser un número') from None
    if not math.isfinite(cfg['event_rate']) or not 0 <= cfg['event_rate'] <= MAX_EVENT_RATE:
        raise ValueError(f"event_rate: probabilidad imposible ({cfg['event_rate']}); rango 0–{MAX_EVENT_RATE}")
    for key in ('ga_crossover', 'ga_mutation'):
        try:
            cfg[key] = float(cfg[key])
        except (TypeError, ValueError):
            raise ValueError(f'{key}: debe ser un número') from None
        if not math.isfinite(cfg[key]) or not 0 <= cfg[key] <= 1:
            raise ValueError(f'{key}: probabilidad imposible ({cfg[key]}); rango 0–1')
    for key, low, high in (('seed', 0, 2**31 - 1), ('science_count', 0, 12), ('hidden_cliffs', 0, 12),
                           ('ga_population', 30, 120), ('ga_generations', 30, 150)):
        value = cfg[key]
        if isinstance(value, float) and value.is_integer():
            value = int(value)
        if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or not low <= value <= high:
            raise ValueError(f'{key}: debe ser entero entre {low} y {high}')
        cfg[key] = int(value)
    if not isinstance(cfg['drone'], (bool, np.bool_)):
        raise ValueError('drone: debe ser verdadero o falso')
    cfg['drone'] = bool(cfg['drone'])
    if cfg['difficulty'] not in ('low', 'medium', 'high'):
        raise ValueError('Dificultad inválida: low, medium o high')
    if cfg['policy'] not in POLICIES:
        raise ValueError(f"Política desconocida: {cfg['policy']}")
    if cfg['weights'] is not None:
        weights = {}
        for key, value in dict(cfg['weights']).items():
            try:
                value = float(value)
            except (TypeError, ValueError):
                raise ValueError(f'Peso {key}: debe ser un número') from None
            if not math.isfinite(value) or value < 0:
                raise ValueError(f'Peso {key}: debe ser finito y no negativo')
            weights[key] = value
        if set(weights) - set(GeneticPlanner.default_weights):
            raise ValueError('Componente de fitness desconocido')
        if not any(weights.values()):
            raise ValueError('Los pesos del fitness no pueden ser todos cero')
        cfg['weights'] = weights
    # Raises a scientific explanation for gas giants + wheels (CA-07).
    PlanetProfile.for_body(cfg['planet'], cfg['mode'])
    return cfg


class CachedPlanner(GeneticPlanner):
    """Same GA, with per-run memoized BFS segments (speed only, same results)."""

    def plan(self, world, start, goal, battery=100, estimated_risk=None, **kwargs):
        self._cache = {}
        try:
            return super().plan(world, start, goal, battery, estimated_risk, **kwargs)
        finally:
            self._cache = None

    def _path(self, world, start, goal):
        cache = getattr(self, '_cache', None)
        if cache is None:
            return GeneticPlanner._path(world, start, goal)
        key = (tuple(start), tuple(goal))
        if key not in cache:
            cache[key] = GeneticPlanner._path(world, start, goal)
        return cache[key]


class HybridMission:
    operational_label = ASSUMPTION

    def __init__(self, **config):
        self.config = validate_config(config)
        cfg = self.config
        self.policy = POLICIES[cfg['policy']]
        self.profile = PlanetProfile.for_body(cfg['planet'], cfg['mode'])
        self.seed = cfg['seed']
        self.rng = np.random.default_rng(self.seed + 7)
        self.event_rng = np.random.default_rng(self.seed + 1013)  # F3: own stream, rate 0 leaves self.rng untouched
        self.world = World(16, 12, seed=self.seed, difficulty=cfg['difficulty'], science_count=cfg['science_count'])
        self.home = (0, 0)
        self._add_second_refuge()
        self.hidden_cliffs, self.hidden_obstacles = self._hide_hazards(cfg['hidden_cliffs'])
        self.rna_marks = set()    # hazards believed only from RNA output (unconfirmed)
        self.rna_retried = set()  # marks already discarded once; a second RNA flag makes them firm
        self.done = set()        # objectives completed: science sites visited + instruments recovered
        self.recovered = set()   # instrument caches recovered (mission goal: recuperar instrumentación)
        self.belief = self._make_belief()
        self.sensors = SensorSuite(self.seed, noise=.05)
        self.perception = shared_perception(0)
        self.fuzzy = FuzzyController()
        self.supervisor = SafetySupervisor()
        self.logger = MissionLogger()
        self.risk = {}
        self.position = self.home
        self.battery = cfg['battery']
        self.health = 100.
        self.daylight, self.visibility = True, 1.
        self.wait_steps = 0
        self.wind_wait = 0
        self.elapsed = 0.
        self.step_index = 0
        self.state = 'CONFIGURACION'
        self.route = None
        self.path = [self.position]
        self.planned_history = []
        self.ga = None
        self.ga_context = None
        self.ga_history_raw = []
        self.ga_runs = []
        self.returning = False
        self.finished = False
        self.outcome = None
        self.replan_reason = 'INICIO'
        self.timers = {'storm': 0, 'link': 0, 'vibration': 0, 'heat': 0, 'wind': 0}
        self.link_down_steps = 0
        self.queued_orders = 0
        self.sensor_timers = {}
        self.destinations = []  # operator waypoints in visiting order (F6, F13)
        self.tour_cells = 0
        self.day_counter = 0
        self.no_route_steps = 0
        self.nothing_left = False
        self.pending_events = []
        self.event_log = []
        self.last = {}
        self.metrics = dict(collisions=0, cliffs_avoided=0, hazards_avoided=0, replans=0, ga_runs=0,
                            science_samples=0, safety_violations=0, energy_used=0., distance=0,
                            decision_seconds=0., planning_seconds=0., anomalies=0, contacts=0, events={},
                            wait_steps=0, degraded_steps=0, link_outage_steps=0,
                            drone_steps=0, drone_discoveries=0, drone_energy=0.)
        self.drone = None  # scout drone state (F16); None when the mission has no drone

    # ------------------------------------------------------------------ setup
    def _free_cells(self):
        world = self.world
        return [(r, c) for r in range(world.height) for c in range(world.width)
                if world.traversable((r, c)) and (r, c) not in world.refuges]

    def _add_second_refuge(self):
        far = [cell for cell in self._free_cells() if cell[0] + cell[1] >= 18
               and cell not in self.world.science_sites and cell not in self.world.resources]
        if far:
            self.world.refuges.add(far[int(self.rng.integers(len(far)))])

    def _hide_hazards(self, count):
        world = self.world
        reserved = world.refuges | world.science_sites | world.resources | {self.home, (0, 1), (1, 0), (1, 1)}
        candidates = [cell for cell in self._free_cells() if cell not in reserved]
        cliffs = set()
        if self.profile.mode == 'rover' and candidates:
            for index in self.rng.permutation(len(candidates))[:count]:
                cell = candidates[int(index)]
                cliffs.add(cell)
                world.cliffs.add(cell)
                world.elevation[cell] -= 10.
        obstacles = sorted(world.obstacles)
        hidden = {obstacles[int(i)] for i in self.rng.permutation(len(obstacles))[:int(len(obstacles) * .3)]}
        return cliffs, hidden

    def _make_belief(self):
        belief = World.__new__(World)
        world = self.world
        belief.width, belief.height, belief.seed, belief.difficulty = world.width, world.height, world.seed, world.difficulty
        belief.rng = None
        belief.elevation = world.elevation.copy()
        for cell in self.hidden_cliffs:
            belief.elevation[cell] += 10.  # orbital map does not resolve the drop
        belief.obstacles = set(world.obstacles) - self.hidden_obstacles
        belief.cliffs = set()
        # Planner objectives = science sites + instrument caches to recover.
        belief.science_sites = world.science_sites | world.resources
        belief.visited_science = self.done
        belief.resources = world.resources
        belief.refuges = world.refuges
        return belief

    # -------------------------------------------------------------- helpers
    @property
    def temperature_internal(self):
        base = 20. + (self.profile.temperature - 20.) * .1  # thermal control model (supuesto)
        return base + (HEAT_INTERNAL if self.timers['heat'] else 0.)

    @property
    def vibration_level(self):
        return VIBRATION_LEVEL if self.timers['vibration'] else 0.

    @property
    def wind_speed(self):
        return WIND_SPEED if self.timers['wind'] else 0.

    @property
    def link_latency(self):
        """Seconds since the link was lost (0 with the link up)."""
        return self.link_down_steps * self.config['step_duration']

    def cell_cost(self, speed=FIXED_SPEED, slope=0.):
        base = .45 * (.7 + .6 * speed) * (1 + .5 * (1 - self.visibility)) + .01 * slope
        factor = (HEAT_ENERGY if self.timers['heat'] else 1.) * (WIND_ENERGY if self.timers['wind'] else 1.)
        return base * self.config['step_duration'] * factor

    def _bfs(self, start, goal=None, avoid=frozenset(), goals=None):
        goals = goals or {goal}
        world = self.belief
        queue, previous = deque([start]), {start: None}
        while queue:
            cell = queue.popleft()
            if cell in goals:
                route = []
                while cell is not None:
                    route.append(cell)
                    cell = previous[cell]
                return route[::-1]
            for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):
                nxt = (cell[0] + dr, cell[1] + dc)
                if nxt in previous or not world.traversable(nxt, self.profile.mode) or (nxt in avoid and nxt not in goals):
                    continue
                if self.profile.mode == 'rover' and world.elevation[nxt] - world.elevation[cell] < -5:
                    continue
                previous[nxt] = cell
                queue.append(nxt)
        return None

    def refuge_route(self):
        return self._bfs(self.position, goals=set(self.world.refuges))

    def energy_to_refuge(self):
        if self.position in self.world.refuges:
            return 0.
        route = self.refuge_route()
        return math.inf if route is None else (len(route) - 1) * self.cell_cost(PLAN_SPEED)

    def _risk_avoid(self):
        return {cell for cell, value in self.risk.items() if value >= .5}

    def _hazard_distance(self):
        """Meters to the first known/predicted hazard along the planned route."""
        for k, cell in enumerate((self.route or [])[1:5], start=1):
            known = cell in self.belief.cliffs or cell in self.belief.obstacles
            if known or self.risk.get(cell, 0.) >= .5:
                return float((k - 1) * 5 + 1)
        return 20.

    # ------------------------------------------------------------ perception
    def _features(self, cell, readings):
        distance = readings.get('distance')
        delta = readings.get('elevation_delta')
        if not _finite(distance) or not _finite(delta):
            raise ValueError('Lectura inválida para la RNA')
        noise = 1 + 2 * (1 - self.visibility)
        rng = self.rng
        return {
            'distance': float(np.clip(distance + rng.normal(0, .8 * noise), .01, 20)),
            'slope': float(np.clip(readings['slope'] + rng.normal(0, 2.), 0, 45)),
            'elevation_delta': float(delta + rng.normal(0, 1.3 * noise)),
            'texture': float(np.clip(.25 + .3 * (cell in self.world.obstacles) + rng.normal(0, .22 * noise), 0, 1)),
            'visibility': float(np.clip(self.visibility, .35, 1.)),
            'vibration': float(np.clip(abs(delta) * .15 + rng.normal(0, .15) + (1.5 if self.timers['vibration'] else 0), 0, 20)),
        }

    def _observe(self, cell):
        readings = self.sensors.observe(self.world, self.position, cell, self.battery, self.profile,
                                        self.daylight, self.visibility)
        if self.timers['heat'] and _finite(readings.get('temperature_external')):
            readings['temperature_external'] += HEAT_EXTERNAL
        readings['vibration'] = self.vibration_level
        readings['wind_speed'] = self.wind_speed
        return readings

    # ------------------------------------------------------------- planning
    def _mark_rna(self, cell, label):
        """Record a hazard inferred by the RNA; it stays revisable until seen twice."""
        (self.belief.cliffs if label == 'barranco' else self.belief.obstacles).add(cell)
        if cell not in self.rna_retried:
            self.rna_marks.add(cell)

    def _plan(self, reason):
        if self.destinations and not self.returning:
            self._order_destinations()
        self._plan_once(reason)
        if len(self.route) <= 1 and self.rna_marks:
            # No route while relying on unconfirmed RNA beliefs: discard them once and
            # replan. Edge sensor and RNA still re-check those cells before entering.
            cells = sorted(self.rna_marks)
            for cell in cells:
                self.belief.cliffs.discard(cell)
                self.belief.obstacles.discard(cell)
            self.rna_retried |= self.rna_marks
            self.rna_marks.clear()
            self.nothing_left = False
            entry = dict(step=self.step_index, kind='revision_rna', label='Revisión de creencias RNA',
                         message='Sin ruta con peligros inferidos solo por la RNA: se descartan '
                                 f'{len(cells)} marca(s) no confirmada(s) y se replanifica.', position=None)
            self.event_log.append(entry)
            self.pending_events.append(entry)
            self._plan_once(reason + '+REVISION_RNA')
        while len(self.route) <= 1 and self.destinations and not self.returning:
            # The next destination became unreachable (new hazard or low energy): drop it and resume.
            self._drop_destination(self.destinations[0])
            self._order_destinations()
            self._plan_once(reason + '+SIN_DESTINO')
        if len(self.route) <= 1 and not self.returning and self.position not in self.world.refuges:
            # No feasible exploration plan (typically low energy): head for the nearest
            # reachable refuge instead of waiting without a route.
            back = self.refuge_route()
            if back and len(back) > 1:
                self.returning = True
                self._plan_once(reason + '+RETORNO')

    @property
    def destination(self):
        """Next operator destination in the optimised visiting order."""
        return self.destinations[0] if self.destinations else None

    def _drop_destination(self, cell):
        self.destinations.remove(cell)
        entry = dict(step=self.step_index, kind='destino_descartado', label='Destino descartado',
                     message=f'El destino {list(cell)} ya no tiene ruta segura: se descarta y se retoma la misión.',
                     position=list(cell))
        self.event_log.append(entry)
        self.pending_events.append(entry)

    def _tour(self, cells):
        """Best order to visit ``cells`` from the current position, ending at a refuge.

        Distances come from the believed map; the order is evolved by the
        permutation GA. Returns (ordered cells, total cells including the return).
        """
        nodes = [self.position] + list(cells)
        hops = lambda path: len(path) - 1 if path else math.inf
        leg = [[0 if a == b else hops(self._bfs(a, b)) for b in nodes] for a in nodes]
        closing = [hops(self._bfs(node, goals=set(self.world.refuges))) for node in nodes]
        result = optimize_visit_order(leg, closing, seed=self.seed)
        return [nodes[i + 1] for i in result['order']], result['cost']

    def _order_destinations(self):
        """Drop unreachable destinations and keep the rest in the best visiting order."""
        for cell in [c for c in self.destinations if self._bfs(self.position, c) is None]:
            self._drop_destination(cell)
        if not self.destinations:
            self.tour_cells = 0
            return
        order, self.tour_cells = self._tour(self.destinations)
        if order != self.destinations and len(order) > 1:
            entry = dict(step=self.step_index, kind='orden_destinos', label='Orden de visita optimizado',
                         message='El AG de permutaciones ordenó los destinos: '
                                 + ' → '.join(str(list(c)) for c in order)
                                 + f' ({self.tour_cells} celdas con retorno al refugio).', position=None)
            self.event_log.append(entry)
            self.pending_events.append(entry)
        self.destinations = order

    def destination_tour(self):
        """Cells joining the destinations in visiting order (the leg to the first is the route)."""
        cells = [self.destinations[0]] if self.destinations else []
        for a, b in zip(self.destinations, self.destinations[1:]):
            cells += (self._bfs(a, b) or [a, b])[1:]
        return cells

    def _plan_once(self, reason):
        started = time.perf_counter()
        self.metrics['replans'] += 1
        self.replan_reason = None
        if self.returning:
            route = self.refuge_route()
            self.route = route if route else [self.position]
            self.ga = dict(reason=reason, mode='retorno', candidates=[], rejected=[], history=[],
                           message='Ruta directa al refugio más cercano.')
        elif self.policy['ga']:
            self._plan_ga(reason)
        else:
            self._plan_greedy(reason)
        self.ga['step'] = max(1, self.step_index)
        self.planned_history.append(list(self.route))
        self.metrics['planning_seconds'] += time.perf_counter() - started

    def risk_map(self):
        cells = [(r, c) for r in range(self.belief.height) for c in range(self.belief.width)]
        if self.policy['fixed_risk']:
            base = {cell: FIXED_RISK for cell in cells}
        else:
            base = {cell: RISK_PRIOR for cell in cells}
            base.update({cell: float(np.clip(v, 0, 1)) for cell, v in self.risk.items()})
        for cell in self.belief.cliffs | self.belief.obstacles:
            base[cell] = 1.
        return base

    def _plan_ga(self, reason):
        self.metrics['ga_runs'] += 1
        cfg = self.config
        planner = CachedPlanner(population_size=cfg['ga_population'], generations=cfg['ga_generations'], waypoints=3,
                                seed=self.seed + self.metrics['ga_runs'], weights=cfg['weights'],
                                crossover_rate=cfg['ga_crossover'], mutation_rate=cfg['ga_mutation'])
        goal = self.destination or self.home
        risk = self.risk_map()
        kwargs = dict(temperature=self.temperature_internal, thermal_limit=self.config['thermal_limit'],
                      energy_per_step=self.cell_cost(PLAN_SPEED))
        result = planner.plan(self.belief, self.position, goal, battery=self.battery, estimated_risk=risk, **kwargs)
        accepted, rejected = self.supervisor.filter_routes(result['candidates'], self.belief.cliffs, self.belief.obstacles)
        chosen = accepted[0] if accepted else None
        self.route = list(result['candidates'][chosen]['route']) if chosen is not None else [self.position]
        if chosen is None or (not result['candidates'][chosen]['science_order'] and self.position in self.world.refuges):
            self.nothing_left = self.position in self.world.refuges
        self.ga_context = dict(planner=planner, goal=goal, battery=self.battery, risk=risk, kwargs=kwargs,
                               start=self.position)
        self.ga_history_raw = result['history']
        self.ga = dict(
            reason=reason, mode='ag', chosen=chosen, rejected=rejected, feasible=result['feasible'],
            message=result['message'], generations_run=result['generations_run'], stopped_early=result['stopped_early'],
            candidates=[dict(index=i, route=[list(p) for p in c['route']], fitness=round(c['fitness'], 4),
                             feasible=c['feasible'], components={k: round(v, 4) for k, v in c['components'].items()},
                             penalty=round(c['penalty'], 3), violations=c['violations'],
                             science_order=[list(p) for p in c['science_order']],
                             chromosome=[list(p) for p in c['chromosome']], distance=c['distance'])
                        for i, c in enumerate(result['candidates'])],
            history=[dict(generation=h['generation'], best=round(h['best_fitness'], 4),
                          best_feasible=None if h['best_feasible_fitness'] is None else round(h['best_feasible_fitness'], 4),
                          tournaments=h['tournaments'], crossovers=h['crossovers'], mutations=h['mutations'])
                     for h in result['history']],
            weights=dict(planner.weights))
        self.ga_runs.append(dict(ga=self.ga, raw=self.ga_history_raw, ctx=self.ga_context,
                                 belief=(set(self.belief.obstacles), set(self.belief.cliffs)),
                                 visited=set(self.belief.visited_science),
                                 step=self.step_index))

    def ga_run(self, run=None):
        if not self.ga_runs:
            raise ValueError('Aún no hay ejecuciones del AG')
        run = len(self.ga_runs) - 1 if run is None else max(0, min(int(run), len(self.ga_runs) - 1))
        return run, self.ga_runs[run]

    def _plan_greedy(self, reason):
        """Nearest-neighbour tour over remaining sites with a battery check (no GA)."""
        avoid = self._risk_avoid() if self.policy['rna'] else set()
        if self.destination is not None:
            self.route = self._bfs(self.position, self.destination, avoid) or self._bfs(self.position, self.destination) or [self.position]
            self.ga = dict(reason=reason, mode='voraz', chosen=None, rejected=[], candidates=[], history=[],
                           message='Ruta directa al destino del operador (sin AG).', science_order=[])
            return
        remaining = sorted(self.belief.science_sites - self.done)
        position, route, order = self.position, [self.position], []
        cost = self.cell_cost(PLAN_SPEED)
        while remaining:
            options = [(self._bfs(position, site, avoid), site) for site in remaining]
            options = [(p, s) for p, s in options if p]
            if not options:
                break
            path, site = min(options, key=lambda o: (len(o[0]), o[1]))
            back = self._bfs(site, goals=set(self.world.refuges), avoid=avoid)
            if back is None or (len(route) - 1 + len(path) - 1 + len(back) - 1) * cost > self.battery:
                break
            route.extend(path[1:])
            order.append(site)
            remaining.remove(site)
            position = site
        back = self._bfs(position, self.home, avoid) or self._bfs(position, goals=set(self.world.refuges), avoid=avoid)
        if back:
            route.extend(back[1:])
        self.route = route
        if not order and self.position in self.world.refuges:
            self.nothing_left = True
        self.ga = dict(reason=reason, mode='voraz', chosen=None, rejected=[], candidates=[], history=[],
                       message='Planificador voraz (sin AG): vecino más cercano.',
                       science_order=[list(p) for p in order])

    def ga_generation(self, generation, run=None):
        """Chromosomes of one generation re-evaluated with that run's planning context."""
        run, record = self.ga_run(run)
        raw, ctx = record['raw'], record['ctx']
        generation = max(0, min(int(generation), len(raw) - 1))
        entry = raw[generation]
        # Re-evaluate against the belief known when that plan was made.
        snapshot = World.__new__(World)
        snapshot.__dict__.update(self.belief.__dict__)
        snapshot.obstacles, snapshot.cliffs = set(record['belief'][0]), set(record['belief'][1])
        snapshot.visited_science = set(record['visited'])  # science reward depends on what was still pending
        rows = []
        for chromosome in entry['chromosomes'][:8]:
            result = ctx['planner'].evaluate(chromosome, snapshot, ctx['start'], ctx['goal'], ctx['battery'],
                                             ctx['risk'], **ctx['kwargs'])
            rows.append(dict(chromosome=[list(p) for p in chromosome], fitness=round(result['fitness'], 4),
                             feasible=result['feasible'], violations=result['violations'],
                             components={k: round(v, 4) for k, v in result['components'].items()}))
        return dict(generation=generation, total=len(raw), rows=rows, run=run, runs=len(self.ga_runs),
                    tournaments=entry['tournaments'], crossovers=entry['crossovers'], mutations=entry['mutations'],
                    best=round(entry['best_fitness'], 4))

    # ---------------------------------------------------------------- events
    def _replan_now(self, reason):
        """Replan at once (F9); before the first plan exists, just record the reason."""
        if self.route is None:
            self.replan_reason = reason
            return False
        self._plan(reason)
        self.ga['step'] = self.step_index + 1  # logged with the next step record
        return True

    def _destination_cell(self, position):
        try:
            cell = tuple(int(v) for v in position)
            assert len(cell) == 2
        except (TypeError, ValueError, AssertionError):
            raise ValueError('Destino inválido: se esperan (fila, columna).') from None
        return cell

    def remove_destination(self, position):
        """Operator removes one destination; the rest are reordered."""
        cell = self._destination_cell(position)
        if cell not in self.destinations:
            raise ValueError('No hay un destino fijado en esa celda.')
        self.destinations.remove(cell)
        entry = dict(step=self.step_index, kind='destino_cancelado', label='Destino cancelado',
                     message=f'El operador quitó el destino {list(cell)}: quedan {len(self.destinations)}.',
                     position=list(cell))
        self.event_log.append(entry)
        self.pending_events.append(entry)
        self._replan_now('DESTINO_CANCELADO')
        return entry

    def set_destination(self, position):
        """Add an operator destination (None clears them all).

        Validated, planned with the same pipeline and never above safety. With
        several destinations the visiting order is the shortest tour found by the GA.
        """
        if self.finished:
            raise ValueError('La misión ya terminó: reinicie para fijar un destino.')
        if position is None:
            if not self.destinations:
                raise ValueError('No hay destino fijado.')
            entry = dict(step=self.step_index, kind='destino_cancelado', label='Destino cancelado',
                         message='El operador quitó los destinos: se retoma la misión.', position=list(self.destinations[0]))
            self.destinations, self.tour_cells = [], 0
        else:
            cell = self._destination_cell(position)
            if not self.world.in_bounds(cell):
                raise ValueError('Destino fuera del mapa.')
            if cell == self.position:
                raise ValueError('El rover ya está en esa celda.')
            if cell in self.destinations:
                raise ValueError('Ese destino ya está fijado.')
            if len(self.destinations) >= MAX_DESTINATIONS:
                raise ValueError(f'Máximo {MAX_DESTINATIONS} destinos a la vez.')
            if not self.belief.traversable(cell, self.profile.mode):
                raise ValueError('Destino inválido: la celda es un obstáculo o barranco conocido.')
            if self._bfs(self.position, cell) is None:
                raise ValueError('Destino inalcanzable: no existe ruta conocida hasta esa celda.')
            order, cells = self._tour(self.destinations + [cell])
            need = cells * self.cell_cost(PLAN_SPEED)
            if need > self.battery:
                raise ValueError('Destino rechazado: la energía no alcanza para visitar los destinos y volver a un refugio '
                                 f'(se requiere {need:.1f} % y hay {self.battery:.1f} %): no hay retorno seguro.')
            self.destinations.append(cell)
            self.returning = False
            entry = dict(step=self.step_index, kind='destino', label='Destino fijado',
                         message=f'Destino del operador en {list(cell)}: {len(self.destinations)} destino(s), '
                                 f'recorrido de {cells} celdas con retorno al refugio.', position=list(cell))
        self.event_log.append(entry)
        self.pending_events.append(entry)
        self._replan_now(entry['kind'].upper())
        return entry

    def inject(self, kind, **data):
        if self.finished:
            raise ValueError('La misión ya terminó: reinicie para inyectar eventos.')
        position = data.pop('position', None)
        origin = data.pop('origin', 'operador')
        event = Event(kind, data)
        kind = event.kind
        if self.timers['link'] and origin == 'operador':
            self.queued_orders += 1  # operator order issued during the outage
        if position is not None:
            position = tuple(int(v) for v in position)
        if kind in ('barranco', 'meteorito', 'alienigena'):
            if kind == 'barranco' and self.profile.mode != 'rover':
                raise ValueError('Barranco no aplica al modo sonda')
            if position is None:
                route = self.route or [self.position]
                offset = {'barranco': 1, 'meteorito': 3, 'alienigena': 2}[kind]
                position = route[min(offset, len(route) - 1)] if len(route) > 1 else None
                if position is None or position in self.world.refuges or position == self.position:
                    neighbors = [(self.position[0] + dr, self.position[1] + dc) for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0))]
                    neighbors = [n for n in neighbors if self.world.traversable(n) and n not in self.world.refuges]
                    if not neighbors:
                        raise ValueError('No hay celda libre adyacente para el evento')
                    position = neighbors[0]
            if not self.world.in_bounds(position):
                raise ValueError('Evento fuera del mapa')
            if position == self.position or position in self.world.refuges:
                raise ValueError('No se puede colocar un peligro sobre el rover o un refugio')
            event.data['position'] = position
        if kind == 'falla_sensor':
            sensor = data.get('sensor', 'distance')
            names = ['distance', 'elevation_delta'] if sensor == 'todos' else [sensor]
            for name in names:
                self.sensors.fail(name, data.get('failure', 'nan'))
                self.sensor_timers[name] = 8
            pretty = {'distance': 'proximidad', 'elevation_delta': 'LIDAR'}
            result = {'kind': kind, 'message': 'Falla simulada de sensor ' + ', '.join(pretty.get(n, n) for n in names)
                      + ' (NaN) durante 8 pasos: degradación segura.'}
        else:
            quake_wait = self.wait_steps
            result = event.apply(self)
            if kind == 'viento':
                self.wait_steps = quake_wait  # wind is not a quake: it has its own hold
        if kind == 'barranco':
            self.world.elevation[position] += 10.  # Event lowers 20 m; keep a 10 m drop
        elif kind == 'meteorito':
            self.belief.add_obstacle(position)
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    cell = (position[0] + dr, position[1] + dc)
                    if self.world.in_bounds(cell) and cell != self.position:
                        self.risk[cell] = max(self.risk.get(cell, 0.), .6)
            self.timers['vibration'] = 2
            if abs(position[0] - self.position[0]) + abs(position[1] - self.position[1]) <= 1:
                self.health = max(0., self.health - METEORITE_DAMAGE)
        elif kind == 'sismo':
            self.wait_steps = 2
            self.timers['vibration'] = 3
            self.health = max(0., self.health - QUAKE_DAMAGE)
        elif kind == 'ola_calor':
            self.timers['heat'] = HEAT_STEPS
        elif kind == 'alienigena':
            self.metrics['contacts'] += 1
        elif kind == 'tormenta_polvo':
            self.timers['storm'] = 15
        elif kind in ('dia', 'noche'):
            self.day_counter = 0
        elif kind == 'perdida_enlace':
            self.timers['link'] = LINK_STEPS
        elif kind == 'biosenal_anomala':
            self.metrics['anomalies'] += 1
            self.battery = max(0., self.battery - BIOSIGNAL_ENERGY)  # spectrometer analysis
        elif kind == 'viento':
            self.timers['wind'] = WIND_STEPS
            if self.profile.mode == 'probe':
                self.wind_wait = 1
        self.metrics['events'][kind] = self.metrics['events'].get(kind, 0) + 1
        entry = dict(step=self.step_index, kind=kind, label=EVENT_LABELS.get(kind, kind),
                     message=result.get('message', kind), position=list(position) if position else None,
                     origin=origin)
        self.event_log.append(entry)
        self.pending_events.append(entry)
        if kind in PERCEIVABLE_EVENTS:
            entry['replanned'] = self._replan_now(kind.upper())  # F9: now, not on the next step
        return entry

    def _random_event(self):
        """F3: with probability event_rate, inject one planet-weighted event inside this step."""
        rate = self.config['event_rate']
        if rate <= 0 or self.finished:
            return None
        if self.event_rng.random() >= rate:
            return None
        weights = dict(RANDOM_EVENT_WEIGHTS)
        for kind, bonus in PLANET_EVENT_BONUS.get(normalize(self.profile.name), {}).items():
            weights[kind] = weights.get(kind, 0.) + bonus
        if self.profile.mode != 'rover':
            weights.pop('barranco', None)  # cliffs do not apply to the probe
        kinds = sorted(weights)
        total = sum(weights[k] for k in kinds)
        kind = kinds[int(self.event_rng.choice(len(kinds), p=[weights[k] / total for k in kinds]))]
        try:
            return self.inject(kind, origin='aleatorio')
        except ValueError:
            return None  # no valid cell for it right now: the draw is simply lost

    def _tick_environment(self):
        self.day_counter += 1
        if self.day_counter >= DAY_LENGTH:
            self.day_counter = 0
            self.daylight = not self.daylight
        if self.timers['storm']:
            self.timers['storm'] -= 1
            self.visibility = .2
        else:
            self.visibility = 1. if self.daylight else .6
        for key in ('vibration', 'heat', 'wind'):
            if self.timers[key]:
                self.timers[key] -= 1
        if self.timers['link']:
            self.timers['link'] -= 1
            self.metrics['link_outage_steps'] += 1
            self.link_down_steps += 1
            if not self.timers['link']:
                self.link_down_steps = 0
                self.queued_orders = 0  # queue is flushed when the link returns
        self.sensors.link_available = self.timers['link'] == 0
        for name in list(self.sensor_timers):
            self.sensor_timers[name] -= 1
            if self.sensor_timers[name] <= 0:
                del self.sensor_timers[name]
                self.sensors.failures.pop(name, None)
        if self.daylight and self.profile.mode == 'rover':
            self.battery = min(100., self.battery + .12 * self.visibility)  # solar (supuesto)
        if self.profile.name == 'Venus':
            self.health = max(0., self.health - .4)  # corrosion (supuesto)

    def set_drone(self, enabled):
        """Turn the scout drone on or off during the mission, without restarting it.

        The change is logged with its step (and exported), so the final config is
        never mistaken for a drone that flew from the start.
        """
        if not isinstance(enabled, (bool, np.bool_)):
            raise ValueError('enabled: debe ser verdadero o falso')
        enabled = bool(enabled)
        if self.finished:
            raise ValueError('La misión ya terminó: reinicie para cambiar el dron.')
        if enabled == self.config['drone']:
            raise ValueError('El dron ya está ' + ('activado.' if enabled else 'apagado.'))
        self.config = dict(self.config, drone=enabled)
        if enabled:
            self.drone = dict(position=list(self.position), flying=False, reason='Despega en el próximo paso.',
                              scanned=[], found=[])
        else:
            self.drone = None
        entry = dict(step=self.step_index, kind='dron_activado' if enabled else 'dron_desactivado',
                     label='Dron activado' if enabled else 'Dron desactivado',
                     message=(f'El operador activó el dron en el paso {self.step_index}: '
                              'explora la ruta desde el próximo paso.') if enabled else
                             f'El operador apagó el dron en el paso {self.step_index}: el rover sigue solo con sus sensores.',
                     position=None)
        self.event_log.append(entry)
        self.pending_events.append(entry)
        return entry

    def _drone_scan(self):
        """Scout drone: flies the next cells of the route and shares hazards with the rover.

        It only reads what is physically there, never future events. Flight energy is
        taken from the rover battery; it stays grounded in dust storms, strong wind
        or below the battery reserve, and does not exist in probe mode.
        """
        if not self.config['drone']:
            return
        ahead = [tuple(c) for c in (self.route or [])[1:DRONE_AHEAD + 1]]
        reason = None
        if self.profile.mode != 'rover':
            reason = 'La sonda atmosférica no lleva dron.'
        elif self.timers['storm']:
            reason = 'En tierra por la tormenta de polvo.'
        elif self.timers['wind']:
            reason = 'En tierra por el viento fuerte.'
        elif self.battery < DRONE_RESERVE:
            reason = f'En tierra: batería bajo la reserva de {DRONE_RESERVE:.0f} %.'
        elif not ahead:
            reason = 'En tierra: no hay ruta que explorar.'
        if reason:
            self.drone = dict(position=list(self.position), flying=False, reason=reason, scanned=[], found=[])
            return
        world, belief = self.world, self.belief
        scan = {(r + dr, c + dc) for r, c in ahead for dr in (-1, 0, 1) for dc in (-1, 0, 1)}
        scan = {cell for cell in scan if world.in_bounds(cell)}
        cliffs = (world.cliffs & scan) - belief.cliffs
        obstacles = (world.obstacles & scan) - belief.obstacles
        found = sorted(cliffs | obstacles)
        belief.cliffs |= cliffs
        belief.obstacles |= obstacles
        self.rna_marks -= cliffs | obstacles  # confirmed by the drone, no longer a revisable belief
        self.battery = max(0., self.battery - DRONE_ENERGY)
        self.metrics['drone_steps'] += 1
        self.metrics['drone_energy'] += DRONE_ENERGY
        self.metrics['drone_discoveries'] += len(found)
        self.drone = dict(position=list(ahead[-1]), flying=True, reason=None,
                          scanned=[list(c) for c in sorted(scan)], found=[list(c) for c in found])
        if found:
            blocked = [c for c in found if c in set(tuple(x) for x in self.route)]
            entry = dict(step=self.step_index, kind='dron', label='Dron explorador',
                         message=f'El dron detectó {len(cliffs)} barranco(s) y {len(obstacles)} obstáculo(s) adelante'
                                 + (': la ruta pasaba por ahí y se recalcula.' if blocked else ', fuera de la ruta.'),
                         position=list(found[0]), cells=[list(c) for c in found])
            self.event_log.append(entry)
            self.pending_events.append(entry)
            if blocked and not self.replan_reason:
                self.replan_reason = 'DRON_DETECTA_PELIGRO'

    # ------------------------------------------------------------------ step
    def step(self):
        if self.finished:
            return self.last.get('record')
        started = time.perf_counter()
        self.step_index += 1
        self.elapsed += self.config['step_duration']
        depleted = self.battery <= 0  # 0 % means end of mission: solar cannot revive it
        self._tick_environment()
        self._random_event()
        if depleted:
            self.battery = 0.
        self._drone_scan()
        complete = self.position in self.world.refuges and self.step_index > 1 and not self.returning and (
            not (self.belief.science_sites - self.done) or self.nothing_left) and self.destination is None
        if not complete:
            if self.route is None or (self.replan_reason and self.replan_reason not in ('RECARGA_COMPLETA',)):
                self._plan(self.replan_reason or 'INICIO')
            elif self.replan_reason == 'RECARGA_COMPLETA':
                self._plan('RECARGA_COMPLETA')
            if self.route and len(self.route) <= 1 and not (self.returning and self.position in self.world.refuges):
                self._plan('OBJETIVO_ALCANZADO')
                complete = self.position in self.world.refuges and self.nothing_left and self.step_index > 1
        next_cell = self.route[1] if self.route and len(self.route) > 1 else None

        # 1-2. observe + validate
        readings = self._observe(next_cell or self.position)
        nav_valid = {'proximidad': _finite(readings.get('distance')) and 0 <= readings['distance'] <= 20,
                     'lidar': _finite(readings.get('elevation_delta'))}
        edge = bool(readings.get('cliff')) if next_cell and nav_valid['lidar'] and self.profile.mode == 'rover' else None

        # 3. RNA lookahead (next two cells)
        rna, rna_error = None, None
        if self.policy['rna'] and next_cell:
            for k, cell in enumerate(self.route[1:3]):
                obs = readings if k == 0 else self._observe(cell)
                try:
                    features = self._features(cell, obs)
                    prediction = self.perception.predict(features)
                except ValueError as error:
                    rna_error = str(error)
                    break
                self.risk[cell] = float(np.clip(prediction['risk'], 0, 1))
                if k == 0:
                    rna = dict(prediction, features=features, cell=list(cell))
                elif prediction['label'] != 'libre' and prediction['confidence'] >= self.supervisor.confidence_threshold:
                    # Early warning two cells ahead: mark the cell and replan next step.
                    self._mark_rna(cell, prediction['label'])
                    self.replan_reason = 'RNA_ANTICIPA_PELIGRO'
                    self.metrics['hazards_avoided'] += 1

        # 4. fuzzy control
        slope = readings['slope'] if _finite(readings.get('slope')) else 20.
        hazard = self._hazard_distance() if nav_valid['proximidad'] else 2.
        if edge:
            hazard = 1.
        fuzzy = None
        if self.policy['fuzzy']:
            fuzzy = self.fuzzy.infer(hazard, float(np.clip(self.battery, 0, 100)), float(np.clip(slope, 0, 45)),
                                     float(np.clip(self.temperature_internal, -273.15, 1000)))
            fuzzy['inputs'] = dict(distance=hazard, battery=round(self.battery, 2), slope=round(slope, 2),
                                   temperature=round(self.temperature_internal, 2))
            speed = fuzzy['speed']
        else:
            speed = FIXED_SPEED

        # 5. supervisor
        at_refuge = self.position in self.world.refuges
        ctx = dict(battery=self.battery, health=self.health, temperature_internal=self.temperature_internal,
                   thermal_limit=self.config['thermal_limit'], nav_valid=nav_valid, edge_cliff=edge,
                   rna=rna, rna_used=self.policy['rna'], fuzzy_speed=speed if self.policy['fuzzy'] else None,
                   has_route=next_cell is not None,
                   needs_replan=self.replan_reason is not None and self.replan_reason != 'RECARGA_COMPLETA',
                   quake_wait=self.wait_steps, wind_wait=self.wind_wait, at_refuge=at_refuge, energy_to_refuge=self.energy_to_refuge(),
                   recharge_target=90. if self.returning else 0., returning=self.returning,
                   mission_complete=complete)
        decision = self.supervisor.decide(ctx)
        moved, collision, cost_total = [], None, 0.

        # 6. act
        action = decision.action
        if action == 'replanificar':
            if decision.reason == 'R04_VETO_BARRANCO_CONFIRMADO':
                self.belief.cliffs.add(next_cell)
                self.metrics['cliffs_avoided'] += 1
            elif decision.reason == 'R05_RNA_PELIGRO':
                self._mark_rna(next_cell, rna['label'])
                self.metrics['hazards_avoided'] += 1
            self._plan(decision.reason)
        elif action == 'volver':
            self.returning = True
            self._plan('RETORNO_ENERGIA')
        elif action == 'recargar':
            self.battery = min(100., self.battery + 6.)
            if self.battery >= 90.:
                self.returning = False
                self.replan_reason = 'RECARGA_COMPLETA'
        elif action == 'avanzar':
            effective = min(speed, decision.speed_cap)
            cells = 2 if (effective >= .65 and len(self.route) > 2
                          and self.risk.get(self.route[2], RISK_PRIOR) < .3) else 1
            for _ in range(cells):
                if len(self.route) < 2:
                    break
                cell = self.route[1]
                cost = self.cell_cost(effective, slope)
                if cell in self.world.cliffs and self.profile.mode == 'rover':
                    self.health = 0.
                    self.metrics['safety_violations'] += 1
                    collision = 'caida_barranco'
                    break
                if cell in self.world.obstacles:
                    self.health = max(0., self.health - 25.)
                    self.metrics['collisions'] += 1
                    self.belief.obstacles.add(cell)
                    self.battery = max(0., self.battery - cost / 2)
                    collision = 'colision'
                    self.replan_reason = 'COLISION'
                    break
                if cost > self.battery:
                    break
                self.battery = max(0., self.battery - cost)
                cost_total += cost
                self.position = cell
                self.route.pop(0)
                self.path.append(cell)
                moved.append(cell)
                self.metrics['distance'] += 1
                if cell in self.world.science_sites and cell not in self.world.visited_science:
                    self.world.visited_science.add(cell)
                    self.done.add(cell)
                    self.metrics['science_samples'] += 1
                if cell in self.destinations:
                    self.destinations.remove(cell)
                    left = len(self.destinations)
                    entry = dict(step=self.step_index, kind='destino_alcanzado', label='Destino alcanzado',
                                 message=f'Destino {list(cell)} alcanzado: '
                                         + (f'quedan {left} destino(s).' if left else 'se retoman los objetivos pendientes y el retorno.'),
                                 position=list(cell))
                    self.event_log.append(entry)
                    self.pending_events.append(entry)
                    self.replan_reason = 'DESTINO_ALCANZADO'
                if cell in self.world.resources and cell not in self.recovered:
                    self.recovered.add(cell)
                    self.done.add(cell)
                    self.battery = min(100., self.battery + INSTRUMENT_BATTERY)
                    entry = dict(step=self.step_index, kind='instrumento', label='Instrumento recuperado',
                                 message=f'Instrumentación recuperada en {list(cell)}: +{INSTRUMENT_BATTERY:.0f} % de batería (celda de energía).',
                                 position=list(cell))
                    self.event_log.append(entry)
                    self.pending_events.append(entry)
        if action not in ('avanzar', 'recargar'):
            idle = .05 * self.config['step_duration']
            self.battery = max(0., self.battery - idle)
            cost_total += idle
        self.metrics['energy_used'] += cost_total
        if self.wait_steps and decision.reason == 'R08_ESPERA_SISMO':
            self.wait_steps -= 1
        if self.wind_wait and decision.reason == 'R16_ESPERA_VIENTO':
            self.wind_wait -= 1
        self.no_route_steps = self.no_route_steps + 1 if decision.reason == 'R10_SIN_RUTA' else 0
        self.metrics['wait_steps'] += decision.reason in WAIT_REASONS
        self.metrics['degraded_steps'] += not all(nav_valid.values())

        # 7. update mission state
        self.state = decision.state
        if collision == 'caida_barranco' or self.health <= 0 or (self.battery <= 0 and self.position not in self.world.refuges):
            self.state, self.finished = 'FRACASO', True
            self.outcome = 'Rover perdido: ' + ('caída en barranco' if collision == 'caida_barranco' else 'recursos agotados')
        elif decision.reason == 'R15_MISION_COMPLETA':
            self.finished, self.outcome = True, 'Misión cumplida: objetivos alcanzables y retorno seguro'
        elif decision.reason in ('R06_REFUGIO_INALCANZABLE', 'R01_LIMITE_TERMICO', 'R00_RECURSOS_AGOTADOS',
                                 'R03_DANO_CRITICO'):
            self.finished, self.outcome = True, decision.explanation
            self.state = 'FRACASO' if decision.reason == 'R00_RECURSOS_AGOTADOS' else 'MODO_SEGURO'
        elif self.no_route_steps >= 10:
            self.state, self.finished, self.outcome = 'FRACASO', True, 'Sin solución: no existe ruta factible'
        elif self.step_index >= MAX_STEPS:
            self.state, self.finished, self.outcome = 'FRACASO', True, 'Tiempo extremo agotado'
        elif collision:
            self.state = 'ALERTA'
        self.metrics['decision_seconds'] += time.perf_counter() - started

        # 8. trace
        events, self.pending_events = self.pending_events, []
        explanation = decision.explanation
        if collision == 'colision':
            explanation += ' Contacto con obstáculo no detectado: daño -25 %, se replanifica.'
        if self.timers['link']:
            explanation += ' Enlace perdido: autonomía local, órdenes en cola.'
        ga_now = self.ga if self.ga is not None and self.ga.get('step') == self.step_index else None
        record = self.logger.record(
            step=self.step_index, t=round(self.elapsed, 2), policy=self.config['policy'], state=self.state,
            position=list(self.position), moved=[list(c) for c in moved], battery=round(self.battery, 3),
            health=round(self.health, 2), action=action, reason=decision.reason, explanation=explanation,
            vetoes=decision.vetoes, permitted_by=decision.permitted_by,
            sensors={k: readings.get(k) for k in ('distance', 'elevation_delta', 'slope', 'temperature_external',
                                                     'visibility', 'solar_power', 'pressure', 'oxygen_ambient',
                                                     'radiation', 'link_available', 'vibration', 'wind_speed')},
            nav_valid=nav_valid, edge_cliff=edge,
            rna=None if rna is None else dict(label=rna['label'], confidence=round(rna['confidence'], 4),
                                              probabilities={k: round(v, 4) for k, v in rna['probabilities'].items()},
                                              low_confidence=rna['low_confidence']),
            rna_error=rna_error,
            fuzzy=None if fuzzy is None else dict(speed=round(fuzzy['speed'], 4), caution=round(fuzzy['caution'], 4),
                                                  rules=[[r['id'], round(r['strength'], 3)] for r in fuzzy['activated_rules']]),
            crisp_speed=round(speed, 4), collision=collision,
            ga=None if ga_now is None else dict(
                mode=ga_now['mode'], reason=ga_now['reason'], chosen=ga_now.get('chosen'),
                best_fitness=(ga_now['candidates'][ga_now['chosen']]['fitness']
                              if ga_now.get('chosen') is not None else None),
                rejected=ga_now.get('rejected')),
            route=[list(c) for c in (self.route or [])[:12]], events=events,
            science=round(self.world.science_progress, 3), seed=self.seed,
            event_count=sum(self.metrics['events'].values()), operational_label=ASSUMPTION)
        self.last = dict(record=record, rna=rna, fuzzy=fuzzy, decision=decision.to_dict(), readings=readings,
                         nav_valid=nav_valid, speed=speed)
        return record

    def prepare(self):
        """Compute the initial plan before the first step so the UI can show it."""
        if self.route is None and self.step_index == 0:
            self._plan(self.replan_reason or 'INICIO')
        return self

    def run(self, max_steps=MAX_STEPS):
        while not self.finished and self.step_index < max_steps:
            self.step()
        return self.summary()

    # ------------------------------------------------------------- reporting
    def summary(self):
        m = self.metrics
        return dict(policy=self.config['policy'], seed=self.seed, planet=self.profile.name,
                    success=self.state == 'EXITO', state=self.state, outcome=self.outcome,
                    steps=self.step_index, collisions=m['collisions'], cliffs_avoided=m['cliffs_avoided'],
                    hazards_avoided=m['hazards_avoided'], battery_left=round(self.battery, 3),
                    energy_used=round(m['energy_used'], 3), distance=m['distance'],
                    science=round(self.world.science_progress, 3), samples=m['science_samples'],
                    instruments=len(self.recovered), instruments_total=len(self.world.resources),
                    events=dict(m['events']), events_total=sum(m['events'].values()),
                    wait_steps=m['wait_steps'], degraded_steps=m['degraded_steps'],
                    anomalies=m['anomalies'], contacts=m['contacts'], link_outage_steps=m['link_outage_steps'],
                    drone_steps=m['drone_steps'], drone_discoveries=m['drone_discoveries'],
                    replans=m['replans'], ga_runs=m['ga_runs'], safety_violations=m['safety_violations'],
                    decision_ms=round(1000 * m['decision_seconds'] / max(1, self.step_index), 3),
                    planning_s=round(m['planning_seconds'], 3), health=round(self.health, 2))

    def export_json(self):
        return clean(dict(version='1.0', operational_label=ASSUMPTION, config=self.config,
                          summary=self.summary(), events=self.event_log, trace=self.logger.trace))

    def export_csv(self):
        return self.logger.export_csv()


In [ ]:
%%writefile ares_x/src/ares/experiments.py
"""Repeated experiments: >= 20 seeds per scenario, ablations and baselines (RF-14, RF-15)."""
import time

import numpy as np
import pandas as pd

from .mission import POLICIES, HybridMission

SCENARIOS = {
    'estandar': dict(label='Estándar (peligros ocultos)', events=[]),
    'eventos': dict(label='Con eventos (barranco, tormenta, sismo, falla de sensor)',
                    events=[(3, 'barranco', {}), (8, 'tormenta_polvo', {}), (14, 'sismo', {}),
                            (20, 'falla_sensor', {'sensor': 'distance'}), (26, 'alienigena', {})]),
    'bateria_baja': dict(label='Batería inicial 25 %', events=[], battery=25.),
}
METRICS = ['success', 'collisions', 'cliffs_avoided', 'hazards_avoided', 'battery_left', 'distance',
           'science', 'replans', 'safety_violations', 'decision_ms', 'steps']


def run_one(policy, seed, scenario='estandar', **overrides):
    spec = SCENARIOS[scenario]
    config = dict(policy=policy, seed=seed, battery=spec.get('battery', 100.), hidden_cliffs=6, difficulty='medium')
    config.update(overrides)
    mission = HybridMission(**config)
    schedule = sorted(spec['events'], key=lambda e: e[0])
    started = time.perf_counter()
    while not mission.finished and mission.step_index < 400:
        for at, kind, data in schedule:
            if mission.step_index == at:
                try:
                    mission.inject(kind, **dict(data))
                except ValueError:
                    pass  # event not applicable in this state (e.g. no free cell)
        mission.step()
    row = mission.summary()
    row.update(scenario=scenario, wall_s=round(time.perf_counter() - started, 3))
    return row


def run_experiments(policies=None, seeds=20, scenario='estandar', first_seed=0, progress=None, **overrides):
    policies = list(policies or POLICIES)
    unknown = set(policies) - set(POLICIES)
    if unknown:
        raise ValueError(f'Políticas desconocidas: {sorted(unknown)}')
    if not isinstance(seeds, int) or not 1 <= seeds <= 200:
        raise ValueError('Cantidad de semillas inválida (1–200)')
    rows = []
    for policy in policies:
        for seed in range(first_seed, first_seed + seeds):
            rows.append(run_one(policy, seed, scenario, **overrides))
            if progress:
                progress(len(rows), len(policies) * seeds)
    return pd.DataFrame(rows)


def summarize(frame):
    """Mean, median and standard deviation per policy (dispersion is reported)."""
    frame = frame.copy()
    frame['success'] = frame['success'].astype(float)
    grouped = frame.groupby('policy')[METRICS]
    table = pd.concat({'media': grouped.mean(), 'mediana': grouped.median(), 'desv': grouped.std(ddof=1).fillna(0.)}, axis=1)
    order = [p for p in POLICIES if p in table.index]
    return table.loc[order]


def summary_records(frame):
    table = summarize(frame)
    records = []
    for policy, row in table.iterrows():
        entry = {'policy': policy, 'label': POLICIES[policy]['label'], 'n': int((frame['policy'] == policy).sum())}
        for metric in METRICS:
            entry[metric] = {stat: float(np.round(row[(stat, metric)], 4)) for stat in ('media', 'mediana', 'desv')}
        records.append(entry)
    return records


In [ ]:
%%writefile ares_x/src/ares/api.py
"""JSON command surface shared by the Colab bridge, the local server and Pyodide.

Every UI action goes through ``Session.dispatch(fn, args)`` so the dashboard is
just a view: the AI engine and the simulation live in Python.
"""
import json
import math

import numpy as np

from .experiments import SCENARIOS, run_experiments, summary_records
from .fuzzy import FuzzyController
from .logger import clean
from .mission import EVENT_LABELS, POLICIES, HybridMission, shared_perception, validate_config
from .neural import FEATURES
from .planet import ASSUMPTION, PROFILES, PlanetProfile
from .supervisor import REASONS

VERSION = '1.0.0'
GHOST_POLICIES = ('hibrido', 'sin_rna', 'sin_difuso', 'sin_ag')


def _r(value, digits=3):
    if isinstance(value, (float, np.floating)):
        return round(float(value), digits) if math.isfinite(value) else None
    return value


def _cells(cells):
    return sorted([int(r), int(c)] for r, c in cells)


def fuzzy_sets():
    """Sampled membership curves for every linguistic variable (for plots)."""
    controller = FuzzyController()
    universes = {'distance': (0, 20), 'battery': (0, 100), 'slope': (0, 45), 'temperature': (-80, 120)}
    out = {}
    for name, sets in controller.membership_sets.items():
        low, high = universes[name]
        xs = np.linspace(low, high, 61)
        out[name] = {'x': [_r(x, 2) for x in xs],
                     'sets': {label: [_r(controller.membership(float(x), spec), 3) for x in xs]
                              for label, spec in sets.items()}}
    xs = np.linspace(0, 1, 61)
    out['speed'] = {'x': [_r(x, 3) for x in xs],
                    'sets': {label: [_r(controller.membership(float(x), spec), 3) for x in xs]
                             for label, spec in controller.output_sets.items()}}
    return out


def fuzzy_payload(result, controller=None):
    controller = controller or FuzzyController()
    descriptions = {rule['id']: rule for rule in controller.rules}
    curve = result.get('aggregated_curve') or []
    return dict(
        speed=_r(result['speed'], 4), caution=_r(result['caution'], 4), centroid=_r(result['centroid'], 4),
        reason=result['reason'], inputs=result.get('inputs'),
        degrees={k: {s: _r(v, 3) for s, v in d.items()} for k, d in result.get('membership_degrees', {}).items()},
        rules=[dict(id=r['id'], strength=_r(r['strength'], 3), description=r['description'], consequent=r['consequent'])
               for r in result.get('activated_rules', [])],
        all_rules=[dict(id=r['id'], description=r['description'], consequent=r['consequent'],
                        antecedents=[list(a) for a in r['antecedents']]) for r in descriptions.values()],
        curve=[_r(v, 3) for v in curve[::4]], fallback='Sin reglas activas: velocidad 0 (parar), cautela 1.')


def ga_view(ga):
    ga = ga or {}
    return dict(mode=ga.get('mode'), reason=ga.get('reason'), chosen=ga.get('chosen'), message=ga.get('message'),
                rejected=ga.get('rejected', []), generations_run=ga.get('generations_run'),
                stopped_early=ga.get('stopped_early'), weights=ga.get('weights'), step=ga.get('step'),
                history=ga.get('history', []),
                candidates=[{k: v for k, v in c.items() if k != 'route'} for c in ga.get('candidates', [])])


class Session:
    def __init__(self, **config):
        self.config = validate_config(config)
        self.ghost_mode = False
        self.ghosts = {}
        self._start()
        self.comparison = None

    def _start(self):
        """(Re)create the mission and, in ghost mode, the other policies on the same map and seed."""
        self.mission = HybridMission(**self.config).prepare()
        others = [p for p in GHOST_POLICIES if p != self.config['policy']][:3] if self.ghost_mode else []
        self.ghosts = {p: HybridMission(**dict(self.config, policy=p)).prepare() for p in others}

    def _ghosts_do(self, action):
        """Mirror an operator action on every ghost; one that cannot take it just skips it."""
        for ghost in self.ghosts.values():
            try:
                action(ghost)
            except ValueError:
                pass

    def ghost_view(self):
        return [dict(policy=policy, label=POLICIES[policy]['label'], position=list(g.position),
                     path=[list(p) for p in g.path], step=g.step_index, state=g.state, finished=g.finished,
                     outcome=g.outcome, battery=_r(g.battery, 1), health=_r(g.health, 1),
                     science=_r(g.world.science_progress, 3), distance=g.metrics['distance'],
                     collisions=g.metrics['collisions'], replans=g.metrics['replans'])
                for policy, g in self.ghosts.items()]

    # ---------------------------------------------------------------- views
    def snapshot(self):
        m = self.mission
        world, belief = m.world, m.belief
        hidden_obstacles = world.obstacles - belief.obstacles
        hidden_cliffs = world.cliffs - belief.cliffs
        ga = m.ga or {}
        route_cells = [tuple(c) for c in (m.route or [])]
        last = m.last
        trace = m.logger.trace
        perception = m.perception
        rna = last.get('rna')
        decision = last.get('decision')
        readings = last.get('readings') or {}
        fuzzy = last.get('fuzzy')
        log = [dict(step=t['step'], state=t['state'], action=t['action'], reason=t['reason'],
                    battery=t['battery'], explanation=t['explanation'], moved=t['moved'],
                    events=[e['label'] + (' (aleatorio)' if e.get('origin') == 'aleatorio' else '') for e in t['events']],
                    rna=(t['rna'] or {}).get('label'), conf=(t['rna'] or {}).get('confidence'),
                    speed=t['crisp_speed'], ga=(t['ga'] or {}).get('best_fitness')) for t in trace[-60:]]
        return clean(dict(
            version=VERSION, operational_label=ASSUMPTION, config=self.config,
            policy_label=POLICIES[self.config['policy']]['label'],
            planet=dict(name=m.profile.name, mode=m.profile.mode, gravity=m.profile.gravity,
                        temperature=m.profile.temperature, pressure=m.profile.pressure,
                        oxygen=m.profile.oxygen, hazards=list(m.profile.hazards)),
            status=dict(step=m.step_index, t=m.elapsed, state=m.state, finished=m.finished, outcome=m.outcome,
                        position=list(m.position), battery=_r(m.battery, 2), health=_r(m.health, 2),
                        daylight=m.daylight, visibility=m.visibility, returning=m.returning,
                        science=_r(world.science_progress, 3), link=m.timers['link'] == 0,
                        instruments=len(m.recovered), instruments_total=len(world.resources),
                        destination_distance=(route_cells.index(m.destination) if m.destination in route_cells else None),
                        destinations_left=len(m.destinations), tour_cells=m.tour_cells,
                        vibration=m.vibration_level, wind=m.wind_speed, heat=m.timers['heat'],
                        link_latency=_r(m.link_latency, 1), queued_orders=m.queued_orders,
                        events_total=sum(m.metrics['events'].values()), events=dict(m.metrics['events']),
                        anomalies=m.metrics['anomalies'], contacts=m.metrics['contacts'],
                        wait_steps=m.metrics['wait_steps'], degraded_steps=m.metrics['degraded_steps'],
                        storm=m.timers['storm'], quake=m.wait_steps,
                        sensor_failures=sorted(m.sensors.failures),
                        temperature_internal=_r(m.temperature_internal, 1)),
            map=dict(width=world.width, height=world.height,
                     elevation=[[_r(v, 2) for v in row] for row in world.elevation.tolist()],
                     obstacles=_cells(belief.obstacles), hidden_obstacles=_cells(hidden_obstacles),
                     cliffs=_cells(belief.cliffs), hidden_cliffs=_cells(hidden_cliffs),
                     science=[[int(r), int(c), (r, c) in world.visited_science] for r, c in sorted(world.science_sites)],
                     refuges=_cells(world.refuges),
                     resources=[[int(r), int(c), (r, c) in m.recovered] for r, c in sorted(world.resources)],
                     risk=[[int(r), int(c), _r(v, 2)] for (r, c), v in sorted(m.risk.items()) if v >= .15],
                     path=[list(p) for p in m.path], route=[list(p) for p in (m.route or [])],
                     candidates=[c['route'] for c in ga.get('candidates', [])][:3],
                     chosen=ga.get('chosen'), destination=list(m.destination) if m.destination else None,
                     destinations=[list(d) for d in m.destinations], tour=[list(p) for p in m.destination_tour()],
                     events=[dict(kind=e['kind'], position=e['position']) for e in m.event_log if e.get('position')]),
            telemetry=dict(
                distance=_r(readings.get('distance')), elevation_delta=_r(readings.get('elevation_delta')),
                slope=_r(readings.get('slope')), temperature_external=_r(readings.get('temperature_external'), 1),
                radiation=_r(readings.get('radiation')), pressure=_r(readings.get('pressure'), 1),
                oxygen_ambient=_r(readings.get('oxygen_ambient')), oxygen_stored=_r(readings.get('oxygen_stored')),
                co2=_r(readings.get('co2')), solar_power=_r(readings.get('solar_power')),
                nav_valid=last.get('nav_valid'), edge_cliff=(trace[-1]['edge_cliff'] if trace else None)),
            decision=decision,
            rna=None if rna is None else dict(
                label=rna['label'], confidence=_r(rna['confidence'], 4), low_confidence=rna['low_confidence'],
                probabilities={k: _r(v, 4) for k, v in rna['probabilities'].items()}, cell=rna.get('cell'),
                features={k: _r(v, 3) for k, v in rna['features'].items()},
                normalized=[_r(v, 3) for v in rna['normalized_input']],
                layers=[{k: _r(v, 3) for k, v in s.items()} for s in rna['activation_summaries']]),
            rna_error=(trace[-1]['rna_error'] if trace else None),
            fuzzy=None if fuzzy is None else fuzzy_payload(fuzzy, m.fuzzy),
            speed=_r(last.get('speed'), 4),
            ga=ga_view(ga),
            ga_runs=[dict(index=i, reason=r['ga']['reason'], step=r['step'],
                          generations=r['ga'].get('generations_run')) for i, r in enumerate(m.ga_runs)],
            metrics=m.summary(),
            series=dict(battery=[t['battery'] for t in trace][-200:],
                        risk=[_r(1 - (t['rna'] or {}).get('probabilities', {}).get('libre', 1.), 3) for t in trace][-200:],
                        speed=[t['crisp_speed'] for t in trace][-200:]),
            log=log,
            nn=dict(test=perception.report_['test'], validation=perception.report_['validation'],
                    converged=perception.report_['converged'], iterations=perception.report_['iterations'],
                    training_samples=perception.report_['training_samples'],
                    split_maps=perception.report_['split_maps'],
                    train_seconds=_r(getattr(perception, 'train_seconds_', None), 2)),
            drone=m.drone, ghosts=self.ghost_view(),
            policy_uses={k: m.policy[k] for k in ('rna', 'fuzzy', 'ga')},
            event_feed=dict(total=len(m.event_log),
                            recent=[dict(step=e['step'], kind=e['kind'], label=e['label'], origin=e.get('origin', 'operador'), message=e.get('message') or '')
                                    for e in m.event_log[-12:]]),
            comparison=self.comparison))

    def meta(self):
        model = shared_perception(0).model
        network = dict(layers=[len(FEATURES)] + list(model.hidden_layer_sizes) + [len(model.classes_)],
                       classes=list(model.classes_),
                       coefs=[[[_r(v, 3) for v in row] for row in layer.tolist()] for layer in model.coefs_],
                       intercepts=[[_r(v, 3) for v in layer.tolist()] for layer in model.intercepts_])
        training = [{k: _r(v, 4) for k, v in row.items()} for row in shared_perception(0).report_['history']]
        return clean(dict(version=VERSION, network=network, training=training, planets=[dict(key=k, name=v[0], rocky=v[1]) for k, v in PROFILES.items()],
                          policies={k: v['label'] for k, v in POLICIES.items()}, events=EVENT_LABELS,
                          scenarios={k: v['label'] for k, v in SCENARIOS.items()}, reasons=REASONS,
                          features=list(FEATURES), fuzzy_sets=fuzzy_sets(), operational_label=ASSUMPTION))

    # ------------------------------------------------------------- commands
    def dispatch(self, fn, args=None):
        args = dict(args or {})
        try:
            handler = getattr(self, 'cmd_' + str(fn))
        except AttributeError:
            return {'error': f'Comando desconocido: {fn}'}
        try:
            return handler(**args)
        except (ValueError, TypeError, KeyError) as error:
            message = error.args[0] if error.args else str(error)
            return {'error': str(message)}

    def cmd_meta(self):
        return self.meta()

    def cmd_state(self):
        return self.snapshot()

    def cmd_configure(self, **config):
        merged = dict(self.config)
        merged.update(config)
        config = validate_config(merged)  # raises readable ValueError (CA-07, NaN sliders)
        self.config = config
        self._start()
        return self.snapshot()

    def cmd_validate(self, **config):
        merged = dict(self.config)
        merged.update(config)
        validate_config(merged)
        return {'ok': True}

    def cmd_reset(self):
        self._start()
        return self.snapshot()

    def cmd_drone(self, enabled=True):
        """Scout drone on or off in the running mission (and every ghost), without restarting."""
        self.mission.set_drone(enabled)
        self.config = dict(self.config, drone=bool(enabled))   # kept on reset and new missions
        self._ghosts_do(lambda ghost: ghost.set_drone(enabled))
        return self.snapshot()

    def cmd_ghosts(self, enabled=True):
        """Ghost race on or off. Runners must start together, so the mission restarts."""
        if not isinstance(enabled, bool):
            raise ValueError('enabled: debe ser verdadero o falso')
        self.ghost_mode = enabled
        self._start()
        return self.snapshot()

    def cmd_step(self, n=1):
        n = int(n)
        if not 1 <= n <= 400:
            raise ValueError('Pasos inválidos (1–400)')
        for _ in range(n):
            if self.mission.finished and all(g.finished for g in self.ghosts.values()):
                break
            for runner in [self.mission, *self.ghosts.values()]:
                if not runner.finished:
                    runner.step()
        return self.snapshot()

    def cmd_inject(self, kind, **data):
        self.mission.inject(kind, **data)
        self._ghosts_do(lambda ghost: ghost.inject(kind, **data))
        return self.snapshot()

    def cmd_destination(self, position=None):
        """Add a destination; the same cell again removes it; no position clears them all."""
        m = self.mission
        if position is not None and tuple(int(v) for v in position) in m.destinations:
            m.remove_destination(position)
            self._ghosts_do(lambda ghost: ghost.remove_destination(position))
        else:
            m.set_destination(position)
            self._ghosts_do(lambda ghost: ghost.set_destination(position))
        return self.snapshot()

    def cmd_place(self, position, kind='obstaculo'):
        """Operator edits the real terrain (RF-02); the rover must discover it."""
        self._place(self.mission, position, kind)
        self._ghosts_do(lambda ghost: self._place(ghost, position, kind))
        return self.snapshot()

    @staticmethod
    def _place(m, position, kind):
        cell = tuple(int(v) for v in position)
        if not m.world.in_bounds(cell):
            raise ValueError('Celda fuera del mapa')
        if cell == m.position or cell in m.world.refuges:
            raise ValueError('No se puede colocar sobre el rover o un refugio')
        if kind == 'barranco':
            m.inject('barranco', position=list(cell))
        elif kind == 'obstaculo':
            m.world.add_obstacle(cell)
            m.event_log.append(dict(step=m.step_index, kind='obstaculo_manual', label='Obstáculo manual',
                                    message='Obstáculo colocado por el operador (oculto al rover).', position=list(cell)))
        elif kind == 'limpiar':
            if cell in m.world.cliffs:
                m.world.elevation[cell] = m.belief.elevation[cell]  # refill the 10 m drop
            m.risk.pop(cell, None)
            m.rna_marks.discard(cell)
            m.world.obstacles.discard(cell)
            m.world.cliffs.discard(cell)
            m.belief.obstacles.discard(cell)
            m.belief.cliffs.discard(cell)
        else:
            raise ValueError('Tipo de celda desconocido')

    def cmd_fuzzy_eval(self, distance, battery, slope, temperature=20.):
        controller = FuzzyController()
        values = [float(v) for v in (distance, battery, slope, temperature)]
        result = controller.infer(*values)
        result['inputs'] = dict(distance=values[0], battery=values[1], slope=values[2], temperature=values[3])
        return clean(fuzzy_payload(result, controller))

    def cmd_rna_eval(self, **features):
        perception = shared_perception(0)
        values = {name: float(features[name]) for name in FEATURES}
        prediction = perception.predict(values)
        return clean(dict(label=prediction['label'], confidence=_r(prediction['confidence'], 4),
                          low_confidence=prediction['low_confidence'],
                          probabilities={k: _r(v, 4) for k, v in prediction['probabilities'].items()},
                          normalized=[_r(v, 3) for v in prediction['normalized_input']],
                          layers=[{k: _r(v, 3) for k, v in s.items()} for s in prediction['activation_summaries']],
                          risk=_r(prediction['risk'], 4)))

    def cmd_ga_generation(self, generation=0, run=None):
        return clean(self.mission.ga_generation(generation, run))

    def cmd_ga_run(self, run=None):
        index, record = self.mission.ga_run(run)
        view = ga_view(record['ga'])
        view['routes'] = [c['route'] for c in record['ga'].get('candidates', [])]  # for the minimap
        return clean(dict(run=index, ga=view))

    def cmd_compare(self, seeds=5, scenario='estandar', policies=None):
        if scenario not in SCENARIOS:
            raise ValueError('Escenario desconocido')
        frame = run_experiments(policies, int(seeds), scenario, planet=self.config['planet'], mode=self.config['mode'])
        self.comparison = dict(scenario=scenario, label=SCENARIOS[scenario]['label'], seeds=int(seeds),
                               summary=summary_records(frame))
        self._comparison_frame = frame
        return clean(self.comparison)

    def cmd_export(self, format='json'):
        if format == 'csv':
            return {'filename': f"ares_x_traza_semilla{self.config['seed']}.csv", 'content': self.mission.export_csv()}
        if format == 'comparison_csv':
            frame = getattr(self, '_comparison_frame', None)
            if frame is None:
                raise ValueError('Primero ejecute una comparación de políticas')
            return {'filename': 'ares_x_comparacion.csv', 'content': frame.to_csv(index=False)}
        payload = json.dumps(self.mission.export_json(), ensure_ascii=False, indent=1)
        return {'filename': f"ares_x_mision_semilla{self.config['seed']}.json", 'content': payload}


def handle_json(session, fn, args_json):
    """String-in/string-out entry point (Colab callback and Pyodide)."""
    try:
        args = json.loads(args_json) if args_json else {}
    except json.JSONDecodeError:
        return json.dumps({'error': 'Argumentos JSON inválidos'})
    return json.dumps(session.dispatch(fn, args), ensure_ascii=False, allow_nan=False)


In [ ]:
%%writefile ares_x/src/ares/dashboard.py
"""Single-screen dashboard bridge for Google Colab (and plain Jupyter).

In Colab the HTML panel calls ``google.colab.kernel.invokeFunction('ares_api')``
and every action is executed by the Python engine in this kernel. Outside Colab
the same panel is served by a local HTTP server and embedded in an IFrame.
"""
import json
import threading
from pathlib import Path

from .api import Session, handle_json

ROOT = Path(__file__).resolve().parents[2]


class Dashboard:
    def __init__(self, session=None, height=880, **config):
        config.setdefault('event_rate', .03)  # F3: the panel starts with 3 % random events
        self.session = session or Session(**config)
        self.height = int(height)

    def html(self):
        page = (ROOT / 'web' / 'index.html').read_text(encoding='utf-8')
        # Inside a notebook output the viewport is the output frame: fix its height.
        return page.replace('--app-h:100vh;', f'--app-h:{self.height}px;', 1)

    def _callback(self, fn, args_json):
        from IPython.display import JSON
        return JSON(json.loads(handle_json(self.session, fn, args_json)))

    def show(self):
        from IPython.display import HTML, IFrame, display
        try:
            from google.colab import output
        except ImportError:
            port = self.serve()
            display(IFrame(f'http://127.0.0.1:{port}/', width='100%', height=self.height + 10))
            return
        output.register_callback('ares_api', self._callback)
        display(HTML(self.html()))

    def serve(self, port=0):
        """Serve this session locally (non-Colab notebooks)."""
        from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
        session, page, lock = self.session, (ROOT / 'web' / 'index.html').read_text(encoding='utf-8'), threading.Lock()

        class Handler(BaseHTTPRequestHandler):
            def _send(self, body, content_type):
                data = body.encode('utf-8')
                self.send_response(200)
                self.send_header('Content-Type', content_type)
                self.send_header('Content-Length', str(len(data)))
                self.end_headers()
                self.wfile.write(data)

            def do_GET(self):
                self._send(page, 'text/html; charset=utf-8')

            def do_POST(self):
                request = json.loads(self.rfile.read(int(self.headers.get('Content-Length') or 0)) or b'{}')
                with lock:
                    body = handle_json(session, request.get('fn'), json.dumps(request.get('args') or {}))
                self._send(body, 'application/json; charset=utf-8')

            def log_message(self, *args):
                pass

        server = ThreadingHTTPServer(('127.0.0.1', port), Handler)
        threading.Thread(target=server.serve_forever, daemon=True).start()
        return server.server_address[1]


Panel HTML (vista). La lógica no vive aquí: cada acción llama al motor Python mediante `google.colab.kernel.invokeFunction`.

In [ ]:
%%writefile ares_x/web/index.html
<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>ARES-X 3D</title>
<link rel="icon" href="data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 32 32'%3E%3Ccircle cx='16' cy='16' r='14' fill='%23c1440e'/%3E%3Ccircle cx='11' cy='12' r='3' fill='%238a2f0a'/%3E%3Ccircle cx='20' cy='20' r='4' fill='%238a2f0a'/%3E%3C/svg%3E">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Barlow+Condensed:wght@500;600;700&family=Barlow:wght@400;500;600&family=JetBrains+Mono:wght@400;600&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<style>
:root{
  --app-h:100vh;
  --bg:#0a0705;--panel:#130e0a;--panel2:#1b140f;--line:#2b2119;--line2:#3d2f24;
  --text:#ecdfcd;--muted:#a08d78;--dim:#6f5f50;
  --accent:#ff9a3c;--amber:#ffc04d;--ok:#52d58e;--warn:#ffc04d;--bad:#ff5a4f;--blue:#5aa9ff;--violet:#b58cff;--sci:#ffd84a;
  --head:"Barlow Condensed","Arial Narrow",sans-serif;--font:"Barlow","Segoe UI",system-ui,sans-serif;--mono:"JetBrains Mono",Consolas,monospace;
}
*{box-sizing:border-box}
html,body{margin:0;height:100%;background:var(--bg);color:var(--text);font:12px/1.35 var(--font);overflow:hidden}
button,select,input{font:inherit;color:inherit}
#app{height:var(--app-h);min-height:600px;display:grid;grid-template-rows:minmax(0,1.42fr) minmax(0,1fr);gap:6px;padding:6px}
.top{display:grid;grid-template-columns:clamp(200px,15.5vw,260px) minmax(0,1fr) clamp(240px,18vw,300px);gap:6px;min-height:0}
.bottom{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:6px;min-height:0}
.panel{background:var(--panel);border:1px solid var(--line);border-radius:6px;display:flex;flex-direction:column;min-height:0;min-width:0;overflow:hidden}
.ph{display:flex;align-items:center;gap:7px;padding:6px 9px;border-bottom:1px solid var(--line);flex:none}
.ph .ic{width:16px;height:16px;border-radius:3px;background:var(--accent);color:#1a0d02;font:700 11px/16px var(--head);text-align:center;flex:none}
.ph h2{margin:0;font:700 13.5px var(--head);letter-spacing:.09em;text-transform:uppercase;white-space:nowrap}
.ph .sub{margin-left:auto;font:500 10.5px var(--mono);color:var(--muted);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.pb{padding:7px 9px;overflow:auto;min-height:0;flex:1;scrollbar-width:thin;scrollbar-color:var(--line2) transparent}
.lbl{font:600 10.5px var(--head);letter-spacing:.12em;text-transform:uppercase;color:var(--muted);margin:8px 0 4px}
.lbl:first-child{margin-top:0}
select,input[type=number]{background:var(--panel2);border:1px solid var(--line2);border-radius:4px;padding:2px 5px;width:100%;min-width:0}
input[type=range]{width:100%;accent-color:var(--amber);margin:0;height:14px}
.btn{background:var(--panel2);border:1px solid var(--line2);border-radius:4px;padding:4px 6px;cursor:pointer;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;text-align:left}
.btn:hover{border-color:var(--accent);color:#fff}.btn:active{transform:translateY(1px)}
.btn.c{text-align:center}
.btn.on,.btn.sel{border-color:var(--accent);color:var(--accent);background:#2a1a0d}
.btn.go{background:#1e3a26;border-color:#2f7a4a;color:#c9f5d9;text-align:center}
.g2{display:grid;grid-template-columns:1fr 1fr;gap:4px}.g3{display:grid;grid-template-columns:repeat(3,1fr);gap:4px}.g4{display:grid;grid-template-columns:repeat(4,1fr);gap:4px}
.small{font-size:10.5px;color:var(--muted)}
.mono{font-family:var(--mono)}
/* telemetry */
.tl{display:grid;grid-template-columns:minmax(0,1fr) auto 58px;gap:2px 6px;align-items:center;padding:5px 0 5px 7px;border-left:2px solid var(--ok);border-bottom:1px solid #1f1812}
.tl.warn{border-left-color:var(--warn)}.tl.bad{border-left-color:var(--bad)}.tl.ref{border-left-color:var(--dim)}
.tl .n{font-weight:600}.tl .u{font-size:10px;color:var(--dim)}
.tl .v{font:600 12.5px var(--mono);text-align:right;white-space:nowrap}
.chip{font:600 9px var(--mono);padding:1px 4px;border-radius:3px;text-align:center;border:1px solid #2f6a46;color:var(--ok);letter-spacing:.03em}
.chip.warn{border-color:#6e5420;color:var(--warn)}.chip.bad{border-color:#6e2622;color:var(--bad)}.chip.ref{border-color:var(--line2);color:var(--muted)}
.planet{display:flex;gap:9px;align-items:center;margin-bottom:6px}
.planet canvas{width:56px;height:56px;flex:none}
.planet .pn{font:700 20px var(--head);letter-spacing:.06em;text-transform:uppercase}
/* 3D */
#view{position:relative;padding:0;border-color:#3a2a1d}
#scene{position:absolute;inset:0;width:100%;height:100%;display:block;cursor:grab}
.ov{position:absolute;z-index:2;background:rgba(14,9,6,.78);border:1px solid var(--line2);border-radius:5px;backdrop-filter:blur(3px)}
#ovTop{left:8px;top:8px;display:flex;gap:6px;padding:4px 8px;align-items:center;font:600 11px var(--mono)}
#ovTop b{font:700 13px var(--head);letter-spacing:.1em;color:var(--accent)}
#ovCam{right:8px;top:8px;display:flex;gap:3px;padding:3px}
#ovCam .btn{padding:2px 7px;font-size:10.5px}
#ovCam .seg button{padding:3px 9px;font-weight:600}
#map2d{position:absolute;inset:0;width:100%;height:100%;display:none;cursor:crosshair;background:#0d0906}
#view.picking #scene,#view.picking #map2d{cursor:crosshair}
#ovState{left:8px;bottom:38px;padding:7px 10px;max-width:min(470px,60%)}
#ovState .st{font:700 17px var(--head);letter-spacing:.12em}
#ovState .ex{font-size:11px;color:var(--muted);margin-top:2px}
#ovLegend{right:8px;bottom:38px;padding:6px 9px;display:grid;grid-template-columns:repeat(3,auto);gap:3px 11px;font-size:10px;color:var(--muted)}
#ovLegend i{display:inline-block;width:9px;height:9px;border-radius:2px;margin-right:4px;vertical-align:-1px}
.st-EXPLORACION{color:var(--ok)}.st-ALERTA,.st-REPLANIFICACION{color:var(--warn)}.st-REFUGIO{color:var(--blue)}
.st-MODO_SEGURO,.st-FRACASO{color:var(--bad)}.st-EXITO{color:var(--ok)}.st-CONFIGURACION{color:var(--muted)}
#alerts{position:absolute;z-index:6;top:44px;left:50%;transform:translateX(-50%);width:min(390px,38%);display:flex;flex-direction:column;gap:4px;pointer-events:none}
.alert{pointer-events:auto;display:flex;gap:8px;align-items:flex-start;padding:6px 8px 6px 10px;border-radius:5px;border:1px solid var(--line2);border-left:4px solid var(--blue);background:rgba(14,9,6,.9);backdrop-filter:blur(3px);font-size:11px;line-height:1.35;animation:alertIn .22s ease-out}
.alert b{display:block;font:700 12px var(--head);letter-spacing:.07em;text-transform:uppercase}
.alert.warn{border-left-color:var(--warn)}.alert.warn b{color:var(--warn)}
.alert.crit{border-left-color:var(--bad);border-color:var(--bad)}.alert.crit b{color:var(--bad)}
.alert.ok{border-left-color:var(--ok)}.alert.ok b{color:var(--ok)}
.alert div{flex:1;min-width:0}.alert button{background:none;border:none;color:var(--muted);cursor:pointer;font-size:14px;line-height:1;padding:0 2px}
@keyframes alertIn{from{opacity:0;transform:translateY(-8px)}to{opacity:1;transform:none}}
@media (prefers-reduced-motion: reduce){.alert{animation:none}}
#ovTime{left:8px;right:8px;bottom:6px;display:flex;gap:6px;align-items:center;padding:2px 8px;font:600 10.5px var(--mono)}
#ovTime input{flex:1;min-width:60px;margin:0}
#ovTime .btn{padding:1px 8px;font-size:10.5px}
#ovTime.replay{border-color:var(--accent)}
#ovWhy{left:8px;top:44px;max-width:min(340px,46%);padding:7px 10px;font-size:11px;line-height:1.45;border-color:var(--accent);display:none}
#ovWhy b{color:var(--accent);font:700 12px var(--head);letter-spacing:.08em}
#ovRace{right:8px;top:44px;padding:6px 9px;font-size:10.5px;display:none}
#ovRace .rh{font:700 11px var(--head);letter-spacing:.1em;color:var(--accent);margin-bottom:3px;text-transform:uppercase}
#ovRace .rr{display:grid;grid-template-columns:10px minmax(80px,auto) 36px 36px auto;gap:6px;align-items:center;padding:1px 0}
#ovRace .rr i{width:9px;height:9px;border-radius:50%;display:inline-block}
#ovRace .rr span{color:var(--muted);text-align:right}
#travel{position:absolute;inset:0;width:100%;height:100%;display:none;z-index:30;cursor:pointer;background:#03040a}
#nogl{position:absolute;inset:0;display:none;align-items:center;justify-content:center;color:var(--muted)}
/* right */
.pbtn{font:600 11px var(--head);letter-spacing:.08em;text-transform:uppercase;text-align:center;padding:5px 2px}
.ev{font-size:11px}
.msg{font-size:11px;padding:5px 7px;border-radius:4px;background:#0f0b08;border:1px solid var(--line);color:var(--muted);margin-top:6px}
.msg.err{color:#ffd2cc;border-color:#6e2622;background:#1d0c0a}
.row{display:flex;gap:4px;align-items:center}.row>*{flex:1}
.kv2{display:grid;grid-template-columns:auto 1fr;gap:2px 6px;align-items:center}
/* bottom */
canvas.fill{display:block;width:100%}
.mt{border-collapse:collapse;width:100%;font:10.5px var(--mono)}
.mt td,.mt th{border-bottom:1px solid #221a13;padding:2px 4px;text-align:left;white-space:nowrap}
.mt th{color:var(--dim);font-weight:600;position:sticky;top:0;background:var(--panel)}
.mt td.x{white-space:normal;font-family:var(--font);color:var(--muted)}
.r-veto{color:var(--bad)}.r-warn{color:var(--warn)}.r-ok{color:var(--ok)}
.rule{display:grid;grid-template-columns:28px 1fr 38px;gap:5px;padding:3px 5px;border-radius:3px;margin:2px 0;font-size:10.5px;background:#17110c}
.rule.act{background:#2b1c0c;outline:1px solid #6e4a1d}
.rule b{font-family:var(--mono);color:var(--amber)}
.feat{display:grid;grid-template-columns:repeat(6,1fr);gap:3px}
.feat label{font-size:9.5px;color:var(--dim);display:block;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.feat input{padding:1px 3px;font:10.5px var(--mono)}
.prob{display:grid;grid-template-columns:62px 1fr 42px;gap:5px;align-items:center;margin:2px 0;font-size:11px}
.prob .t{height:9px;background:#221a13;border-radius:2px;overflow:hidden}.prob .t span{display:block;height:100%}
.seg{display:inline-flex;border:1px solid var(--line2);border-radius:4px;overflow:hidden}
.seg button{background:none;border:none;padding:1px 7px;font-size:10.5px;cursor:pointer;color:var(--muted)}
.seg button.on{background:#2a1a0d;color:var(--accent)}
.sl{display:grid;grid-template-columns:1fr 46px;gap:2px 6px;align-items:center;font-size:10.5px;margin:1px 0}
.sl input{grid-column:1/3}
.sl span{text-align:right;font-family:var(--mono);color:var(--amber)}
.gaTop{display:grid;grid-template-columns:minmax(0,1fr) minmax(0,1fr);gap:6px}
.pill{display:inline-block;padding:0 5px;border-radius:3px;font-size:9.5px;border:1px solid var(--line2)}
.pill.ok{color:var(--ok);border-color:#2f6a46}.pill.bad{color:var(--bad);border-color:#6e2622}
</style>
</head>
<body>
<div id="app">
 <div class="top">
  <section class="panel">
   <div class="ph"><span class="ic">A</span><h2>Telemetría</h2><span class="sub" id="tSub">—</span></div>
   <div class="pb">
    <div class="planet"><canvas id="planetIcon" width="112" height="112"></canvas>
      <div><div class="pn" id="pName">—</div><div class="small" id="pInfo">—</div></div></div>
    <div class="small" id="pNote" style="margin-bottom:6px">Valores de referencia aproximados (NASA); todo valor operacional es supuesto de simulación.</div>
    <div id="tele"></div>
   </div>
  </section>

  <section class="panel" id="view">
   <canvas id="scene"></canvas><canvas id="map2d"></canvas><canvas id="travel" title="Clic para saltar la animación"></canvas><div id="nogl">WebGL no disponible en este navegador.</div>
   <div class="ov" id="ovTop"><b>ARES-X 3D</b><span id="oPlanet">—</span><span id="oStep">—</span><span id="oPolicy" class="small">—</span></div>
   <div class="ov" id="ovCam">
     <span class="seg" title="Cambiar la vista del mapa"><button class="on" data-view="3d">3D</button><button data-view="2d">2D</button></span>
     <button class="btn c on" data-cam="follow">Seguir rover</button><button class="btn c" data-cam="orbit">Vista general</button><button class="btn c" data-cam="top">Cenital</button>
     <label class="btn c" title="Muestra peligros que el rover aún no descubrió"><input type="checkbox" id="fReveal" style="vertical-align:-2px;margin:0 3px 0 0">ocultos</label>
   </div>
   <div class="ov" id="ovState"><div class="st" id="sState">EN ESPERA</div><div class="ex" id="sExpl">Configure la misión y presione Iniciar.</div></div>
   <div id="alerts" role="status" aria-live="polite"></div>
   <div class="ov" id="ovWhy"></div>
   <div class="ov" id="ovRace" title="Al terminar se ordena por: misión cumplida, ciencia, batería restante y colisiones"></div>
   <div class="ov" id="ovTime"><button class="btn c" id="tPrev" title="Paso anterior">◀</button><input type="range" id="tSlider" min="0" max="0" value="0" title="Línea de tiempo: arrastre para repetir la misión"><button class="btn c" id="tNext" title="Paso siguiente">▶</button><span id="tLabel">paso 0/0</span><button class="btn c on" id="tLive" title="Volver al presente">● En vivo</button></div>
   <div class="ov" id="ovLegend">
     <span><i style="background:#52d58e"></i>Despejado</span><span><i style="background:#ffc04d"></i>Obstáculo</span><span><i style="background:#e8e05a"></i>Pendiente</span>
     <span><i style="background:#ff5a4f"></i>Barranco</span><span><i style="background:#5aa9ff"></i>Turbulencia</span><span><i style="background:#d07bff"></i>Impacto</span>
     <span><i style="background:#ff9a3c"></i>Riesgo RNA</span><span><i style="background:#ffd84a;height:3px"></i>Ruta AG</span><span><i style="background:#5aa9ff;height:3px"></i>Recorrido</span>
     <span><i style="background:#ff4fd8"></i>Destino</span><span><i style="background:#fff;border-radius:50%"></i>Edición operador</span><span><i style="background:#22d3ee"></i>Dron</span>
   </div>
  </section>

  <section class="panel">
   <div class="ph"><span class="ic">C</span><h2>Control de misión</h2><span class="sub" id="cSeed">—</span></div>
   <div class="pb">
    <div class="g4" id="planets"></div>
    <div class="small" style="margin-top:4px" id="planetHint">Gigantes gaseosos: solo sonda atmosférica virtual.</div>
    <div class="lbl">Configuración</div>
    <div class="g2">
      <div><div class="small">Modo</div><select id="fMode"><option value="rover">Rover (ruedas)</option><option value="sonda">Sonda atmosférica</option></select></div>
      <div><div class="small">Política</div><select id="fPolicy"></select></div>
      <div><div class="small">Semilla</div><input id="fSeed" type="number" min="0" step="1" value="42"></div>
      <div><div class="small">Dificultad</div><select id="fDiff"><option value="low">Baja</option><option value="medium" selected>Media</option><option value="high">Alta</option></select></div>
    </div>
    <div class="sl" style="margin-top:4px">Batería inicial<span id="lBat">100 %</span><input id="fBattery" type="range" min="0" max="100" step="1" value="100"></div>
    <div class="sl">Duración de paso (tiempo y energía por ciclo)<span id="lDur">1.0 s</span><input id="fDur" type="range" min="0.5" max="3" step="0.5" value="1"></div>
    <div class="sl" title="Probabilidad de un evento al azar en cada paso, según el planeta">Eventos aleatorios<span id="lEv">3 %</span><input id="fEvents" type="range" min="0" max="10" step="1" value="3"></div>
    <button class="btn c" id="bApply" style="width:100%;margin-top:4px">Aplicar configuración</button>
    <div class="lbl">Simulación</div>
    <div class="g3"><button class="btn go" id="bRun">▶ Iniciar</button><button class="btn c" id="bStep">Paso</button><button class="btn c" id="bReset">Reiniciar</button></div>
    <div class="g2" style="margin-top:4px"><button class="btn c" id="bGhost" title="Las otras políticas corren el mismo mapa y semilla al mismo tiempo">Carrera de fantasmas</button><button class="btn c" id="bDrone" title="Un dron vuela adelante y comparte los peligros que ve">Dron explorador</button></div>
    <div class="sl" style="margin-top:4px">Velocidad del modo automático<span id="lSpeed">×1</span><input id="fSpeed" type="range" min="0" max="5" step="1" value="2"></div>
    <div class="lbl">Motor de eventos</div>
    <div class="g2" id="events"></div>
    <div class="lbl">Destinos del rover</div>
    <div class="g2"><button class="btn c" id="bDest">◎ Agregar destinos</button><button class="btn c" id="bDestClear">Quitar todos</button></div>
    <div class="small" id="destInfo" style="margin-top:3px"></div>
    <div class="lbl">Editar terreno real (clic en el mapa)</div>
    <select id="fTool"><option value="">Sin edición</option><option value="obstaculo">Colocar obstáculo</option><option value="barranco">Colocar barranco</option><option value="limpiar">Limpiar celda</option></select>
    <div class="lbl">Exportar</div>
    <div class="g3"><button class="btn c" data-export="csv">Traza CSV</button><button class="btn c" data-export="json">Misión JSON</button><button class="btn c" data-export="comparison_csv">Comparación</button></div>
    <div class="msg" id="msg">Listo.</div>
   </div>
  </section>
 </div>

 <div class="bottom">
  <section class="panel">
   <div class="ph"><span class="ic">D</span><h2>Red neuronal</h2><span class="sub" id="nnSub">—</span></div>
   <div class="pb">
    <div class="row" style="justify-content:space-between;margin-bottom:3px"><span class="small" id="nnLayers">Entrada (6) · Oculta 1 (32) · Oculta 2 (16) · Salida (3)</span>
      <span class="seg" style="flex:none"><button class="on" data-nn="live">En vivo</button><button data-nn="test">Prueba</button></span></div>
    <canvas id="nnNet" class="fill" style="height:150px"></canvas>
    <div class="lbl">Entradas · celda siguiente <span class="small" style="text-transform:none;letter-spacing:0" id="nnCell"></span></div>
    <div class="feat" id="nnFeat"></div>
    <div class="lbl">Salida softmax</div>
    <div id="nnProbs"></div>
    <div class="small" id="nnDecision" style="margin-top:3px"></div>
    <div class="lbl">Evaluación en test (mapas no vistos)</div>
    <div class="small mono" id="nnMetrics"></div>
    <canvas id="nnConf" class="fill" style="height:74px;margin-top:4px"></canvas>
    <div class="lbl">Entrenamiento · pérdida y exactitud por iteración</div>
    <canvas id="nnCurve" class="fill" style="height:92px"></canvas>
    <input type="range" id="fIter" min="1" max="1" value="1">
    <div class="small mono" id="nnIter"></div>
   </div>
  </section>

  <section class="panel">
   <div class="ph"><span class="ic">E</span><h2>Lógica difusa · Mamdani</h2><span class="sub" id="fzSub">—</span></div>
   <div class="pb">
    <div class="row" style="justify-content:space-between"><span class="lbl" style="margin:0">1 · Fusificación</span>
      <span class="seg" style="flex:none"><button class="on" data-fz="live">En vivo</button><button data-fz="test">Prueba (clic en curvas)</button></span></div>
    <canvas id="fzSets" class="fill" style="height:250px;cursor:crosshair"></canvas>
    <div class="lbl">2 · Agregación (máx) y centroide</div>
    <canvas id="fzAgg" class="fill" style="height:80px"></canvas>
    <div class="lbl">3 · Reglas y fuerza de activación (Y = mínimo)</div>
    <div id="fzRules"></div>
   </div>
  </section>

  <section class="panel">
   <div class="ph"><span class="ic">F</span><h2>Algoritmo genético</h2><span class="sub" id="gaSub">—</span></div>
   <div class="pb">
    <div class="gaTop">
      <canvas id="gaMap" class="fill" style="height:118px"></canvas>
      <div><canvas id="gaFit" class="fill" style="height:84px"></canvas>
        <div class="small" id="gaInfo" style="margin-top:2px"></div></div>
    </div>
    <div class="row" style="margin-top:5px"><select id="fRun" style="flex:1.4"></select><div class="small" style="flex:none">gen <b id="gaGenLbl" class="mono">—</b></div></div>
    <input type="range" id="fGen" min="0" max="0" value="0" style="margin-top:3px">
    <div class="small" id="gaOps" style="margin:2px 0"></div>
    <div id="gaChrom"></div>
    <div class="lbl">Rutas candidatas · desglose de aptitud</div>
    <div id="gaCand"></div>
    <div class="lbl">Parámetros</div>
    <div class="g2" id="gaParams"></div>
    <div class="lbl">Pesos del fitness</div>
    <div class="g2" id="gaWeights"></div>
    <button class="btn c" id="bGaApply" style="width:100%;margin-top:4px">Aplicar parámetros y reiniciar misión</button>
   </div>
  </section>

  <section class="panel">
   <div class="ph"><span class="ic">G</span><h2 id="logTitle">Bitácora de decisiones</h2>
     <span class="sub" style="overflow:visible"><span class="seg"><button class="on" data-log="log">Bitácora</button><button data-log="cmp">Comparación</button></span></span></div>
   <div class="pb" id="logBody">
    <div id="logPane">
      <div class="row" style="margin-bottom:4px"><select id="fFilter"><option value="">Todo</option><option value="veto">Vetos y alertas</option><option value="ga">Replanificaciones AG</option><option value="ev">Eventos</option></select></div>
      <div id="trace"></div>
    </div>
    <div id="cmpPane" style="display:none">
      <div class="g3"><select id="fScenario" style="grid-column:1/3"></select><select id="fSeeds"><option>3</option><option selected>5</option><option>10</option><option>20</option></select></div>
      <button class="btn go" id="bCompare" style="width:100%;margin:4px 0">Ejecutar comparación (6 políticas, mismas semillas)</button>
      <div id="cmpTable" class="small">Sin resultados todavía.</div>
      <canvas id="cmpChart" class="fill" style="height:90px;margin-top:5px"></canvas>
    </div>
   </div>
  </section>
 </div>
</div>
<script>
'use strict';
// ------------------------------------------------------------ transport
async function api(fn, args){
  args = args || {};
  if (window.ARES_TRANSPORT) return JSON.parse(await window.ARES_TRANSPORT(fn, JSON.stringify(args)));
  const colab = window.google && google.colab && google.colab.kernel;
  if (colab && colab.invokeFunction){ const r = await colab.invokeFunction('ares_api', [fn, JSON.stringify(args)], {}); return r.data['application/json']; }
  const res = await fetch('api', {method:'POST', headers:{'Content-Type':'application/json'}, body: JSON.stringify({fn, args})});
  return res.json();
}
const $ = id => document.getElementById(id);
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const fmt = (v,d=2) => v===null||v===undefined||Number.isNaN(v)?'—':(typeof v==='number'?v.toFixed(d):v);
let S=null, META=null, running=false, busy=false;
function msg(t,err){ const m=$('msg'); m.textContent=t; m.className='msg'+(err?' err':''); }
async function call(fn,args){
  if (busy && fn==='step') return null; busy=true;
  try{ const out=await api(fn,args); if(out&&out.error){ msg(out.error,true); return null; } if(out&&out.status){ notify(out,S); recordFrame(out); S=out; render(); } return out; }
  catch(e){ msg('Error de comunicación con el motor Python: '+e,true); running=false; syncRun(); return null; }
  finally{ busy=false; }
}
function ctx2d(c){ const r=c.getBoundingClientRect(), dpr=window.devicePixelRatio||1; const w=Math.max(10,Math.floor(r.width)), h=Math.max(10,Math.floor(r.height));
  if(c.width!==w*dpr||c.height!==h*dpr){ c.width=w*dpr; c.height=h*dpr; } const g=c.getContext('2d'); g.setTransform(dpr,0,0,dpr,0,0); g.clearRect(0,0,w,h); return [g,w,h]; }

// ------------------------------------------------------------ planets
const PLANETS={
  'Mercurio':{ground:[0x8d8780,0x5f5a55],sky:0x07070b,fog:0x1a1817,sun:1.35,fogD:.004,icon:['#b7b0a7','#5b5650']},
  'Venus':{ground:[0xd7b06c,0x8b6a3c],sky:0xd9b06a,fog:0xc79a52,sun:.75,fogD:.03,icon:['#f3d9a0','#a8793d']},
  'Tierra':{ground:[0x7d8a4f,0x5e5137],sky:0x8fbde6,fog:0xaecbe3,sun:1.1,fogD:.008,icon:['#6fb2ff','#2f6a3a']},
  'Marte':{ground:[0xc7703f,0x7d3b1f],sky:0xe3b08a,fog:0xd59a70,sun:1.0,fogD:.012,icon:['#f09a5e','#8a3a1c']},
  'Júpiter':{ground:[0xd2ad80,0x9a6a45],sky:0xdcb990,fog:0xd8b48a,sun:.8,fogD:.016,icon:['#e9c9a0','#a0703f'],gas:true},
  'Saturno':{ground:[0xe7d39b,0xb59d62],sky:0xf0deb0,fog:0xeedcae,sun:.8,fogD:.016,icon:['#f7e6b8','#b89a5a'],gas:true},
  'Urano':{ground:[0x9adbe0,0x5aa6ad],sky:0xbfeef1,fog:0xb5e6ea,sun:.65,fogD:.018,icon:['#c8f4f6','#5fb2b9'],gas:true},
  'Neptuno':{ground:[0x4a78d6,0x26428a],sky:0x6f95e8,fog:0x5f86d8,sun:.55,fogD:.018,icon:['#8fb0ff','#2a4aa8'],gas:true},
};
function drawPlanetIcon(name){ const c=$('planetIcon'), g=c.getContext('2d'), p=PLANETS[name]||PLANETS.Marte; g.clearRect(0,0,112,112);
  const gr=g.createRadialGradient(40,38,6,56,56,52); gr.addColorStop(0,p.icon[0]); gr.addColorStop(1,p.icon[1]); g.fillStyle=gr; g.beginPath(); g.arc(56,56,50,0,7); g.fill();
  if(p.gas){ g.globalAlpha=.25; g.fillStyle='#fff'; for(let i=0;i<5;i++){ g.fillRect(8,24+i*14,96,4); } g.globalAlpha=1; }
  if(name==='Saturno'){ g.strokeStyle='#e8d7a8'; g.lineWidth=4; g.beginPath(); g.ellipse(56,58,58,12,-.3,0,7); g.stroke(); } }

// ------------------------------------------------------------ 3D scene
const CS=2;
const V={};
function initScene(){
  if(!window.THREE){ $('nogl').style.display='flex'; $('nogl').textContent='No se pudo cargar Three.js (sin conexión al CDN).'; return; }
  const canvas=$('scene');
  try{ V.renderer=new THREE.WebGLRenderer({canvas,antialias:true}); }catch(e){ $('nogl').style.display='flex'; return; }
  V.renderer.setPixelRatio(Math.min(2,window.devicePixelRatio||1));
  V.renderer.shadowMap.enabled=true; V.renderer.shadowMap.type=THREE.PCFSoftShadowMap;
  V.scene=new THREE.Scene();
  V.camera=new THREE.PerspectiveCamera(48,1,.1,500);
  V.hemi=new THREE.HemisphereLight(0xffe6cc,0x3a2414,.75); V.scene.add(V.hemi);
  V.sun=new THREE.DirectionalLight(0xfff0dd,1.0); V.sun.position.set(-18,30,14); V.sun.castShadow=true;
  V.sun.shadow.mapSize.set(2048,2048); const sc=V.sun.shadow.camera; sc.left=-26; sc.right=26; sc.top=22; sc.bottom=-22; sc.far=90; V.scene.add(V.sun);
  V.static=new THREE.Group(); V.dyn=new THREE.Group(); V.fx=new THREE.Group(); V.scene.add(V.static,V.dyn,V.fx);
  V.rover=buildRover(); V.scene.add(V.rover);
  V.cam={mode:'follow',theta:-.75,phi:1.02,radius:17,target:new THREE.Vector3(),goal:new THREE.Vector3()};
  V.anim={from:null,to:null,t:1,queue:[],heading:0};
  V.shake=0; V.clock=new THREE.Clock();
  bindOrbit(canvas); resize3d(); new ResizeObserver(resize3d).observe($('view'));
  requestAnimationFrame(loop3d);
}
function resize3d(){ if(!V.renderer) return; const r=$('view').getBoundingClientRect(); V.renderer.setSize(r.width,r.height,false); V.camera.aspect=r.width/Math.max(1,r.height); V.camera.updateProjectionMatrix(); }
function cellXZ(r,c,M){ return [(c-(M.width-1)/2)*CS,(r-(M.height-1)/2)*CS]; }
function hash2(x,y){ const s=Math.sin(x*127.1+y*311.7)*43758.5453; return s-Math.floor(s); }
function noise2(x,y){ const xi=Math.floor(x), yi=Math.floor(y), xf=x-xi, yf=y-yi; const u=xf*xf*(3-2*xf), v=yf*yf*(3-2*yf);
  const a=hash2(xi,yi), b=hash2(xi+1,yi), c=hash2(xi,yi+1), d=hash2(xi+1,yi+1); return a+(b-a)*u+(c-a)*v+(a-b-c+d)*u*v; }
function fbm(x,y){ let s=0,a=.5,f=1; for(let i=0;i<4;i++){ s+=a*noise2(x*f,y*f); a*=.5; f*=2.03; } return s; }
function elevAt(x,z){ // world -> interpolated elevation (meters) from the map grid
  const M=S.map; const c=x/CS+(M.width-1)/2, r=z/CS+(M.height-1)/2;
  const cc=Math.max(0,Math.min(M.width-1,c)), rr=Math.max(0,Math.min(M.height-1,r));
  const c0=Math.floor(cc), r0=Math.floor(rr), c1=Math.min(M.width-1,c0+1), r1=Math.min(M.height-1,r0+1), fc=cc-c0, fr=rr-r0;
  const E=M.elevation; const sm=t=>t*t*(3-2*t);
  const e=(E[r0][c0]*(1-sm(fc))+E[r0][c1]*sm(fc))*(1-sm(fr))+(E[r1][c0]*(1-sm(fc))+E[r1][c1]*sm(fc))*sm(fr);
  const out=Math.max(0,Math.max(Math.abs(c-cc),Math.abs(r-rr))); // distance outside the map (cells)
  return {e, out};
}
const PLANET_R=95; // visual radius of the planet cap: the ground falls away towards the horizon
const curveAt=(x,z)=>-(x*x+z*z)/(2*PLANET_R);
function heightAt(x,z){ const {e,out}=elevAt(x,z); const gas=PLANETS[S.planet.name]&&PLANETS[S.planet.name].gas;
  if(gas) return curveAt(x,z)+Math.sin(x*.35+z*.2)*.3+fbm(x*.15,z*.15)*.6;
  const fade=Math.max(0,1-out*.3);  // relief fades outside the grid: no raised rim
  const drop=Math.min(0,e)*.38;     // cliffs: deep depressions
  return curveAt(x,z)+Math.max(-4.2,e*.3+drop)*fade+(fbm(x*.22+3,z*.22)-.5)*.7; }
function buildTerrain(){
  const M=S.map, P=PLANETS[S.planet.name]||PLANETS.Marte;
  V.static.traverse(o=>{ if(o.geometry) o.geometry.dispose(); });
  V.static.clear();
  const W=M.width*CS+110, H=M.height*CS+110, seg=Math.round(W*2.2);
  const geo=new THREE.PlaneGeometry(W,H,seg,Math.round(H*2.2)); geo.rotateX(-Math.PI/2);
  const pos=geo.attributes.position, col=new Float32Array(pos.count*3);
  const ca=new THREE.Color(P.ground[0]), cb=new THREE.Color(P.ground[1]), tmp=new THREE.Color();
  for(let i=0;i<pos.count;i++){ const x=pos.getX(i), z=pos.getZ(i); const y=heightAt(x,z); pos.setY(i,y); const rel=y-curveAt(x,z); // relief without the planet curvature
    const t=Math.max(0,Math.min(1,.55+rel*.12+(fbm(x*.9,z*.9)-.5)*.5)); tmp.copy(cb).lerp(ca,t);
    if(!P.gas && rel<-1.2){ tmp.lerp(new THREE.Color(0x3a1a10),Math.min(.7,(-rel-1.2)*.35)); }
    if(P.gas){ const band=Math.sin(z*.55)*.5+.5; tmp.lerp(new THREE.Color(0xffffff),band*.12); }
    col[i*3]=tmp.r; col[i*3+1]=tmp.g; col[i*3+2]=tmp.b; }
  geo.setAttribute('color',new THREE.BufferAttribute(col,3)); geo.computeVertexNormals();
  const mat=new THREE.MeshStandardMaterial({vertexColors:true,roughness:.96,metalness:0,flatShading:false,transparent:!!P.gas,opacity:P.gas?.92:1});
  V.terrain=new THREE.Mesh(geo,mat); V.terrain.receiveShadow=true; V.static.add(V.terrain);
  // decorative pebbles
  if(!P.gas){ const rg=new THREE.DodecahedronGeometry(.18,0), rm=new THREE.MeshStandardMaterial({color:P.ground[1],roughness:1,flatShading:true});
    const n=420, inst=new THREE.InstancedMesh(rg,rm,n), m4=new THREE.Matrix4(), q=new THREE.Quaternion(), s=new THREE.Vector3();
    for(let i=0;i<n;i++){ const x=(hash2(i,S.config.seed)-.5)*(M.width*CS+44), z=(hash2(S.config.seed,i*1.7)-.5)*(M.height*CS+44); const k=.4+hash2(i*3.1,7)*1.6;
      q.setFromEuler(new THREE.Euler(hash2(i,1)*3,hash2(i,2)*3,0)); s.set(k,k*.7,k); m4.compose(new THREE.Vector3(x,heightAt(x,z)+.05,z),q,s); inst.setMatrixAt(i,m4); }
    inst.castShadow=true; V.static.add(inst); }
  V.scene.background=new THREE.Color(P.sky); V.scene.fog=new THREE.FogExp2(P.fog,P.fogD);
  V.terrainKey=terrainKey();
  V.rover.visible=true; setRoverStyle(P.gas);
}
function terrainKey(){ const M=S.map; let s=0; M.elevation.forEach((row,r)=>row.forEach((v,c)=>{ s+=v*(r*31+c+1); })); return S.planet.name+'|'+S.config.seed+'|'+S.config.difficulty+'|'+s.toFixed(2); }
function rockCluster(x,z,seed,color,scale){ const g=new THREE.Group(); const n=2+Math.floor(hash2(seed,3)*3);
  for(let i=0;i<n;i++){ const k=(.35+hash2(seed,i)*.45)*scale; const m=new THREE.Mesh(new THREE.DodecahedronGeometry(k,0),new THREE.MeshStandardMaterial({color,roughness:.95,flatShading:true}));
    const ox=(hash2(seed+i,5)-.5)*1.1, oz=(hash2(seed,i+9)-.5)*1.1; m.position.set(x+ox,heightAt(x+ox,z+oz)+k*.45,z+oz); m.rotation.set(hash2(i,seed)*3,hash2(seed,i)*3,0); m.scale.y=.75;
    m.castShadow=true; m.receiveShadow=true; g.add(m); } return g; }
function tile(x,z,color,opacity,size){ const sz=size||CS*.94; const geo=new THREE.PlaneGeometry(sz,sz,6,6); geo.rotateX(-Math.PI/2);
  const p=geo.attributes.position; for(let i=0;i<p.count;i++){ p.setY(i,heightAt(x+p.getX(i),z+p.getZ(i))+.06); }
  const m=new THREE.Mesh(geo,new THREE.MeshBasicMaterial({color,transparent:true,opacity,depthWrite:false})); m.position.set(x,0,z); return m; }
function pathLine(cells,color,width,dashed,lift){ if(!cells||cells.length<2) return null; const M=S.map; const pts=[];
  for(let i=0;i<cells.length-1;i++){ const [x0,z0]=cellXZ(cells[i][0],cells[i][1],M), [x1,z1]=cellXZ(cells[i+1][0],cells[i+1][1],M);
    for(let k=0;k<6;k++){ const t=k/6, x=x0+(x1-x0)*t, z=z0+(z1-z0)*t; pts.push(new THREE.Vector3(x,heightAt(x,z)+(lift||.35),z)); } }
  const [xl,zl]=cellXZ(cells[cells.length-1][0],cells[cells.length-1][1],M); pts.push(new THREE.Vector3(xl,heightAt(xl,zl)+(lift||.35),zl));
  const curve=new THREE.CatmullRomCurve3(pts); const geo=new THREE.TubeGeometry(curve,pts.length*2,width,5,false);
  return new THREE.Mesh(geo,new THREE.MeshBasicMaterial({color,transparent:!!dashed,opacity:dashed?.55:1})); }
function buildDynamic(){
  const M=S.map, P=PLANETS[S.planet.name]||PLANETS.Marte, reveal=$('fReveal').checked, gas=P.gas;
  V.dyn.traverse(o=>{ if(o.geometry) o.geometry.dispose(); if(o.material&&o.material.dispose) o.material.dispose(); }); V.dyn.clear();
  const key=c=>c[0]+','+c[1];
  const known=new Set(M.obstacles.map(key)), cliffs=new Set(M.cliffs.map(key)), hidObs=new Set(M.hidden_obstacles.map(key));
  // obstacles (all physically present); hidden ones only shown with reveal
  M.obstacles.concat(reveal?M.hidden_obstacles:[]).forEach(([r,c])=>{ const [x,z]=cellXZ(r,c,M);
    if(gas){ const s=new THREE.Mesh(new THREE.SphereGeometry(.9,16,12),new THREE.MeshStandardMaterial({color:0x5aa9ff,transparent:true,opacity:.35,emissive:0x1d4f8a})); s.position.set(x,heightAt(x,z)+1.1,z); V.dyn.add(s); V.dyn.add(tile(x,z,0x5aa9ff,.28)); }
    else { V.dyn.add(rockCluster(x,z,r*97+c*13,hidObs.has(key([r,c]))?0x6d4a36:0x8a5a3c,1)); if(known.has(key([r,c]))) V.dyn.add(tile(x,z,0xffc04d,.3)); } });
  M.cliffs.forEach(([r,c])=>{ const [x,z]=cellXZ(r,c,M); V.dyn.add(tile(x,z,0xff5a4f,.5)); });
  if(reveal) M.hidden_cliffs.forEach(([r,c])=>{ const [x,z]=cellXZ(r,c,M); V.dyn.add(tile(x,z,0xff5a4f,.18)); });
  M.risk.forEach(([r,c,v])=>{ if(cliffs.has(key([r,c]))||known.has(key([r,c]))) return; const [x,z]=cellXZ(r,c,M); V.dyn.add(tile(x,z,0xff9a3c,Math.min(.5,v*.55))); });
  // steep slopes (pendiente) from the elevation grid
  if(!gas) for(let r=0;r<M.height;r++) for(let c=0;c<M.width;c++){ const e=M.elevation[r][c]; let d=0;
    [[0,1],[1,0]].forEach(([dr,dc])=>{ if(r+dr<M.height&&c+dc<M.width){ const o=M.elevation[r+dr][c+dc]; if(o>-5&&e>-5) d=Math.max(d,Math.abs(o-e)); } });
    if(d>2.2&&!known.has(key([r,c]))&&!cliffs.has(key([r,c]))){ const [x,z]=cellXZ(r,c,M); V.dyn.add(tile(x,z,0xe8e05a,.16)); } }
  const hidCliffs=new Set(M.hidden_cliffs.map(key));
  const pin=(x,z,color)=>{ const y=heightAt(x,z); const stick=new THREE.Mesh(new THREE.CylinderGeometry(.03,.03,2.2,6),new THREE.MeshBasicMaterial({color:0xffffff})); stick.position.set(x,y+1.1,z);
    const head=new THREE.Mesh(new THREE.SphereGeometry(.2,12,8),new THREE.MeshBasicMaterial({color})); head.position.set(x,y+2.3,z); V.dyn.add(stick,head); };
  (M.destinations||[]).forEach(([dr,dc],i)=>{ const [x,z]=cellXZ(dr,dc,M), y=heightAt(x,z), next=i===0;
    const beam=new THREE.Mesh(new THREE.CylinderGeometry(.07,.07,next?8:5,8),new THREE.MeshBasicMaterial({color:0xff4fd8,transparent:true,opacity:next?.75:.4})); beam.position.set(x,y+(next?4:2.5),z);
    const ring=new THREE.Mesh(new THREE.TorusGeometry(.75,.07,8,28),new THREE.MeshBasicMaterial({color:0xff4fd8})); ring.rotation.x=Math.PI/2; ring.position.set(x,y+.25,z); ring.userData.pulse=next;
    const tag=new THREE.Sprite(new THREE.SpriteMaterial({map:numberTexture(i+1),depthTest:false})); tag.scale.set(1.5,1.5,1); tag.position.set(x,y+(next?8.6:5.6),z);
    V.dyn.add(beam,ring,tag,tile(x,z,0xff4fd8,next?.3:.18)); });
  if((M.tour||[]).length>1){ const tour=pathLine(M.tour,0xff4fd8,.05,false,.5); if(tour) V.dyn.add(tour); }
  (M.events||[]).forEach(e=>{ if(!e.position) return; const [x,z]=cellXZ(e.position[0],e.position[1],M);
    // Operator edits are shown at once, even while the rover has not sensed them yet.
    if(e.kind==='barranco'&&hidCliffs.has(key(e.position))){ V.dyn.add(tile(x,z,0xff5a4f,.34)); pin(x,z,0xff5a4f); }
    if(e.kind==='obstaculo_manual'&&hidObs.has(key(e.position))){ if(!reveal) V.dyn.add(rockCluster(x,z,e.position[0]*97+e.position[1]*13,0x6d4a36,1)); V.dyn.add(tile(x,z,0xffc04d,.3)); pin(x,z,0xffc04d); }
    if(e.kind==='meteorito'){ V.dyn.add(tile(x,z,0xd07bff,.35,CS*2.6)); }
    if(e.kind==='alienigena'){ const a=new THREE.Mesh(new THREE.CylinderGeometry(.3,.38,1.6,12),new THREE.MeshStandardMaterial({color:0x7dff9a,emissive:0x2aff6a,emissiveIntensity:.8,transparent:true,opacity:.75})); a.position.set(x,heightAt(x,z)+1,z); a.userData.alien=true; V.dyn.add(a); } });
  // refuges
  M.refuges.forEach(([r,c])=>{ const [x,z]=cellXZ(r,c,M), y=heightAt(x,z);
    const pad=new THREE.Mesh(new THREE.CylinderGeometry(.95,1.05,.12,24),new THREE.MeshStandardMaterial({color:0x4b4f55,roughness:.7})); pad.position.set(x,y+.06,z); pad.receiveShadow=true;
    const dome=new THREE.Mesh(new THREE.SphereGeometry(.75,24,12,0,Math.PI*2,0,Math.PI/2),new THREE.MeshStandardMaterial({color:0xdfe6ea,roughness:.4,metalness:.2})); dome.position.set(x,y+.12,z); dome.castShadow=true;
    const light=new THREE.Mesh(new THREE.SphereGeometry(.12,10,8),new THREE.MeshBasicMaterial({color:0x52d58e})); light.position.set(x,y+.95,z);
    V.dyn.add(pad,dome,light,tile(x,z,0x52d58e,.18)); });
  // science sites
  M.science.forEach(([r,c,done])=>{ const [x,z]=cellXZ(r,c,M), y=heightAt(x,z); const color=done?0x52d58e:0xffd84a;
    const gem=new THREE.Mesh(new THREE.OctahedronGeometry(.32,0),new THREE.MeshStandardMaterial({color,emissive:color,emissiveIntensity:.6,roughness:.3})); gem.position.set(x,y+.9,z); gem.userData.spin=true;
    const beam=new THREE.Mesh(new THREE.CylinderGeometry(.04,.04,6,6),new THREE.MeshBasicMaterial({color,transparent:true,opacity:.35})); beam.position.set(x,y+3,z);
    V.dyn.add(gem,beam); });
  // instrument caches to recover (mission goal)
  M.resources.forEach(([r,c,done])=>{ if(done) return; const [x,z]=cellXZ(r,c,M), y=heightAt(x,z);
    const crate=new THREE.Mesh(new THREE.BoxGeometry(.6,.42,.6),new THREE.MeshStandardMaterial({color:0x9aa4b0,roughness:.5,metalness:.6})); crate.position.set(x,y+.25,z); crate.rotation.y=.5; crate.castShadow=true;
    const lamp=new THREE.Mesh(new THREE.SphereGeometry(.1,10,8),new THREE.MeshBasicMaterial({color:0x5aa9ff})); lamp.position.set(x,y+.6,z);
    const beam=new THREE.Mesh(new THREE.CylinderGeometry(.03,.03,4,6),new THREE.MeshBasicMaterial({color:0x5aa9ff,transparent:true,opacity:.35})); beam.position.set(x,y+2.4,z);
    V.dyn.add(crate,lamp,beam); });
  // routes
  (M.candidates||[]).forEach((route,i)=>{ if(i===M.chosen) return; const l=pathLine(route,0xb58cff,.05,true,.3); if(l) V.dyn.add(l); });
  const planned=pathLine(M.route,0xffd84a,.08,false,.42); if(planned) V.dyn.add(planned);
  const done=pathLine(M.path,0x5aa9ff,.11,false,.25); if(done) V.dyn.add(done);
  // ghost race: the other policies on the same map and seed
  (S.ghosts||[]).forEach((G,i)=>{ const hex=new THREE.Color(GHOST_COL[G.policy]||'#ffffff').getHex();
    const trail=pathLine(G.path,hex,.045,true,.16+i*.04); if(trail) V.dyn.add(trail);
    const [gx,gz]=cellXZ(G.position[0],G.position[1],M), x=gx+(i-1)*.45, y=heightAt(gx,gz);
    const mat=new THREE.MeshBasicMaterial({color:hex,transparent:true,opacity:G.finished?.22:.5,depthWrite:false});
    const body=new THREE.Mesh(new THREE.BoxGeometry(.9,.45,1.3),mat); body.position.set(x,y+.55,gz);
    const mast=new THREE.Mesh(new THREE.CylinderGeometry(.05,.05,.9,6),mat); mast.position.set(x,y+1.2,gz);
    const tag=new THREE.Sprite(new THREE.SpriteMaterial({map:labelTexture(GHOST_LBL[G.policy]||G.policy,GHOST_COL[G.policy]),depthTest:false,transparent:true,opacity:.9})); tag.scale.set(2.6,.65,1); tag.position.set(x,y+2.1+i*.5,gz);
    V.dyn.add(body,mast,tag); });
  // scout drone
  const D=S.drone; if(D){ const [x,z]=cellXZ(D.position[0],D.position[1],M), y=heightAt(x,z)+(D.flying?3.2:1.7); const q=new THREE.Group();
    q.add(new THREE.Mesh(new THREE.BoxGeometry(.5,.14,.5),new THREE.MeshStandardMaterial({color:0x22d3ee,emissive:0x0b6b78,roughness:.4})));
    [[1,1],[1,-1],[-1,1],[-1,-1]].forEach(([a,b])=>{ const arm=new THREE.Mesh(new THREE.BoxGeometry(.5,.04,.06),new THREE.MeshBasicMaterial({color:0x0e7490})); arm.position.set(a*.25,0,b*.25); arm.rotation.y=a*b>0?-Math.PI/4:Math.PI/4;
      const rotor=new THREE.Mesh(new THREE.CylinderGeometry(.26,.26,.02,14),new THREE.MeshBasicMaterial({color:0xd8faff,transparent:true,opacity:D.flying?.5:.9})); rotor.position.set(a*.42,.09,b*.42); q.add(arm,rotor); });
    q.position.set(x,y,z); V.dyn.add(q);
    if(D.flying){ D.scanned.forEach(([r,c])=>{ const [sx,sz]=cellXZ(r,c,M); V.dyn.add(tile(sx,sz,0x22d3ee,.09)); });
      const ray=new THREE.Mesh(new THREE.CylinderGeometry(.03,.9,3.1,10,1,true),new THREE.MeshBasicMaterial({color:0x22d3ee,transparent:true,opacity:.16,side:THREE.DoubleSide,depthWrite:false})); ray.position.set(x,y-1.6,z); V.dyn.add(ray); }
    (D.found||[]).forEach(([r,c])=>{ const [fx,fz]=cellXZ(r,c,M); V.dyn.add(tile(fx,fz,0x22d3ee,.5)); }); }
}
function chileFlagTexture(){ // drawn by code: no external image files
  const c=document.createElement('canvas'); c.width=300; c.height=200; const g=c.getContext('2d');
  g.fillStyle='#ffffff'; g.fillRect(0,0,300,100); g.fillStyle='#d52b1e'; g.fillRect(0,100,300,100); g.fillStyle='#0039a6'; g.fillRect(0,0,100,100);
  g.fillStyle='#ffffff'; g.beginPath(); for(let i=0;i<10;i++){ const r=i%2?13:33, a=-Math.PI/2+i*Math.PI/5; g.lineTo(50+r*Math.cos(a),50+r*Math.sin(a)); } g.closePath(); g.fill();
  const t=new THREE.CanvasTexture(c); t.anisotropy=4; return t; }
function flagPlane(tex,w,h,x,y,z,ry,double){ const m=new THREE.Mesh(new THREE.PlaneGeometry(w,h),new THREE.MeshBasicMaterial({map:tex,side:double?THREE.DoubleSide:THREE.FrontSide}));
  m.position.set(x,y,z); m.rotation.y=ry; return m; }
function buildRover(){ const g=new THREE.Group(); const white=new THREE.MeshStandardMaterial({color:0xe9e6df,roughness:.5,metalness:.3}), dark=new THREE.MeshStandardMaterial({color:0x2b2f38,roughness:.6,metalness:.4});
  const body=new THREE.Mesh(new THREE.BoxGeometry(1.05,.38,1.35),white); body.position.y=.62; body.castShadow=true; g.add(body);
  const deck=new THREE.Mesh(new THREE.BoxGeometry(1.25,.05,1.55),new THREE.MeshStandardMaterial({color:0x1f3a7a,roughness:.3,metalness:.6,emissive:0x0a1430})); deck.position.y=.84; deck.castShadow=true; g.add(deck);
  const wg=new THREE.CylinderGeometry(.2,.2,.18,14); wg.rotateZ(Math.PI/2);
  [-.48,0,.48].forEach(z=>[-.62,.62].forEach(x=>{ const w=new THREE.Mesh(wg,dark); w.position.set(x,.22,z); w.castShadow=true; g.add(w); }));
  const mast=new THREE.Mesh(new THREE.CylinderGeometry(.04,.04,.75,8),white); mast.position.set(0,1.2,.45); g.add(mast);
  const head=new THREE.Mesh(new THREE.BoxGeometry(.32,.18,.2),white); head.position.set(0,1.6,.48); head.castShadow=true; g.add(head);
  const eye=new THREE.Mesh(new THREE.SphereGeometry(.05,8,6),new THREE.MeshBasicMaterial({color:0x7fd3ff})); eye.position.set(0,1.6,.6); g.add(eye);
  const ant=new THREE.Mesh(new THREE.CylinderGeometry(.015,.015,.6,6),white); ant.position.set(-.35,1.15,-.45); g.add(ant);
  const beacon=new THREE.Mesh(new THREE.SphereGeometry(.07,8,6),new THREE.MeshBasicMaterial({color:0xffc04d})); beacon.position.set(-.35,1.47,-.45); g.add(beacon); g.userData.beacon=beacon;
  const probe=new THREE.Group(); const balloon=new THREE.Mesh(new THREE.SphereGeometry(.9,20,14),new THREE.MeshStandardMaterial({color:0xf2efe8,roughness:.4})); balloon.position.y=2.2;
  const gondola=new THREE.Mesh(new THREE.BoxGeometry(.6,.35,.6),dark); gondola.position.y=.9; const rope=new THREE.Mesh(new THREE.CylinderGeometry(.02,.02,.9,4),white); rope.position.y=1.4;
  const flag=chileFlagTexture();
  const pole=new THREE.Mesh(new THREE.CylinderGeometry(.02,.02,1.0,6),white); pole.position.set(.42,1.34,-.55); g.add(pole);
  const banner=flagPlane(flag,.54,.36,.42,1.64,-.28,Math.PI/2,true); g.add(banner); g.userData.banner=banner;
  g.add(flagPlane(flag,.42,.28,.531,.62,0,Math.PI/2), flagPlane(flag,.42,.28,-.531,.62,0,-Math.PI/2));
  const topFlag=(w,h,y,z)=>{ const m=flagPlane(flag,w,h,0,y,z,0); m.rotation.x=-Math.PI/2; return m; }; // readable from the top-down camera
  g.add(topFlag(.66,.44,.872,-.38));
  probe.add(balloon,gondola,rope, flagPlane(flag,.42,.28,.305,.9,0,Math.PI/2), flagPlane(flag,.42,.28,-.305,.9,0,-Math.PI/2),
            flagPlane(flag,.42,.28,0,.9,.305,0), flagPlane(flag,.42,.28,0,.9,-.305,Math.PI), topFlag(.9,.6,3.105,0)); probe.visible=false; g.add(probe); g.userData.probe=probe; g.userData.roverParts=g.children.filter(o=>o!==probe);
  return g; }
function setRoverStyle(gas){ V.rover.userData.roverParts.forEach(o=>o.visible=!gas); V.rover.userData.probe.visible=!!gas; }
let lastPos=null, lastStep=-1, lastConfigKey='';
function sync3d(){ if(!V.renderer||!S) return;
  const cfgKey=S.config.seed+'|'+S.planet.name+'|'+S.config.difficulty+'|'+S.config.mode;
  if(cfgKey!==lastConfigKey){ lastConfigKey=cfgKey; buildTerrain(); lastPos=null; lastStep=-1; }
  else if(terrainKey()!==V.terrainKey) buildTerrain();
  buildDynamic();
  const M=S.map; const pos=S.status.position;
  if(lastPos===null||S.status.step<lastStep){ const [x,z]=cellXZ(pos[0],pos[1],M); V.rover.position.set(x,heightAt(x,z),z); V.anim.queue=[]; V.anim.t=1; V.cam.target.set(x,0,z); }
  else if(S.status.step!==lastStep){ const path=M.path; const newCells=path.slice(Math.max(0,path.length-1-(S.log.length?6:0))); // queue recent moves
    const last=S.log.length?S.log[S.log.length-1]:null; const moved=(window._lastMoved||[]);
    moved.forEach(c=>V.anim.queue.push(c)); }
  lastPos=pos; lastStep=S.status.step;
  // lighting: day / night / storm
  const P=PLANETS[S.planet.name]||PLANETS.Marte; const night=!S.status.daylight, storm=S.status.storm>0;
  V.sun.intensity=P.sun*(night?.18:1)*(storm?.55:1)*(S.status.heat>0?1.35:1); V.sun.color.setHex(S.status.heat>0?0xffb070:0xfff0dd); V.hemi.intensity=night?.28:.75;
  const heat=S.status.heat>0; const sky=new THREE.Color(P.sky); if(night) sky.lerp(new THREE.Color(0x05060c),.82); if(storm) sky.lerp(new THREE.Color(0xa0603a),.5); if(heat) sky.lerp(new THREE.Color(0xff5a1f),.45);
  V.scene.background=sky; V.scene.fog.color.copy(storm?new THREE.Color(0xa86a42):night?new THREE.Color(0x0b0b12):new THREE.Color(P.fog)); V.scene.fog.density=P.fogD*(storm?4.2:night?1.6:1);
  setDust(storm); if(S.status.quake>0) V.shake=.7;
}
function setDust(on){ if(on&&!V.dust){ const n=1600, g=new THREE.BufferGeometry(), p=new Float32Array(n*3); for(let i=0;i<n;i++){ p[i*3]=(Math.random()-.5)*50; p[i*3+1]=Math.random()*10; p[i*3+2]=(Math.random()-.5)*40; }
    g.setAttribute('position',new THREE.BufferAttribute(p,3)); V.dust=new THREE.Points(g,new THREE.PointsMaterial({color:0xd89a66,size:.12,transparent:true,opacity:.7})); V.fx.add(V.dust); }
  if(!on&&V.dust){ V.fx.remove(V.dust); V.dust.geometry.dispose(); V.dust=null; } }
function meteorFx(cell){ if(!V.renderer||!S) return; const [x,z]=cellXZ(cell[0],cell[1],S.map); const m=new THREE.Mesh(new THREE.SphereGeometry(.4,12,10),new THREE.MeshBasicMaterial({color:0xffb070}));
  m.position.set(x+14,26,z-8); m.userData.meteor={to:new THREE.Vector3(x,heightAt(x,z),z),t:0}; V.fx.add(m); }
function loop3d(){ requestAnimationFrame(loop3d); if(!V.renderer||!S) return; if(viewMode==='2d'){ V.anim.queue.length=0; return; } const dt=Math.min(.05,V.clock.getDelta()), tt=V.clock.elapsedTime, M=S.map;
  // rover animation along queued cells
  const a=V.anim; if(a.t>=1&&a.queue.length){ const c=a.queue.shift(); const [x,z]=cellXZ(c[0],c[1],M); a.from=V.rover.position.clone(); a.to=new THREE.Vector3(x,0,z); a.t=0;
    const dx=a.to.x-a.from.x, dz=a.to.z-a.from.z; if(Math.abs(dx)+Math.abs(dz)>.01) a.heading=Math.atan2(dx,dz); }
  if(a.t<1&&a.to){ a.t=Math.min(1,a.t+dt*(2.2+a.queue.length*1.5)); const x=a.from.x+(a.to.x-a.from.x)*a.t, z=a.from.z+(a.to.z-a.from.z)*a.t; V.rover.position.set(x,0,z); }
  const gas=PLANETS[S.planet.name]&&PLANETS[S.planet.name].gas;
  V.rover.position.y=heightAt(V.rover.position.x,V.rover.position.z)+(gas?1.2+Math.sin(tt*1.4)*.15:0);
  let dh=a.heading-V.rover.rotation.y; dh=Math.atan2(Math.sin(dh),Math.cos(dh)); V.rover.rotation.y+=dh*Math.min(1,dt*6);
  if(V.rover.userData.beacon) V.rover.userData.beacon.material.color.setHex(Math.sin(tt*6)>0?0xffc04d:0x553311);
  V.dyn.children.forEach(o=>{ if(o.userData.spin){ o.rotation.y+=dt*1.2; o.position.y+=Math.sin(tt*2+o.position.x)*.002; } if(o.userData.alien) o.material.emissiveIntensity=.5+Math.sin(tt*4)*.4; if(o.userData.pulse){ const k=1+Math.sin(tt*3)*.15; o.scale.set(k,k,1); } });
  V.fx.children.slice().forEach(o=>{ const mm=o.userData.meteor; if(mm){ mm.t+=dt*.9; o.position.lerp(mm.to,Math.min(1,mm.t*.25+.02)); if(mm.t>2.2||o.position.distanceTo(mm.to)<.3){ V.fx.remove(o); } } });
  if(V.dust){ const p=V.dust.geometry.attributes.position; for(let i=0;i<p.count;i++){ let x=p.getX(i)+dt*6; if(x>25) x=-25; p.setX(i,x); } p.needsUpdate=true; }
  // camera
  const C=V.cam; const rp=V.rover.position;
  if(C.mode==='follow') C.goal.set(rp.x,rp.y,rp.z); else C.goal.set(0,0,0);
  C.target.lerp(C.goal,Math.min(1,dt*3));
  const phi=C.mode==='top'?.08:C.phi, rad=C.mode==='top'?Math.max(C.radius,30):C.radius;
  V.camera.position.set(C.target.x+rad*Math.sin(phi)*Math.sin(C.theta),C.target.y+rad*Math.cos(phi),C.target.z+rad*Math.sin(phi)*Math.cos(C.theta));
  if(V.shake>0){ V.camera.position.x+=(Math.random()-.5)*V.shake; V.camera.position.y+=(Math.random()-.5)*V.shake*.5; V.shake=Math.max(0,V.shake-dt*.9); }
  V.camera.lookAt(C.target); V.renderer.render(V.scene,V.camera); }
function bindOrbit(canvas){ let drag=null;
  canvas.addEventListener('pointerdown',e=>{ drag={x:e.clientX,y:e.clientY,moved:false}; canvas.setPointerCapture(e.pointerId); canvas.style.cursor='grabbing'; });
  canvas.addEventListener('pointermove',e=>{ if(!drag) return; const dx=e.clientX-drag.x, dy=e.clientY-drag.y; if(Math.abs(dx)+Math.abs(dy)>3) drag.moved=true;
    V.cam.theta-=dx*.006; V.cam.phi=Math.max(.15,Math.min(1.45,V.cam.phi-dy*.005)); drag.x=e.clientX; drag.y=e.clientY; if(V.cam.mode==='top'&&drag.moved) setCam('orbit'); });
  canvas.addEventListener('pointerup',e=>{ canvas.style.cursor='grab'; const wasClick=drag&&!drag.moved; drag=null; if(wasClick) pick(e); });
  canvas.addEventListener('wheel',e=>{ e.preventDefault(); V.cam.radius=Math.max(6,Math.min(60,V.cam.radius*(1+Math.sign(e.deltaY)*.09))); },{passive:false}); }
const GHOST_COL={hibrido:'#52d58e',sin_rna:'#5aa9ff',sin_difuso:'#c58cff',sin_ag:'#ff9a3c'}, GHOST_LBL={hibrido:'Híbrido',sin_rna:'Sin RNA',sin_difuso:'Sin difuso',sin_ag:'Sin AG'};
function labelTexture(text,color){ const cache=labelTexture.cache||(labelTexture.cache={}), k=text+color; if(cache[k]) return cache[k]; const c=document.createElement('canvas'); c.width=256; c.height=64; const g=c.getContext('2d');
  g.fillStyle='rgba(8,6,12,.72)'; g.fillRect(4,8,248,48); g.strokeStyle=color; g.lineWidth=3; g.strokeRect(4,8,248,48); g.fillStyle=color; g.font='bold 30px system-ui,sans-serif'; g.textAlign='center'; g.textBaseline='middle'; g.fillText(text,128,34); return cache[k]=new THREE.CanvasTexture(c); }
function numberTexture(n){ const numberCache=numberTexture.cache||(numberTexture.cache={}); if(numberCache[n]) return numberCache[n]; const c=document.createElement('canvas'); c.width=c.height=64; const g=c.getContext('2d');
  g.fillStyle='#ff4fd8'; g.beginPath(); g.arc(32,32,28,0,7); g.fill(); g.strokeStyle='#1a0516'; g.lineWidth=4; g.stroke();
  g.fillStyle='#1a0516'; g.font='bold 38px system-ui,sans-serif'; g.textAlign='center'; g.textBaseline='middle'; g.fillText(n,32,35); return numberCache[n]=new THREE.CanvasTexture(c); }
let destMode=false;
function setDestMode(on){ destMode=on; $('bDest').classList.toggle('on',on); $('view').classList.toggle('picking',on||!!$('fTool').value);
  $('bDest').textContent=on?'✓ Terminar':'◎ Agregar destinos'; if(on) msg('Haga clic en las celdas del mapa para agregar destinos. Un clic sobre un destino lo quita.'); }
function cellAction(row,c){ if(!S) return; const M=S.map; if(row<0||c<0||row>=M.height||c>=M.width) return;
  if(destMode){ const had=(M.destinations||[]).some(d=>d[0]===row&&d[1]===c); call('destination',{position:[row,c]}).then(o=>{ if(o){ const n=o.map.destinations.length; msg(had?`Destino (${row},${c}) quitado. Quedan ${n}.`:`Destino agregado en (${row},${c}). ${n>1?`El AG ordenó los ${n} destinos: recorrido de ${o.status.tour_cells} celdas.`:'Ruta recalculada.'}`); } }); return; }
  const tool=$('fTool').value; if(!tool) return;
  const label={obstaculo:'Obstáculo colocado',barranco:'Barranco colocado',limpiar:'Celda limpiada'}[tool];
  call('place',{position:[row,c],kind:tool}).then(o=>{ if(o) msg(`${label} en (${row},${c}).`+(tool==='limpiar'?'':' Queda marcado en el mapa; el rover debe descubrirlo con sus sensores.')); }); }
function pick(e){ if((!destMode&&!$('fTool').value)||!V.terrain||!S) return; const r=$('scene').getBoundingClientRect();
  const m=new THREE.Vector2(((e.clientX-r.left)/r.width)*2-1,-((e.clientY-r.top)/r.height)*2+1); const ray=new THREE.Raycaster(); ray.setFromCamera(m,V.camera);
  const hit=ray.intersectObject(V.terrain)[0]; if(!hit) return; const M=S.map; const c=Math.round(hit.point.x/CS+(M.width-1)/2), row=Math.round(hit.point.z/CS+(M.height-1)/2);
  cellAction(row,c); }
function setCam(mode){ V.cam.mode=mode; document.querySelectorAll('[data-cam]').forEach(b=>b.classList.toggle('on',b.dataset.cam===mode)); if(mode==='orbit'){ V.cam.radius=Math.max(V.cam.radius,30); } if(mode==='follow') V.cam.radius=17; }
document.querySelectorAll('[data-cam]').forEach(b=>b.addEventListener('click',()=>setCam(b.dataset.cam)));
let viewMode='3d', geom2d=null;
function setView(mode){ viewMode=mode; document.querySelectorAll('[data-view]').forEach(b=>b.classList.toggle('on',b.dataset.view===mode));
  $('scene').style.display=mode==='3d'?'block':'none'; $('map2d').style.display=mode==='2d'?'block':'none';
  document.querySelectorAll('[data-cam]').forEach(b=>b.style.display=mode==='3d'?'':'none');
  document.querySelector('#ovTop b').textContent='ARES-X '+mode.toUpperCase();
  try{ localStorage.setItem('ares_view',mode); }catch(e){}
  if(mode==='2d') draw2d(); else { lastPos=null; resize3d(); if(S){ try{ sync3d(); }catch(e){ console.error(e); } } } }
document.querySelectorAll('[data-view]').forEach(b=>b.addEventListener('click',()=>setView(b.dataset.view)));
const rgb=h=>[(h>>16)&255,(h>>8)&255,h&255];
function draw2d(){ if(!S||viewMode!=='2d') return; const [g,w,h]=ctx2d($('map2d')); const M=S.map, P=PLANETS[S.planet.name]||PLANETS.Marte, reveal=$('fReveal').checked;
  const padT=46, padB=74, padX=16; const cs=Math.max(8,Math.floor(Math.min((w-2*padX)/M.width,(h-padT-padB)/M.height)));
  const ox=Math.floor((w-cs*M.width)/2), oy=padT+Math.floor((h-padT-padB-cs*M.height)/2); geom2d={cs,ox,oy};
  const X=c=>ox+c*cs+cs/2, Y=r=>oy+r*cs+cs/2, key=c=>c[0]+','+c[1]; const hi=rgb(P.ground[0]), lo=rgb(P.ground[1]);
  let emin=Infinity, emax=-Infinity; M.elevation.forEach(row=>row.forEach(v=>{ if(v>-5){ emin=Math.min(emin,v); emax=Math.max(emax,v); } }));
  for(let r=0;r<M.height;r++) for(let c=0;c<M.width;c++){ const e=M.elevation[r][c], t=e<-5?0:Math.max(0,Math.min(1,(e-emin)/((emax-emin)||1)));
    g.fillStyle=`rgb(${Math.round(lo[0]+(hi[0]-lo[0])*t)},${Math.round(lo[1]+(hi[1]-lo[1])*t)},${Math.round(lo[2]+(hi[2]-lo[2])*t)})`; g.fillRect(ox+c*cs,oy+r*cs,cs-1,cs-1); }
  M.risk.forEach(([r,c,v])=>{ g.fillStyle=`rgba(255,154,60,${Math.min(.6,v*.65)})`; g.fillRect(ox+c*cs,oy+r*cs,cs-1,cs-1); });
  const rock=(r,c,dashed)=>{ g.beginPath(); g.arc(X(c),Y(r),cs*.34,0,7); if(dashed){ g.setLineDash([3,3]); g.strokeStyle='#3a2415'; g.lineWidth=1.6; g.stroke(); g.setLineDash([]); } else { g.fillStyle=P.gas?'rgba(90,169,255,.75)':'#4a2f1e'; g.fill(); g.strokeStyle='rgba(0,0,0,.35)'; g.lineWidth=1; g.stroke(); } };
  const cliff=(r,c,dashed)=>{ const x=ox+c*cs, y=oy+r*cs; g.save(); g.beginPath(); g.rect(x,y,cs-1,cs-1); g.clip(); g.fillStyle=dashed?'rgba(120,20,15,.35)':'rgba(120,20,15,.8)'; g.fillRect(x,y,cs,cs);
    g.strokeStyle=dashed?'rgba(255,90,79,.6)':'#ff5a4f'; g.lineWidth=dashed?1:2; for(let k=-cs;k<cs*2;k+=7){ g.beginPath(); g.moveTo(x+k,y); g.lineTo(x+k+cs,y+cs); g.stroke(); } g.restore(); };
  M.obstacles.forEach(([r,c])=>rock(r,c,false)); M.cliffs.forEach(([r,c])=>cliff(r,c,false));
  if(reveal){ M.hidden_obstacles.forEach(([r,c])=>rock(r,c,true)); M.hidden_cliffs.forEach(([r,c])=>cliff(r,c,true)); }
  const hidO=new Set(M.hidden_obstacles.map(key)), hidC=new Set(M.hidden_cliffs.map(key));
  (M.events||[]).forEach(e=>{ if(!e.position) return; const [r,c]=e.position;
    if(e.kind==='meteorito'){ g.strokeStyle='#d07bff'; g.lineWidth=2; g.beginPath(); g.arc(X(c),Y(r),cs*1.3,0,7); g.stroke(); }
    if(e.kind==='alienigena'){ g.fillStyle='#7dff9a'; g.beginPath(); g.arc(X(c),Y(r),cs*.22,0,7); g.fill(); }
    const op=(e.kind==='barranco'&&hidC.has(key(e.position)))||(e.kind==='obstaculo_manual'&&hidO.has(key(e.position)));
    if(op){ if(e.kind==='barranco') cliff(r,c,true); else rock(r,c,true); g.strokeStyle='#fff'; g.lineWidth=2; g.strokeRect(ox+c*cs+1,oy+r*cs+1,cs-3,cs-3); } });
  M.refuges.forEach(([r,c])=>{ g.fillStyle='#2f8f5b'; g.fillRect(ox+c*cs+2,oy+r*cs+2,cs-5,cs-5); g.fillStyle='#eafff2'; g.font=`700 ${Math.round(cs*.5)}px Barlow Condensed, sans-serif`; g.textAlign='center'; g.fillText('R',X(c),Y(r)+cs*.18); });
  M.resources.forEach(([r,c,d])=>{ g.fillStyle=d?'#52d58e':'#5aa9ff'; g.fillRect(X(c)-cs*.2,Y(r)-cs*.2,cs*.4,cs*.4); g.strokeStyle='#0b1420'; g.lineWidth=1; g.strokeRect(X(c)-cs*.2,Y(r)-cs*.2,cs*.4,cs*.4); });
  M.science.forEach(([r,c,d])=>{ g.beginPath(); g.moveTo(X(c),Y(r)-cs*.33); g.lineTo(X(c)+cs*.3,Y(r)); g.lineTo(X(c),Y(r)+cs*.33); g.lineTo(X(c)-cs*.3,Y(r)); g.closePath(); g.fillStyle=d?'#52d58e':'#ffd84a'; g.fill(); g.strokeStyle='#1a1205'; g.lineWidth=1; g.stroke(); });
  const line=(cells,color,width,dash)=>{ if(!cells||cells.length<2) return; g.strokeStyle=color; g.lineWidth=width; g.lineJoin='round'; g.setLineDash(dash||[]); g.beginPath(); cells.forEach(([r,c],i)=>i?g.lineTo(X(c),Y(r)):g.moveTo(X(c),Y(r))); g.stroke(); g.setLineDash([]); };
  (M.candidates||[]).forEach((rt,i)=>{ if(i!==M.chosen) line(rt,'rgba(181,140,255,.7)',1.5,[3,4]); });
  line(M.route,'#ffd84a',2.5,[7,4]); line(M.path,'#5aa9ff',3.2);
  line(M.tour,'rgba(255,79,216,.85)',1.8,[3,5]);
  (M.destinations||[]).forEach(([r,c],i)=>{ g.strokeStyle='#ff4fd8'; g.lineWidth=i?1.6:2.5; g.beginPath(); g.arc(X(c),Y(r),cs*.42,0,7); g.stroke();
    g.fillStyle='#ff4fd8'; g.beginPath(); g.arc(X(c),Y(r),cs*.28,0,7); g.fill(); g.fillStyle='#1a0516'; g.font=`bold ${Math.max(9,cs*.36)}px system-ui,sans-serif`; g.textAlign='center'; g.textBaseline='middle'; g.fillText(i+1,X(c),Y(r)+1); });
  (S.ghosts||[]).forEach((G,i)=>{ const col=GHOST_COL[G.policy]||'#fff'; g.globalAlpha=G.finished?.35:.7; line(G.path,col,1.6,[2,3]); const [r,c]=G.position; g.fillStyle=col; g.beginPath(); g.arc(X(c)+(i-1)*cs*.2,Y(r)+cs*.24,cs*.17,0,7); g.fill(); g.globalAlpha=1; });
  if(S.drone){ const D=S.drone, q=cs*.24; if(D.flying){ g.fillStyle='rgba(34,211,238,.13)'; D.scanned.forEach(([r,c])=>g.fillRect(X(c)-cs/2,Y(r)-cs/2,cs,cs)); }
    (D.found||[]).forEach(([r,c])=>{ g.strokeStyle='#22d3ee'; g.lineWidth=2; g.strokeRect(X(c)-cs*.45,Y(r)-cs*.45,cs*.9,cs*.9); });
    const [r,c]=D.position, x=X(c), y=Y(r)-(D.flying?0:cs*.3); g.strokeStyle='#22d3ee'; g.lineWidth=2; g.beginPath(); g.moveTo(x-q,y-q); g.lineTo(x+q,y+q); g.moveTo(x+q,y-q); g.lineTo(x-q,y+q); g.stroke();
    g.fillStyle='#0b1418'; [[-1,-1],[1,-1],[-1,1],[1,1]].forEach(([a,b])=>{ g.beginPath(); g.arc(x+a*q,y+b*q,cs*.1,0,7); g.fill(); g.stroke(); }); }
  const [pr,pc]=S.status.position; const pth=M.path; let ang=0; if(pth.length>1){ const a=pth[pth.length-2]; ang=Math.atan2(pr-a[0],pc-a[1]); }
  g.save(); g.translate(X(pc),Y(pr)); g.rotate(ang); g.fillStyle='#f3f1ea'; g.strokeStyle='#10131a'; g.lineWidth=1.5; g.beginPath(); g.moveTo(cs*.42,0); g.lineTo(-cs*.3,cs*.3); g.lineTo(-cs*.14,0); g.lineTo(-cs*.3,-cs*.3); g.closePath(); g.fill(); g.stroke(); g.restore();
  const fw=cs*.42, fh=cs*.28, fx=X(pc)-cs*.5, fy=Y(pr)-cs*.62; g.fillStyle='#fff'; g.fillRect(fx,fy,fw,fh/2); g.fillStyle='#d52b1e'; g.fillRect(fx,fy+fh/2,fw,fh/2); g.fillStyle='#0039a6'; g.fillRect(fx,fy,fw/3,fh/2);
  if(!S.status.daylight||S.status.storm){ g.fillStyle=S.status.storm?'rgba(168,106,66,.28)':'rgba(5,6,14,.38)'; g.fillRect(ox,oy,cs*M.width,cs*M.height); }
  if(S.status.heat){ g.fillStyle='rgba(255,90,31,.14)'; g.fillRect(ox,oy,cs*M.width,cs*M.height); }
  g.fillStyle='#a08d78'; g.font='10px JetBrains Mono, monospace'; g.textAlign='center'; for(let c=0;c<M.width;c++) g.fillText(c,X(c),oy-4); g.textAlign='right'; for(let r=0;r<M.height;r++) g.fillText(r,ox-5,Y(r)+3); g.textAlign='left'; }
$('map2d').addEventListener('click',e=>{ if(!geom2d||!S) return; const b=$('map2d').getBoundingClientRect(); cellAction(Math.floor((e.clientY-b.top-geom2d.oy)/geom2d.cs),Math.floor((e.clientX-b.left-geom2d.ox)/geom2d.cs)); });
$('bDest').addEventListener('click',()=>{ $('fTool').value=''; setDestMode(!destMode); });
$('bDestClear').addEventListener('click',async()=>{ setDestMode(false); if(await call('destination',{position:null})) msg('Destinos quitados: el rover retoma la misión.'); });
$('fTool').addEventListener('change',()=>{ if($('fTool').value) destMode=false; setDestMode(destMode); if($('fTool').value) msg('Haga clic en una celda del mapa para aplicar la edición.'); });
$('fReveal').addEventListener('change',()=>{ if(S){ buildDynamic(); draw2d(); } });

// ------------------------------------------------------------ render
function render(){ if(!S) return; const st=S.status;
  window._lastMoved=S.log.length&&S.log[S.log.length-1].step===st.step?(S.log[S.log.length-1].moved||[]):[];
  renderTelemetry(); renderTop(); renderRace(); renderTimeline();
  try{ sync3d(); draw2d(); }catch(e){ console.error(e); msg('Error al dibujar el mapa: '+e.message,true); } // never freeze the panels
  renderNN(); renderFuzzy(); renderGA(); renderLog();
  if(raceDone()&&running){ running=false; syncRun(); msg((S.ghosts||[]).length?'Carrera terminada: revise el resultado sobre el mapa.':'Misión finalizada: '+st.outcome,st.state!=='EXITO'); } }
const raceDone=()=>S.status.finished&&(S.ghosts||[]).every(g=>g.finished);
function renderRace(){ const G=S.ghosts||[], el=$('ovRace'); if(!G.length){ el.style.display='none'; return; } el.style.display='block'; const st=S.status;
  const rows=[{policy:S.config.policy,battery:st.battery,science:st.science,state:st.state,finished:st.finished,collisions:S.metrics.collisions,main:true},...G];
  const done=rows.every(r=>r.finished), score=r=>(r.state==='EXITO'?1e6:0)+r.science*1e4+r.battery-r.collisions*500; if(done) rows.sort((a,b)=>score(b)-score(a));
  el.innerHTML=`<div class="rh">${done?'Resultado de la carrera':'Carrera de fantasmas'}</div><div class="rr small"><i></i><span style="text-align:left">política</span><span>bat.</span><span>ciencia</span><span>estado</span></div>`+rows.map((r,i)=>`<div class="rr"><i style="background:${r.main?'#e9e6df':GHOST_COL[r.policy]}"></i><b>${done?(i+1)+'. ':''}${GHOST_LBL[r.policy]||r.policy}${r.main?' · rover':''}</b><span>${fmt(r.battery,0)} %</span><span>${Math.round(r.science*100)} %</span><span class="st-${r.state}">${(r.state==='CONFIGURACION'?'EN ESPERA':r.state).replace('_',' ')}</span></div>`).join(''); }
// ------------------------------------------------------------ alerts at the top of the map (F2)
const ALERT_MS=6000, ALERT_MAX=3;
const EVENT_SEV={barranco:'warn',meteorito:'warn',sismo:'warn',tormenta_polvo:'warn',viento:'warn',ola_calor:'warn',falla_sensor:'warn',perdida_enlace:'warn',bateria_baja:'warn',corrosion:'warn',alienigena:'warn',obstaculo:'warn',obstaculo_manual:'warn',destino_descartado:'warn',revision_rna:'warn',destino_alcanzado:'ok',instrumento:'ok',dron_activado:'ok',dron_desactivado:'info'};
const REASON_SEV={R00_RECURSOS_AGOTADOS:'crit',R01_LIMITE_TERMICO:'crit',R03_DANO_CRITICO:'crit',R06_REFUGIO_INALCANZABLE:'crit',R02_SENSORES_NAVEGACION_FALLIDOS:'warn',R04_VETO_BARRANCO_CONFIRMADO:'warn',R05_RNA_PELIGRO:'warn',R07_RETORNO_ENERGIA:'warn',R10_SIN_RUTA:'warn'};
const REPEATING=new Set(['R04_VETO_BARRANCO_CONFIRMADO','R05_RNA_PELIGRO']);   // every veto is a new fact; the others alert once per change
const EVENT_TEXT={barranco:'Barranco real en el terreno, oculto para el rover: debe descubrirlo con sus sensores.',tormenta_polvo:'Baja la visibilidad: el rover reduce la velocidad y recalcula la ruta.',sismo:'Vibración y daño leve: el rover frena, espera y vuelve a medir.',meteorito:'Impacto en el terreno: aparece un obstáculo y la ruta se recalcula.',viento:'Ráfaga fuerte: sube el consumo y la ruta se recalcula.',ola_calor:'Sube la temperatura: más consumo y menor velocidad; ruta recalculada.',bateria_baja:'Cae la energía disponible: se recalcula la ruta priorizando el refugio.',falla_sensor:'Un sensor queda fuera de servicio: avance conservador con los restantes.',perdida_enlace:'Sin enlace con el operador: el rover sigue de forma autónoma.',biosenal_anomala:'Anomalía registrada para análisis. No confirma vida.',alienigena:'Evento narrativo ficticio: se registra el contacto y se evita la zona.',dia:'Amanece: mejora la visibilidad y vuelve la recarga solar.',noche:'Anochece: baja la visibilidad y se detiene la recarga solar.',corrosion:'La integridad baja por corrosión.'};
function pushAlert(sev,title,text){ const box=$('alerts'), el=document.createElement('div'); el.className='alert '+sev;
  const body=document.createElement('div'), b=document.createElement('b'), x=document.createElement('button'); b.textContent=title; body.append(b,document.createTextNode(text)); x.textContent='×'; x.title='Cerrar'; x.setAttribute('aria-label','Cerrar alerta'); x.onclick=()=>el.remove();
  el.append(body,x); box.append(el); while(box.children.length>ALERT_MAX) box.firstElementChild.remove();
  if(sev!=='crit') setTimeout(()=>el.remove(),ALERT_MS); }
function notify(o,prev){ const feed=o.event_feed; if(!feed) return;
  if(!prev||o.status.step===0&&feed.total<(prev.event_feed||{total:0}).total||o.config.seed!==prev.config.seed||o.planet.name!==prev.planet.name){ if(prev) $('alerts').replaceChildren(); return; }   // first load or a new mission: nothing to announce
  const fresh=Math.min(feed.recent.length,Math.max(0,feed.total-(prev.event_feed||{total:feed.total}).total));
  feed.recent.slice(feed.recent.length-fresh).forEach(e=>pushAlert(EVENT_SEV[e.kind]||'info',e.origin==='aleatorio'?e.label+' (aleatorio)':e.label,/^Evento aplicado/.test(e.message)&&EVENT_TEXT[e.kind]?EVENT_TEXT[e.kind]:e.message));
  const r=o.decision&&o.decision.reason, pr=prev.decision&&prev.decision.reason;
  if(r&&REASON_SEV[r]&&o.status.step!==prev.status.step&&(r!==pr||REPEATING.has(r))) pushAlert(REASON_SEV[r],r.replace(/_/g,' '),o.decision.explanation||(META.reasons||{})[r]||'');
  if(o.status.finished&&!prev.status.finished) pushAlert(o.status.state==='EXITO'?'ok':'crit','Fin de misión',o.status.outcome||''); }
// ------------------------------------------------------------ replay with timeline (F15)
let frames=[], replayAt=null;
function recordFrame(o){ const n=o.status.step; if(n===0||n<frames.length-1) frames=[]; frames[n]=o; frames.length=n+1; replayAt=null; }
function firstFrame(){ return Math.max(0,frames.findIndex(f=>f)); }
function seek(i){ if(!frames.length) return; running=false; syncRun(); const last=frames.length-1; i=Math.max(firstFrame(),Math.min(last,Math.round(i))); while(i>0&&!frames[i]) i--; if(!frames[i]) return;
  replayAt=i>=last?null:i; S=frames[i]; lastPos=null; V.anim&&(V.anim.queue=[]); render(); }
function renderTimeline(){ const last=Math.max(0,frames.length-1), at=replayAt===null?last:replayAt, sl=$('tSlider'); sl.min=firstFrame(); sl.max=last; sl.value=at;
  $('tLabel').textContent=`paso ${at}/${last}`; $('tLive').classList.toggle('on',replayAt===null); $('tLive').textContent=replayAt===null?'● En vivo':'Volver al presente'; $('ovTime').classList.toggle('replay',replayAt!==null);
  $('alerts').style.display=replayAt===null?'flex':'none';
  const why=$('ovWhy'); if(replayAt===null){ why.style.display='none'; return; } why.style.display='block';
  const L=S.log.length&&S.log[S.log.length-1].step===S.status.step?S.log[S.log.length-1]:null;
  why.innerHTML=`<b>REPETICIÓN · PASO ${at}</b> · ¿por qué hizo esto?<br>`+(L?`<span class="mono">${L.reason}</span> → ${L.action}. ${L.explanation||(META.reasons||{})[L.reason]||''}<br><span class="small">RNA: ${L.rna||'—'}${L.conf!=null?' ('+fmt(L.conf*100,0)+' %)':''} · velocidad difusa ${L.speed!=null?fmt(L.speed,2):'—'} · batería ${fmt(L.battery,1)} %${(L.events||[]).length?' · eventos: '+L.events.join(', '):''}</span>`:'Inicio de la misión: ruta inicial planificada, aún sin decisiones.'); }
$('tSlider').addEventListener('input',e=>seek(+e.target.value));
$('tPrev').addEventListener('click',()=>seek((replayAt===null?frames.length-1:replayAt)-1));
$('tNext').addEventListener('click',()=>seek((replayAt===null?frames.length-1:replayAt)+1));
$('tLive').addEventListener('click',()=>seek(frames.length-1));
function tl(name,unit,val,status,chipText){ return `<div class="tl ${status}"><div><div class="n">${name}</div><div class="u">${unit}</div></div><div class="v">${val}</div><span class="chip ${status}">${chipText||{ok:'NOMINAL',warn:'ALERTA',bad:'CRÍTICO',ref:'REF.'}[status||'ok']}</span></div>`; }
function renderTelemetry(){ const st=S.status, T=S.telemetry, P=S.planet;
  $('pName').textContent=P.name; $('pInfo').innerHTML=`${P.mode==='rover'?'Rover de superficie':'Sonda atmosférica virtual'}<br>g ${fmt(P.gravity,2)} m/s² · ${fmt(P.temperature,0)} °C ref.`; drawPlanetIcon(P.name);
  $('tSub').textContent=`SOL ${String(Math.floor(st.step/60)+1).padStart(3,'0')} · ${st.daylight?'día':'noche'}${st.storm?' · tormenta':''}${st.heat?' · ola de calor':''}${st.wind?' · viento':''}`;
  const bat=st.battery, nv=T.nav_valid||{};
  // a sensor row without a sample says so; it is never classified as nominal or alert (F02)
  const has=v=>v!==null&&v!==undefined&&!Number.isNaN(v);
  const sensed=(name,unit,v,val,status,chip)=>has(v)?tl(name,unit,val,status,chip):tl(name,unit,'—','ref','SIN DATOS');
  const pa=has(T.pressure)&&T.pressure>=1e5;   // the unit label follows the unit of the value (F01)
  $('tele').innerHTML=[
    tl('Batería','% · ciclo solar',fmt(bat,1)+' %',bat<15?'bad':bat<35?'warn':'ok'),
    tl('Integridad','% mecánica',fmt(st.health,0)+' %',st.health<30?'bad':st.health<70?'warn':'ok'),
    sensed('Temp. exterior','°C · sensor',T.temperature_external,fmt(T.temperature_external,1),'ref','AMBIENTE'),
    tl('Temp. componentes','°C · interna',fmt(st.temperature_internal,1),st.temperature_internal>60?'bad':st.temperature_internal>45?'warn':'ok',st.heat?`CALOR ${st.heat}`:null),
    sensed('O₂ exterior (ambiental)','% vol. · no respiración',T.oxygen_ambient,fmt(T.oxygen_ambient,2),'ref'),
    sensed('O₂ almacenado','% · experimentos',T.oxygen_stored,fmt(T.oxygen_stored,0),'ok'),
    sensed('Presión',(pa?'bar':'Pa')+' · referencia',T.pressure,pa?(T.pressure/1e5).toFixed(1)+' bar':Math.round(T.pressure),'ref'),
    tl('Gravedad','m/s²',fmt(P.gravity,2),'ref'),
    sensed('Radiación (índice)','mSv/h simulado',T.radiation,fmt(T.radiation,2),'ok'),
    sensed('Pendiente','° inclinómetro',T.slope,fmt(T.slope,1),T.slope>30?'bad':T.slope>15?'warn':'ok'),
    nv.proximidad===false?tl('Proximidad','m · frontal','NaN','bad','FALLA'):sensed('Proximidad','m · frontal',T.distance,fmt(T.distance,2),'ok'),   // a confirmed sensor failure stays apart
    nv.lidar===false?tl('LIDAR / borde','Δ elevación m','NaN','bad','FALLA'):sensed('LIDAR / borde','Δ elevación m',T.elevation_delta,fmt(T.elevation_delta,1),T.edge_cliff?'bad':'ok',T.edge_cliff?'BARRANCO':null),
    tl('Visibilidad','0–1 · partículas',fmt(st.visibility,2),st.visibility<.4?'bad':st.visibility<.8?'warn':'ok'),
    sensed('Potencia solar','W simulados',T.solar_power,fmt(T.solar_power,1),T.solar_power<1?'warn':'ok'),
    tl('Enlace','latencia s · órdenes en cola',st.link?'OK':`${fmt(st.link_latency,0)} s · ${st.queued_orders}`,st.link?'ok':'warn',st.link?null:'AUTÓNOMO'),
    tl('Vibración','m/s² · sismómetro',fmt(st.vibration,1),st.vibration>0?'warn':'ok'),
    tl('Viento','m/s · consumo +30 %',fmt(st.wind,0),st.wind>0?'warn':'ok'),
    tl('Eventos','total · anomalías · contactos',`${st.events_total} · ${st.anomalies} · ${st.contacts}`,'ref','CONTEO'),
    tl('Pasos perdidos','espera · sensor degradado',`${st.wait_steps} · ${st.degraded_steps}`,'ref','CONTEO'),
    tl('Ciencia','% sitios visitados',Math.round(st.science*100)+' %','ok','PROGRESO'),
    tl('Dron explorador','peligros detectados',S.drone?S.metrics.drone_discoveries:'—',S.drone?(S.drone.flying?'ok':'warn'):'ref',S.drone?(S.drone.flying?'VOLANDO':'EN TIERRA'):'APAGADO'),
    tl('Destinos','pendientes · celdas al próximo',S.map.destination?`${st.destinations_left} · ${st.destination_distance??'—'}`:'—',S.map.destination?'warn':'ref',S.map.destination?'EN RUTA':'SIN FIJAR'),
    tl('Instrumentación','cajas recuperadas',`${st.instruments}/${st.instruments_total}`,st.instruments>=st.instruments_total?'ok':'ref',st.instruments>=st.instruments_total?'COMPLETO':'PENDIENTE')].join(''); }
function renderTop(){ const st=S.status; $('bGhost').classList.toggle('on',(S.ghosts||[]).length>0); $('bDrone').classList.toggle('on',!!S.config.drone); $('bDrone').title=S.drone&&S.drone.reason?S.drone.reason:'Un dron vuela adelante y comparte los peligros que ve';
  if(document.activeElement!==$('fEvents')){ const ev=Math.round((S.config.event_rate||0)*100); $('fEvents').value=ev; $('lEv').textContent=ev+' %'; }
  if(document.activeElement!==$('fDur')){ $('fDur').value=S.config.step_duration; $('lDur').textContent=(+S.config.step_duration).toFixed(1)+' s'; }
  $('oPlanet').textContent=`${S.planet.name.toUpperCase()} · SOL ${String(Math.floor(st.step/60)+1).padStart(3,'0')}`;
  $('oStep').textContent=`paso ${st.step} · t=${fmt(st.t,1)} s`; $('oPolicy').textContent=S.policy_label; $('cSeed').textContent=`semilla ${S.config.seed}`;
  const label={CONFIGURACION:'EN ESPERA'}[st.state]||st.state.replace('_',' ');
  $('sState').textContent=label+(st.finished?' · FIN':''); $('sState').className='st st-'+st.state;
  const D=S.decision; $('sExpl').textContent=st.finished?st.outcome:(D?`${D.reason}: ${D.explanation}`:'Configure la misión y presione Iniciar.');
  document.querySelectorAll('[data-planet]').forEach(b=>b.classList.toggle('sel',b.dataset.planet===S.planet.name));
  const ds=S.map.destinations||[]; $('destInfo').textContent=ds.length?`${ds.length} destino(s) en orden de visita: `+ds.map((d,i)=>`${i+1}·(${d[0]},${d[1]})`).join(' → ')+` · recorrido ${st.tour_cells} celdas con retorno`:'Sin destinos. Presione «Agregar destinos» y haga clic en una o varias celdas: el AG calcula el mejor orden de visita.'; }

// ------------------------------------------------------------ D · neural network
const FEAT_LBL={distance:'Distancia',slope:'Pendiente',elevation_delta:'Δ elevación',texture:'Textura',visibility:'Visibilidad',vibration:'Vibración'};
let nnMode='live', nnTest=null;
const NN_DEFAULTS={distance:10,visibility:1,texture:.25};
let nnTestInputs=null; // last values evaluated in Prueba mode, kept apart from the live mission inference
function buildFeat(){ nnTestInputs=Object.fromEntries(META.features.map(f=>[f,NN_DEFAULTS[f]??0]));
  $('nnFeat').innerHTML=META.features.map(f=>`<div><label title="${FEAT_LBL[f]}">${FEAT_LBL[f]}</label><input type="number" step="0.1" id="nf_${f}" value="${nnTestInputs[f]}"></div>`).join('');
  META.features.forEach(f=>$('nf_'+f).addEventListener('input',()=>{ setNN('test'); nnEvalSoon(); })); }
let nnT=null; function nnEvalSoon(){ clearTimeout(nnT); nnT=setTimeout(async()=>{ const a={}; META.features.forEach(f=>a[f]=+$('nf_'+f).value); nnTestInputs={...a}; const o=await api('rna_eval',a); if(o.error){ msg(o.error,true); return; } nnTest=o; renderNN(); },120); }
function setNN(m){ if(m==='test') META.features.forEach(f=>{ const el=$('nf_'+f); if(el.value==='') el.value=nnTestInputs[f]; }); // a blank live field starts from the last Prueba value
  nnMode=m; document.querySelectorAll('[data-nn]').forEach(b=>b.classList.toggle('on',b.dataset.nn===m)); if(m==='test'&&!nnTest) nnEvalSoon(); renderNN(); }
document.querySelectorAll('[data-nn]').forEach(b=>b.addEventListener('click',()=>setNN(b.dataset.nn)));
function forward(x){ const N=META.network; const acts=[x]; let a=x; N.coefs.forEach((W,li)=>{ const b=N.intercepts[li]; const out=b.map((bj,j)=>{ let s=bj; for(let i=0;i<a.length;i++) s+=a[i]*W[i][j]; return s; });
  a=li<N.coefs.length-1?out.map(v=>Math.max(0,v)):(()=>{ const m=Math.max(...out), e=out.map(v=>Math.exp(v-m)), s=e.reduce((p,q)=>p+q,0); return e.map(v=>v/s); })(); acts.push(a); }); return acts; }
function renderNN(){ if(!META||!S) return; const src=nnMode==='test'?nnTest:S.rna;
  const t=S.nn.test; $('nnSub').textContent=`${META.network.layers.join('-')} · acc ${fmt(t.accuracy,3)}`;
  $('nnMetrics').innerHTML=`exactitud ${fmt(t.accuracy,3)} · baseline ${fmt(t.majority_baseline,3)} · macro-F1 ${fmt(t.macro_f1,3)}<br>recall barranco ${fmt(t.barranco_recall,3)} · ECE ${fmt(t.calibration_error,3)} · n=${t.samples}`;
  if(nnMode==='live') META.features.forEach(f=>{ const el=$('nf_'+f); if(document.activeElement===el) return; // no prediction: blank fields, never stale readings
    el.value=S.rna?fmt(S.rna.features[f],2):''; el.placeholder=S.rna?'':'—'; el.title=S.rna?'':'Sin predicción vigente: ejecute un paso o escriba un valor para usar el modo Prueba'; });
  $('nnCell').textContent=nnMode==='test'?'(modo prueba: edite valores)':S.rna?`(${S.rna.cell})`:'';
  const c=$('nnNet'), [g,w,h]=ctx2d(c); const N=META.network; const L=N.layers;
  const x=src&&src.normalized?src.normalized:null; const acts=x?forward(x):null;
  const colX=i=>62+(w-152)*i/(L.length-1); const nodeY=(li,k)=>{ const n=L[li]; const span=h-16; return 8+(n===1?span/2:span*k/(n-1)); };
  for(let li=0;li<N.coefs.length;li++){ const W=N.coefs[li]; let mx=0; W.forEach(r=>r.forEach(v=>mx=Math.max(mx,Math.abs(v))));
    for(let i=0;i<L[li];i++) for(let j=0;j<L[li+1];j++){ const v=W[i][j]; const act=acts?Math.min(1,Math.abs(acts[li][i])):.5; const al=Math.min(.55,Math.abs(v)/mx*.6*(acts?.25+act:.5)); if(al<.03) continue;
      g.strokeStyle=v>0?`rgba(255,154,60,${al})`:`rgba(90,169,255,${al})`; g.lineWidth=.7; g.beginPath(); g.moveTo(colX(li),nodeY(li,i)); g.lineTo(colX(li+1),nodeY(li+1,j)); g.stroke(); } }
  L.forEach((n,li)=>{ let mx=1; if(acts) mx=Math.max(1e-6,...acts[li].map(Math.abs)); for(let k=0;k<n;k++){ const v=acts?Math.abs(acts[li][k])/mx:0; const r=li===0||li===L.length-1?4:2.6;
      g.fillStyle=acts?`rgba(255,${Math.round(150+v*90)},${Math.round(60+v*40)},${.25+v*.75})`:'#3d2f24'; g.beginPath(); g.arc(colX(li),nodeY(li,k),r,0,7); g.fill(); } });
  g.font='10px Barlow, sans-serif'; g.fillStyle=css('--muted'); g.textAlign='right';
  META.features.forEach((f,k)=>g.fillText(FEAT_LBL[f],colX(0)-7,nodeY(0,k)+3)); g.textAlign='left';
  N.classes.forEach((cl,k)=>{ const p=acts?acts[L.length-1][k]:null; g.fillStyle=cl==='barranco'?css('--bad'):cl==='obstaculo'?css('--amber'):css('--ok'); g.fillText(`${cl} ${p===null?'':fmt(p,2)}`,colX(L.length-1)+7,nodeY(L.length-1,k)+3); });
  const P=src?src.probabilities:null; const cols={libre:css('--ok'),obstaculo:css('--amber'),barranco:css('--bad')};
  $('nnProbs').innerHTML=P?Object.entries(P).map(([k,v])=>`<div class="prob"><span>${k}</span><div class="t"><span style="width:${v*100}%;background:${cols[k]}"></span></div><span class="mono">${fmt(v,3)}</span></div>`).join(''):'<div class="small">Sin predicción: ejecute un paso con una política que use RNA o active el modo Prueba.</div>';
  $('nnDecision').innerHTML=src?`→ <b>${src.label}</b> (confianza ${fmt(src.confidence,2)})${src.low_confidence?' · <span class="r-warn">baja confianza: fusión con sensor determinista</span>':''}${src.confidence>=.65&&src.label!=='libre'?' · <span class="r-veto">veto del supervisor (p ≥ 0,65)</span>':''}`:(S.rna_error?`<span class="r-veto">${S.rna_error}</span>`:'');
  const cm=t.confusion_matrix, labs=t.labels, [g2,w2,h2]=ctx2d($('nnConf')); const cw=(w2-70)/3, ch=(h2-14)/3; let mxv=Math.max(...cm.flat());
  g2.font='9.5px JetBrains Mono, monospace'; labs.forEach((l,i)=>{ g2.fillStyle=css('--dim'); g2.textAlign='right'; g2.fillText(l,64,14+i*ch+ch/2); g2.textAlign='center'; g2.fillText(l,70+i*cw+cw/2,10);
    labs.forEach((_,j)=>{ const v=cm[i][j]; g2.fillStyle=i===j?`rgba(82,213,142,${.15+v/mxv*.6})`:v?`rgba(255,90,79,${.2+v/mxv*.6})`:'#17110c'; g2.fillRect(70+j*cw+1,14+i*ch+1,cw-2,ch-2); g2.fillStyle=css('--text'); g2.fillText(v,70+j*cw+cw/2,14+i*ch+ch/2+3); }); }); g2.textAlign='left'; renderCurve(); }

function renderCurve(){ const H=META&&META.training; if(!H||!H.length) return; const sl=$('fIter');
  if(+sl.max!==H.length){ sl.max=H.length; sl.value=H.length; }
  const k=Math.max(1,Math.min(H.length,+sl.value)), row=H[k-1], [g,w,h]=ctx2d($('nnCurve')); const mx=Math.max(...H.map(r=>r.loss));
  const X=i=>26+(w-32)*i/Math.max(1,H.length-1), Y=v=>6+(h-18)*(1-v);
  g.strokeStyle='#2b2119'; g.font='9px JetBrains Mono'; [0,.5,1].forEach(t=>{ g.beginPath(); g.moveTo(26,Y(t)); g.lineTo(w-6,Y(t)); g.stroke(); g.fillStyle=css('--dim'); g.fillText(t.toFixed(1),2,Y(t)+3); });
  [[css('--bad'),r=>r.loss/mx],[css('--amber'),r=>r.train_accuracy],[css('--ok'),r=>r.validation_accuracy]].forEach(([col,f])=>{ g.strokeStyle=col; g.lineWidth=1.5; g.beginPath(); H.forEach((r,i)=>{ i?g.lineTo(X(i),Y(f(r))):g.moveTo(X(i),Y(f(r))); }); g.stroke(); });
  g.strokeStyle='#fff'; g.setLineDash([3,3]); g.beginPath(); g.moveTo(X(k-1),4); g.lineTo(X(k-1),h-12); g.stroke(); g.setLineDash([]);
  g.fillStyle=css('--bad'); g.fillText('pérdida (norm.)',28,h-1); g.fillStyle=css('--amber'); g.fillText('acc entren.',124,h-1); g.fillStyle=css('--ok'); g.fillText('acc valid.',196,h-1);
  $('nnIter').textContent=`iteración ${row.iteration}/${H.length} · pérdida ${fmt(row.loss,4)} · acc entren. ${fmt(row.train_accuracy,3)} · acc valid. ${fmt(row.validation_accuracy,3)}`; }
$('fIter').addEventListener('input',renderCurve);

// ------------------------------------------------------------ E · fuzzy
const FZV=[['distance','Distancia a riesgo','m'],['battery','Batería','%'],['slope','Pendiente','°'],['temperature','Temp. componentes','°C']];
const FZC=['#5aa9ff','#ffc04d','#ff5a4f','#52d58e'];
let fzMode='live', fzTest=null, fzInputs={distance:10,battery:60,slope:5,temperature:12};
function setFZ(m){ fzMode=m; document.querySelectorAll('[data-fz]').forEach(b=>b.classList.toggle('on',b.dataset.fz===m)); if(m==='test') fzEvalSoon(); else renderFuzzy(); }
document.querySelectorAll('[data-fz]').forEach(b=>b.addEventListener('click',()=>setFZ(b.dataset.fz)));
let fzT=null; function fzEvalSoon(){ clearTimeout(fzT); fzT=setTimeout(async()=>{ const o=await api('fuzzy_eval',fzInputs); if(o.error){ msg(o.error,true); return; } fzTest=o; renderFuzzy(); },80); }
$('fzSets').addEventListener('click',e=>{ if(!META) return; const r=$('fzSets').getBoundingClientRect(); const row=Math.floor((e.clientY-r.top)/(r.height/4)); const v=FZV[row]; if(!v) return;
  const set=META.fuzzy_sets[v[0]]; const lo=set.x[0], hi=set.x[set.x.length-1]; const t=Math.max(0,Math.min(1,(e.clientX-r.left-8)/(r.width-16)));
  fzInputs[v[0]]=+(lo+(hi-lo)*t).toFixed(1); if(fzMode==='live'&&S&&S.fuzzy) Object.assign(fzInputs,{...S.fuzzy.inputs,[v[0]]:fzInputs[v[0]]}); setFZ('test'); });
function renderFuzzy(){ if(!META||!S) return; const F=fzMode==='test'?fzTest:S.fuzzy;
  // three states: inference shown, inference pending (policy uses fuzzy), controller disabled by the policy
  const disabled=fzMode==='live'&&S.policy_uses&&!S.policy_uses.fuzzy;
  const pending=fzMode==='test'?'Prueba: evaluando…':S.status.step===0?'Sin inferencia todavía: ejecute un paso.':'Sin inferencia difusa en este paso.';
  $('fzSub').textContent=F?`${F.rules.length}/${F.all_rules.length} reglas activas · v=${fmt(F.speed,2)}`:disabled?'desactivada en esta política':pending;
  const c=$('fzSets'), [g,w,h]=ctx2d(c); const rh=h/4;
  FZV.forEach(([k,lbl,u],i)=>{ const set=META.fuzzy_sets[k], y0=i*rh, xs=set.x, lo=xs[0], hi=xs[xs.length-1]; const X=v=>8+(w-16)*(v-lo)/(hi-lo), Y=v=>y0+rh-12-(rh-26)*v;
    g.strokeStyle='#2b2119'; g.beginPath(); g.moveTo(8,y0+rh-12); g.lineTo(w-8,y0+rh-12); g.stroke();
    Object.entries(set.sets).forEach(([name,ys],j)=>{ g.strokeStyle=FZC[j]; g.lineWidth=1.5; g.beginPath(); ys.forEach((yv,n)=>{ n?g.lineTo(X(xs[n]),Y(yv)):g.moveTo(X(xs[n]),Y(yv)); }); g.stroke(); });
    const val=F&&F.inputs?F.inputs[k]:null; g.font='600 10.5px Barlow, sans-serif'; g.fillStyle=css('--text'); g.fillText(lbl,8,y0+11);
    if(val!==null&&val!==undefined){ const xv=X(Math.max(lo,Math.min(hi,val))); g.strokeStyle='#fff'; g.setLineDash([3,3]); g.beginPath(); g.moveTo(xv,y0+14); g.lineTo(xv,y0+rh-12); g.stroke(); g.setLineDash([]);
      const deg=F.degrees[k]||{}; g.textAlign='right'; g.font='10px JetBrains Mono, monospace'; g.fillStyle=css('--amber');
      g.fillText(`x=${fmt(val,1)}${u} · `+Object.entries(deg).filter(([,d])=>d>0).map(([s,d])=>`${s} ${fmt(d,2)}`).join(' · '),w-8,y0+11); g.textAlign='left';
      Object.entries(set.sets).forEach(([name,ys],j)=>{ const d=(F.degrees[k]||{})[name]; if(d>0){ g.fillStyle=FZC[j]; g.beginPath(); g.arc(xv,Y(d),3,0,7); g.fill(); } }); }
    g.font='9.5px Barlow, sans-serif'; g.fillStyle=css('--dim'); g.fillText(fmt(lo,0),8,y0+rh-2); g.textAlign='right'; g.fillText(fmt(hi,0)+' '+u,w-8,y0+rh-2); g.textAlign='left';
    Object.keys(set.sets).forEach((name,j)=>{ g.fillStyle=FZC[j]; g.fillText(name,40+j*((w-80)/3),y0+rh-2); }); });
  const sp=META.fuzzy_sets.speed, [g2,w2,h2]=ctx2d($('fzAgg')); const X=v=>8+(w2-16)*v, Y=v=>h2-12-(h2-20)*v;
  Object.entries(sp.sets).forEach(([name,ys],j)=>{ g2.strokeStyle=FZC[j]+'88'; g2.lineWidth=1; g2.beginPath(); ys.forEach((yv,n)=>{ n?g2.lineTo(X(sp.x[n]),Y(yv)):g2.moveTo(X(sp.x[n]),Y(yv)); }); g2.stroke(); g2.fillStyle=FZC[j]; g2.font='9.5px Barlow'; g2.fillText(name,12+j*((w2-24)/4),h2-1); });
  if(F&&F.curve&&F.curve.length){ g2.fillStyle='rgba(255,192,77,.35)'; g2.strokeStyle=css('--amber'); g2.lineWidth=1.8; g2.beginPath(); g2.moveTo(X(0),Y(0)); F.curve.forEach((v,n)=>g2.lineTo(X(n/(F.curve.length-1)),Y(v))); g2.lineTo(X(1),Y(0)); g2.closePath(); g2.fill(); g2.stroke();
    const xc=X(F.centroid); g2.strokeStyle='#fff'; g2.lineWidth=2; g2.beginPath(); g2.moveTo(xc,4); g2.lineTo(xc,h2-12); g2.stroke(); g2.fillStyle='#fff'; g2.font='600 10.5px JetBrains Mono'; g2.fillText(`centroide ${fmt(F.centroid,3)} → velocidad`,Math.min(xc+5,w2-190),12); }
  else { g2.fillStyle=css('--muted'); g2.font='11px Barlow'; g2.fillText(F&&F.reason==='SIN_REGLAS_ACTIVAS'?F.fallback:disabled?'Política sin lógica difusa: velocidad fija 0,50.':pending,10,22); }
  const act=new Map((F?F.rules:[]).map(r=>[r.id,r.strength])); const rules=(F?F.all_rules:null)||(META.rules_fz||[]);
  const VAR={distance:'distancia',battery:'batería',slope:'pendiente',temperature:'temperatura'};
  $('fzRules').innerHTML=(F?F.all_rules:[]).slice().sort((a,b)=>(act.get(b.id)||0)-(act.get(a.id)||0)).map(r=>`<div class="rule ${act.has(r.id)?'act':''}"><b>R${r.id}</b><span>SI ${r.antecedents.map(([v,s])=>`${VAR[v]} <i>${s}</i>`).join(' Y ')} → <i>${r.consequent}</i></span><b>${act.has(r.id)?fmt(act.get(r.id),2):'0'}</b></div>`).join('')
    ||`<div class="small">${disabled?'Política sin lógica difusa: velocidad fija 0,50.':pending}</div>`; }

// ------------------------------------------------------------ F · genetic algorithm
let gaRunSel=null, gaRunData=null, gaParamsBuilt=false;
const GA_P=[['ga_population','Tamaño de población',30,120,1,0],['ga_generations','Generaciones',30,150,1,0],['ga_crossover','Probabilidad de cruce',0,1,.01,2],['ga_mutation','Probabilidad de mutación',0,1,.01,2]];
const WN={science:'w1 ciencia',distance:'w2 distancia',energy:'w3 energía',predicted_risk:'w4 riesgo',time:'w5 tiempo'};
function buildGaParams(){ const cfg=S.config, w=cfg.weights||{science:3,distance:1,energy:1,predicted_risk:2,time:1};
  $('gaParams').innerHTML=GA_P.map(([k,l,lo,hi,st,d])=>`<div class="sl">${l}<span id="gv_${k}">${fmt(+cfg[k],d)}</span><input type="range" id="gp_${k}" min="${lo}" max="${hi}" step="${st}" value="${cfg[k]}"></div>`).join('');
  $('gaWeights').innerHTML=Object.entries(WN).map(([k,l])=>`<div class="sl">${l}<span id="wv_${k}">${fmt(w[k]??1,1)}</span><input type="range" id="w_${k}" min="0" max="6" step=".1" value="${w[k]??1}"></div>`).join('');
  GA_P.forEach(([k,,,,,d])=>$('gp_'+k).addEventListener('input',()=>$('gv_'+k).textContent=fmt(+$('gp_'+k).value,d)));
  Object.keys(WN).forEach(k=>$('w_'+k).addEventListener('input',()=>$('wv_'+k).textContent=fmt(+$('w_'+k).value,1))); gaParamsBuilt=true; }
$('bGaApply').addEventListener('click',async()=>{ const args={...formConfig(),weights:{}}; GA_P.forEach(([k])=>args[k]=+$('gp_'+k).value); Object.keys(WN).forEach(k=>args.weights[k]=+$('w_'+k).value);
  running=false; syncRun(); const o=await call('configure',args); if(o){ gaRunSel=null; msg('Parámetros del AG aplicados; misión reiniciada con la misma semilla.'); } });
$('fRun').addEventListener('change',()=>{ gaRunSel=+$('fRun').value; $('fGen').value=5; renderGA(true); });
$('fGen').addEventListener('input',()=>{ clearTimeout(window._gt); window._gt=setTimeout(gaGen,70); });
async function gaGen(){ if(!S||!(S.ga_runs||[]).length){ $('gaChrom').innerHTML=''; $('gaOps').textContent=''; $('gaGenLbl').textContent='—'; return; }
  const o=await api('ga_generation',{generation:+$('fGen').value,...(gaRunSel===null?{}:{run:gaRunSel})}); if(o.error) return;
  $('gaGenLbl').textContent=`${o.generation}/${o.total-1}`; $('gaOps').textContent=`torneos ${o.tournaments} · cruces ${o.crossovers} · genes mutados ${o.mutations} · élite 2 · mejor ${fmt(o.best,3)}`;
  $('gaChrom').innerHTML='<table class="mt"><tr><th>#</th><th>cromosoma (puntos de paso)</th><th>fitness</th><th></th></tr>'+o.rows.slice(0,5).map((r,i)=>`<tr><td>${i}</td><td>${r.chromosome.map(p=>`(${p})`).join('→')}</td><td>${fmt(r.fitness,3)}</td><td>${r.feasible?'<span class="pill ok">ok</span>':`<span class="pill bad">${r.violations[0]}</span>`}</td></tr>`).join('')+'</table>'; }
async function renderGA(force){ if(!S) return; if(!gaParamsBuilt) buildGaParams();
  const runs=S.ga_runs||[]; if(gaRunSel!==null&&gaRunSel>=runs.length) gaRunSel=null;
  $('fRun').innerHTML=runs.length?runs.map(r=>`<option value="${r.index}">Ejecución #${r.index} · paso ${r.step} · ${r.reason}</option>`).join(''):'<option>sin ejecuciones del AG</option>';
  $('fRun').value=gaRunSel===null?Math.max(0,runs.length-1):gaRunSel;
  let G=S.ga; if(runs.length&&gaRunSel!==null){ const o=await api('ga_run',{run:gaRunSel}); if(!o.error) G=o.ga; }
  const cands=G.candidates||[]; $('gaSub').textContent=G.mode==='ag'?`${cands.length} rutas candidatas`:(G.mode||'—');
  const n=(G.history||[]).length; $('fGen').max=Math.max(0,n-1); if(+$('fGen').value>n-1) $('fGen').value=Math.max(0,n-1);
  // mini map
  const M=S.map, [g,w,h]=ctx2d($('gaMap')); const cs=Math.min(w/M.width,h/M.height), ox=(w-cs*M.width)/2, oy=(h-cs*M.height)/2;
  g.fillStyle='#1a120c'; g.fillRect(ox,oy,cs*M.width,cs*M.height);
  for(let r=0;r<M.height;r++) for(let c=0;c<M.width;c++){ const e=M.elevation[r][c]; const t=Math.max(0,Math.min(1,e/3)); g.fillStyle=`rgb(${48+t*40},${32+t*22},${22+t*12})`; g.fillRect(ox+c*cs,oy+r*cs,cs-.6,cs-.6); }
  M.risk.forEach(([r,c,v])=>{ g.fillStyle=`rgba(255,154,60,${v*.6})`; g.fillRect(ox+c*cs,oy+r*cs,cs,cs); });
  M.obstacles.forEach(([r,c])=>{ g.fillStyle='#8a6a4c'; g.fillRect(ox+c*cs+1,oy+r*cs+1,cs-2,cs-2); }); M.cliffs.forEach(([r,c])=>{ g.fillStyle='#c0392b'; g.fillRect(ox+c*cs,oy+r*cs,cs,cs); });
  const P=(cells,col,lw)=>{ if(!cells||cells.length<2) return; g.strokeStyle=col; g.lineWidth=lw; g.beginPath(); cells.forEach(([r,c],i)=>{ const X=ox+c*cs+cs/2, Y=oy+r*cs+cs/2; i?g.lineTo(X,Y):g.moveTo(X,Y); }); g.stroke(); };
  if(gaRunSel===null){ (M.candidates||[]).forEach((rt,i)=>{ if(i!==M.chosen) P(rt,'rgba(181,140,255,.6)',1); }); P(M.route,'#ffd84a',2); }
  else { const R=G.routes||[]; R.forEach((rt,i)=>{ if(i!==G.chosen) P(rt,'rgba(181,140,255,.6)',1); }); P(R[G.chosen],'#ffd84a',2); } // routes of the selected past run
  M.science.forEach(([r,c,d])=>{ g.fillStyle=d?'#52d58e':'#ffd84a'; g.beginPath(); g.arc(ox+c*cs+cs/2,oy+r*cs+cs/2,cs*.32,0,7); g.fill(); });
  M.resources.forEach(([r,c,d])=>{ g.fillStyle=d?'#52d58e':'#5aa9ff'; g.fillRect(ox+c*cs+cs*.25,oy+r*cs+cs*.25,cs*.5,cs*.5); });
  M.refuges.forEach(([r,c])=>{ g.strokeStyle='#52d58e'; g.lineWidth=1.5; g.strokeRect(ox+c*cs+1,oy+r*cs+1,cs-2,cs-2); });
  (M.destinations||[]).forEach(([r,c],i)=>{ g.strokeStyle='#ff4fd8'; g.lineWidth=i?1:2; g.beginPath(); g.arc(ox+c*cs+cs/2,oy+r*cs+cs/2,cs*.55,0,7); g.stroke(); });
  const [pr,pc]=S.status.position; g.fillStyle='#5aa9ff'; g.beginPath(); g.arc(ox+pc*cs+cs/2,oy+pr*cs+cs/2,cs*.38,0,7); g.fill();
  // fitness chart
  const [g2,w2,h2]=ctx2d($('gaFit')); const hb=(G.history||[]).map(x=>x.best), hf=(G.history||[]).map(x=>x.best_feasible);
  const all=hb.concat(hf).filter(v=>v!==null&&v!==undefined); const lo=Math.min(...all,0), hi=Math.max(...all,1);
  g2.strokeStyle='#2b2119'; [0,.5,1].forEach(t=>{ const y=6+(h2-16)*t; g2.beginPath(); g2.moveTo(26,y); g2.lineTo(w2-4,y); g2.stroke(); g2.fillStyle=css('--dim'); g2.font='9px JetBrains Mono'; g2.fillText((hi-(hi-lo)*t).toFixed(1),0,y+3); });
  [[hb,css('--violet')],[hf,css('--amber')]].forEach(([d,col])=>{ g2.strokeStyle=col; g2.lineWidth=1.6; g2.beginPath(); let s=false; d.forEach((v,i)=>{ if(v===null||v===undefined) return; const X=26+(w2-30)*i/Math.max(1,d.length-1), Y=6+(h2-16)*(1-(v-lo)/((hi-lo)||1)); s?g2.lineTo(X,Y):g2.moveTo(X,Y); s=true; }); g2.stroke(); });
  g2.fillStyle=css('--violet'); g2.fillText('mejor',30,h2-1); g2.fillStyle=css('--amber'); g2.fillText('mejor factible',72,h2-1);
  $('gaInfo').textContent=G.mode==='ag'?`pob. ${S.config.ga_population} · ${G.generations_run} gen${G.stopped_early?' (parada temprana)':''} · torneo 3 · elitismo 2`:(G.message||'La política actual no usa AG.');
  const rej=new Map((G.rejected||[]).map(r=>[r.index,r]));
  $('gaCand').innerHTML=cands.length?'<table class="mt"><tr><th>#</th><th>fitness</th><th>cien.</th><th>dist.</th><th>energ.</th><th>riesgo</th><th>estado</th></tr>'+cands.map(c=>`<tr style="${c.index===G.chosen?'color:var(--sci)':''}"><td>${c.index}</td><td>${fmt(c.fitness,3)}</td><td>${fmt(c.components.science,2)}</td><td>${fmt(c.components.distance,2)}</td><td>${fmt(c.components.energy,2)}</td><td>${fmt(c.components.predicted_risk,2)}</td><td>${c.index===G.chosen?'ELEGIDA':rej.has(c.index)?`<span class="r-veto">vetada: ${rej.get(c.index).reason}</span>`:'alternativa'}</td></tr>`).join('')+'</table>'
    :`<div class="small">${G.message||'Sin candidatas.'}</div>`;
  gaGen(); }

// ------------------------------------------------------------ G · log / comparison
let logMode='log';
document.querySelectorAll('[data-log]').forEach(b=>b.addEventListener('click',()=>{ logMode=b.dataset.log; document.querySelectorAll('[data-log]').forEach(x=>x.classList.toggle('on',x===b));
  $('logPane').style.display=logMode==='log'?'':'none'; $('cmpPane').style.display=logMode==='cmp'?'':'none'; $('logTitle').textContent=logMode==='log'?'Bitácora de decisiones':'Comparación de políticas'; renderCompare(); }));
$('fFilter').addEventListener('change',renderLog);
function renderLog(){ if(!S) return; const f=$('fFilter').value; let rows=S.log.slice().reverse();
  if(f==='veto') rows=rows.filter(r=>/R0[0-8]|R1[0-36]/.test(r.reason)); if(f==='ga') rows=rows.filter(r=>r.ga!==null&&r.ga!==undefined); if(f==='ev') rows=rows.filter(r=>r.events.length);
  const cls=r=>/R0[0-6]|R10/.test(r)?'r-veto':/R0[7-9]|R1[1-36]/.test(r)?'r-warn':'r-ok';
  $('trace').innerHTML=rows.length?'<table class="mt"><tr><th>paso</th><th>estado</th><th>acción</th><th>RNA</th><th>v</th><th>motivo</th></tr>'+rows.map(r=>`<tr><td>${String(r.step).padStart(3,'0')}</td><td>${r.state}</td><td>${r.action}</td><td>${r.rna?r.rna.slice(0,4)+' '+fmt(r.conf,2):'—'}</td><td>${fmt(r.speed,2)}</td><td class="${cls(r.reason)}">${r.reason}</td></tr><tr><td></td><td colspan="5" class="x">${r.events.length?'<b>⚡ '+r.events.join(', ')+'</b> · ':''}${r.ga!==null&&r.ga!==undefined?'AG fitness '+fmt(r.ga,3)+' · ':''}${r.explanation} · bat ${fmt(r.battery,1)} %</td></tr>`).join('')+'</table>'
    :'<div class="small">Sin decisiones todavía. Presione Iniciar o Paso.</div>'; }
$('bCompare').addEventListener('click',async()=>{ running=false; syncRun(); msg('Ejecutando comparación con las mismas semillas…'); $('bCompare').disabled=true;
  const o=await api('compare',{seeds:+$('fSeeds').value,scenario:$('fScenario').value}); $('bCompare').disabled=false; if(o.error){ msg(o.error,true); return; } S.comparison=o; renderCompare(); msg('Comparación lista.'); });
function renderCompare(){ const C=S&&S.comparison; if(!C||logMode!=='cmp') return;
  const cols=[['success','éxito',2],['collisions','colis.',2],['safety_violations','violac.',2],['battery_left','energía',0],['science','ciencia',2],['replans','replan.',1]];
  $('cmpTable').innerHTML=`<div style="margin-bottom:3px">${C.label} · n=${C.seeds} por política · media ± desv (mediana)</div><table class="mt"><tr><th>política</th>${cols.map(c=>`<th>${c[1]}</th>`).join('')}</tr>`+
    C.summary.map(r=>`<tr><td>${r.policy}</td>${cols.map(([k,,d])=>`<td>${fmt(r[k].media,d)}±${fmt(r[k].desv,d)}<br><span class="small">(${fmt(r[k].mediana,d)})</span></td>`).join('')}</tr>`).join('')+'</table>';
  const [g,w,h]=ctx2d($('cmpChart')); const n=C.summary.length, bw=(w-10)/n;
  C.summary.forEach((r,i)=>{ const x=5+i*bw, s=r.success.media, cc=Math.min(1,r.collisions.media/3); g.fillStyle=css('--ok'); g.fillRect(x+bw*.12,h-14-(h-28)*s,bw*.36,(h-28)*s); g.fillStyle=css('--bad'); g.fillRect(x+bw*.52,h-14-(h-28)*cc,bw*.36,(h-28)*cc);
    g.fillStyle=css('--muted'); g.font='9.5px Barlow'; g.textAlign='center'; g.fillText(r.policy,x+bw/2,h-2); g.fillText(fmt(s*100,0)+'%',x+bw*.3,h-17-(h-28)*s); }); g.textAlign='left';
  g.fillStyle=css('--ok'); g.fillText('éxito',4,10); g.fillStyle=css('--bad'); g.fillText('colisiones (0–3)',40,10); }

// ------------------------------------------------------------ controls
function formConfig(){ return {planet:(S&&S.planet.name)||'Marte', mode:$('fMode').value, seed:Number($('fSeed').value), difficulty:$('fDiff').value, battery:Number($('fBattery').value), policy:$('fPolicy').value, step_duration:Number($('fDur').value), event_rate:Number($('fEvents').value)/100, drone:!!(S&&S.config.drone)}; }
function syncRun(){ const b=$('bRun'); b.textContent=running?'⏸ Pausa':'▶ Iniciar'; }
const BASE_STEP_MS=700, SPEEDS=[.5,.75,1,1.5,2,4];
const speedMult=()=>SPEEDS[+$('fSpeed').value]||1;
let runId=0; // each start gets its own id, so a stale timer from a previous run can never step
async function loop(id){ if(!running||id!==runId) return; const t0=performance.now(); await call('step',{n:1});
  if(id!==runId) return;
  // Cadence is measured step to step, so engine latency does not slow the simulation further.
  const wait=Math.max(0,BASE_STEP_MS/speedMult()-(performance.now()-t0));
  if(running&&S&&!raceDone()) setTimeout(()=>loop(id),wait); else { running=false; syncRun(); } }
$('bRun').addEventListener('click',()=>{ if(replayAt!==null) seek(frames.length-1); if(S&&raceDone()){ msg('La misión terminó. Reinicie o cambie la configuración.'); return; } running=!running; syncRun(); if(running){ msg('Simulación en curso.'); loop(++runId); } else msg('Pausada.'); });
$('bStep').addEventListener('click',()=>{ running=false; syncRun(); call('step',{n:1}); });
$('bGhost').addEventListener('click',async()=>{ running=false; syncRun(); const on=!(S.ghosts||[]).length; if(await call('ghosts',{enabled:on})){ gaRunSel=null; msg(on?'Carrera de fantasmas activada: las otras políticas corren el mismo mapa y la misma semilla. Misión reiniciada.':'Carrera de fantasmas desactivada. Misión reiniciada.'); } });
$('bDrone').addEventListener('click',async()=>{ if(replayAt!==null) seek(frames.length-1); const on=!S.config.drone; // toggled in place: the mission keeps its step and route
  if(await call('drone',{enabled:on})) msg(on?`Dron explorador activado en el paso ${S.status.step}: desde el próximo paso revisa las 4 celdas siguientes de la ruta y comparte los peligros que ve.`:`Dron explorador apagado en el paso ${S.status.step}: el rover sigue solo con sus sensores.`); });
$('bReset').addEventListener('click',async()=>{ running=false; syncRun(); if(await call('reset',{})){ gaRunSel=null; msg('Misión reiniciada con la misma semilla.'); } });
$('bApply').addEventListener('click',async()=>{ running=false; syncRun(); if(await call('configure',formConfig())){ gaRunSel=null; msg('Configuración aplicada. Nuevo terreno generado.'); } });
$('fBattery').addEventListener('input',()=>$('lBat').textContent=$('fBattery').value+' %');
$('fSpeed').addEventListener('input',()=>$('lSpeed').textContent='×'+String(speedMult()).replace('.',','));
$('fDur').addEventListener('input',()=>$('lDur').textContent=(+$('fDur').value).toFixed(1)+' s (aplicar)');
$('fEvents').addEventListener('input',()=>$('lEv').textContent=$('fEvents').value+' % (aplicar)');
// ------------------------------------------------------------ interplanetary travel (F1)
const TRAVEL_MS=3000; let travelSkip=null;
function planetDisc(g,name,x,y,r){ const p=PLANETS[name]||PLANETS.Marte; if(r<=.5) return;
  const gr=g.createRadialGradient(x-r*.3,y-r*.35,r*.1,x,y,r); gr.addColorStop(0,p.icon[0]); gr.addColorStop(1,p.icon[1]);
  g.save(); g.beginPath(); g.arc(x,y,r,0,7); g.fillStyle=gr; g.fill(); g.clip();
  if(p.gas){ g.globalAlpha=.22; g.fillStyle='#fff'; for(let i=-2;i<=2;i++) g.fillRect(x-r,y+i*r*.3-r*.05,r*2,r*.1); g.globalAlpha=1; }
  const sh=g.createLinearGradient(x-r*.4,y-r*.4,x+r,y+r*.8); sh.addColorStop(0,'rgba(0,0,0,0)'); sh.addColorStop(1,'rgba(0,0,0,.6)'); g.fillStyle=sh; g.fillRect(x-r,y-r,r*2,r*2); g.restore();
  if(name==='Saturno'){ g.strokeStyle='#e8d7a8'; g.lineWidth=Math.max(1,r*.08); g.beginPath(); g.ellipse(x,y,r*1.6,r*.34,-.3,0,7); g.stroke(); } }
function travelTo(from,to){ return new Promise(done=>{ const c=$('travel'); c.style.display='block'; c.style.opacity=1; const [g,w,h]=ctx2d(c);
  const reduced=window.matchMedia&&matchMedia('(prefers-reduced-motion: reduce)').matches, total=reduced?700:TRAVEL_MS;
  const stars=Array.from({length:140},(_,i)=>({x:(i*97%211)/211,y:(i*53%199)/199,z:.3+(i*31%100)/140})); const t0=performance.now(); let ended=false;
  const ease=t=>t<.5?2*t*t:1-Math.pow(-2*t+2,2)/2, clamp=(v)=>Math.max(0,Math.min(1,v));
  const end=()=>{ if(ended) return; ended=true; travelSkip=null; c.removeEventListener('click',end); c.style.display='none'; done(); };
  travelSkip=end; c.addEventListener('click',end);
  (function frame(now){ if(ended) return; const t=clamp((now-t0)/total), u=Math.min(w,h); g.clearRect(0,0,w,h); g.fillStyle='#03040a'; g.fillRect(0,0,w,h);
    const warp=reduced?0:Math.sin(Math.PI*clamp((t-.2)/.6));                        // star streaks peak mid-flight
    stars.forEach(s=>{ const x=((s.x-t*s.z*1.4)%1+1)%1*w, y=s.y*h, len=2+warp*90*s.z; g.strokeStyle=`rgba(210,225,255,${.35+.5*s.z})`; g.lineWidth=s.z*1.6; g.beginPath(); g.moveTo(x,y); g.lineTo(x+len,y); g.stroke(); });
    const a=ease(clamp(t/.45)), b=ease(clamp((t-.55)/.45));                          // origin recedes, destination approaches
    planetDisc(g,from,w*(.3-.42*a),h*(.55+.1*a),u*.34*(1-a*.93));
    planetDisc(g,to,w*(1.12-.42*b),h*(.45+.1*(1-b)),u*.34*(.07+b*.93));
    if(!reduced){ const k=ease(clamp((t-.12)/.76)), x=w*(.12+.76*k), y=h*(.5-.12*Math.sin(Math.PI*k)), sz=u*.035;      // ship
      g.save(); g.translate(x,y); g.rotate(-.35*Math.cos(Math.PI*k));
      const fl=g.createLinearGradient(-sz*4,0,-sz,0); fl.addColorStop(0,'rgba(255,170,60,0)'); fl.addColorStop(1,'rgba(255,220,140,.95)'); g.fillStyle=fl; g.beginPath(); g.moveTo(-sz,-sz*.35); g.lineTo(-sz*(3+Math.sin(now/40)),0); g.lineTo(-sz,sz*.35); g.fill();
      g.fillStyle='#e9edf5'; g.beginPath(); g.moveTo(sz*1.6,0); g.lineTo(-sz,-sz*.55); g.lineTo(-sz,sz*.55); g.closePath(); g.fill();
      g.fillStyle='#9aa3b5'; g.beginPath(); g.moveTo(-sz,-sz*.55); g.lineTo(-sz*1.5,-sz); g.lineTo(-sz*.3,-sz*.3); g.closePath(); g.fill(); g.beginPath(); g.moveTo(-sz,sz*.55); g.lineTo(-sz*1.5,sz); g.lineTo(-sz*.3,sz*.3); g.closePath(); g.fill();
      g.fillStyle='#d52b1e'; g.fillRect(-sz*.5,-sz*.22,sz*.9,sz*.44); g.fillStyle='#fff'; g.fillRect(-sz*.5,-sz*.22,sz*.9,sz*.22); g.fillStyle='#0039a6'; g.fillRect(-sz*.5,-sz*.22,sz*.3,sz*.22); g.restore(); }
    g.textAlign='center'; g.fillStyle='#e8ecf5'; g.font=`600 ${Math.max(13,u*.05)}px system-ui,sans-serif`; g.fillText(`${from.toUpperCase()}  →  ${to.toUpperCase()}`,w/2,h*.88);
    g.fillStyle='rgba(200,210,230,.7)'; g.font=`${Math.max(10,u*.028)}px system-ui,sans-serif`; g.fillText('Viaje interplanetario simulado · clic para saltar',w/2,h*.94);
    c.style.opacity=t>.9?1-(t-.9)/.1:1;
    if(t>=1) end(); else requestAnimationFrame(frame); })(t0); }); }
async function choosePlanet(name){ running=false; syncRun(); const P=PLANETS[name]; const cfg={...formConfig(),planet:name};
  const v=await api('validate',cfg); if(v.error){ msg(v.error+(P&&P.gas?' Presione otra vez para usar el modo sonda.':''),true);
    if(P&&P.gas){ $('fMode').value='sonda'; } return; }
  const from=S&&S.planet.name, trip=from&&from!==name?travelTo(from,name):null;      // only for a valid change of planet
  const ok=await call('configure',cfg); if(!ok&&travelSkip) travelSkip(); if(trip) await trip;
  if(ok){ gaRunSel=null; msg(`${name}: ${P&&P.gas?'sonda atmosférica virtual (sin suelo sólido para ruedas)':'superficie simulada'} · nuevo terreno.`); } }
document.querySelectorAll('[data-export]').forEach(b=>b.addEventListener('click',async()=>{ const o=await api('export',{format:b.dataset.export}); if(o.error){ msg(o.error,true); return; }
  const blob=new Blob([o.content],{type:'text/plain;charset=utf-8'}); const a=document.createElement('a'); a.href=URL.createObjectURL(blob); a.download=o.filename; document.body.appendChild(a); a.click(); a.remove(); msg('Exportado '+o.filename); }));
window.addEventListener('resize',()=>{ clearTimeout(window._rt); window._rt=setTimeout(()=>{ draw2d(); renderNN(); renderFuzzy(); renderGA(); renderCompare(); },150); });

async function init(){
  META=await api('meta'); if(META.error){ msg(META.error,true); return; }
  $('planets').innerHTML=META.planets.map(p=>`<button class="btn pbtn" data-planet="${p.name}" title="${p.rocky?'Superficie rocosa':'Gigante: solo sonda'}">${p.name}</button>`).join('');
  document.querySelectorAll('[data-planet]').forEach(b=>b.addEventListener('click',()=>choosePlanet(b.dataset.planet)));
  $('fPolicy').innerHTML=Object.entries(META.policies).map(([k,v])=>`<option value="${k}">${v}</option>`).join('');
  $('fScenario').innerHTML=Object.entries(META.scenarios).map(([k,v])=>`<option value="${k}">${v}</option>`).join('');
  $('events').innerHTML=Object.entries(META.events).map(([k,v])=>`<button class="btn ev" data-ev="${k}" title="${v}">${v}</button>`).join('')+'<button class="btn ev" data-ev="falla_sensor" data-sensor="todos" title="Fallan proximidad y LIDAR">Falla total navegación</button>';
  document.querySelectorAll('[data-ev]').forEach(b=>b.addEventListener('click',async()=>{ const args={kind:b.dataset.ev}; if(b.dataset.sensor) args.sensor=b.dataset.sensor;
    const o=await call('inject',args); if(o){ const e=o.map.events.slice(-1)[0]; if(b.dataset.ev==='meteorito'&&e&&e.position) meteorFx(e.position); msg('Evento: '+b.title+'. El supervisor lo evaluará en el próximo paso.'); } }));
  buildFeat(); initScene();
  try{ if(localStorage.getItem('ares_view')==='2d') setView('2d'); }catch(e){}
  await call('state',{});
  $('fMode').value=S.config.mode==='rover'?'rover':'sonda'; $('fSeed').value=S.config.seed; $('fDiff').value=S.config.difficulty; $('fBattery').value=S.config.battery; $('lBat').textContent=S.config.battery+' %'; $('fPolicy').value=S.config.policy; $('fDur').value=S.config.step_duration; $('lDur').textContent=(+S.config.step_duration).toFixed(1)+' s';
}
if (window.ARES_BOOT) window.ARES_BOOT.then(init).catch(e=>msg('No se pudo iniciar el motor Python: '+e,true)); else init();
</script>
</body>
</html>


In [ ]:
import sys, importlib
sys.path.insert(0, 'ares_x/src')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from ares.api import Session
from ares.mission import HybridMission, shared_perception, POLICIES
from ares.fuzzy import FuzzyController
from ares.genetic import GeneticPlanner
from ares.experiments import run_experiments, summarize
from ares.dashboard import Dashboard
print('Motor ARES-X cargado.')

## 2. RNA — percepción (RF-04, CA-06)
Datos sintéticos etiquetados por el simulador (la verdad es la pertenencia de la celda a barranco/obstáculo, no la variable que recibe la red).
Partición **por mapa** en entrenamiento/validación/test para evitar fuga. MLP (32,16) ReLU con softmax de 3 clases.

In [ ]:
perception = shared_perception(0)
report = perception.report_
print(f"Entrenamiento: {report['training_samples']} muestras · {report['iterations']} iteraciones · {perception.train_seconds_:.2f} s")
print('Mapas por partición:', report['split_maps'])
for split in ('validation', 'test'):
    m = report[split]
    print(f"{split:>10}: exactitud {m['accuracy']:.3f} | baseline mayoría {m['majority_baseline']:.3f} | macro-F1 {m['macro_f1']:.3f} | recall barranco {m['barranco_recall']:.3f} | ECE {m['calibration_error']:.3f}")
fig, ax = plt.subplots(figsize=(4, 3.4))
cm = np.array(report['test']['confusion_matrix'])
ax.imshow(cm, cmap='Blues'); ax.set_xticks(range(3), report['test']['labels']); ax.set_yticks(range(3), report['test']['labels'])
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha='center', va='center')
ax.set_xlabel('predicho'); ax.set_ylabel('real'); ax.set_title('Matriz de confusión (test)'); plt.show()
sample = dict(distance=.05, slope=30, elevation_delta=-10, texture=.3, visibility=.9, vibration=1.5)
pred = perception.predict(sample)
print('Lectura con caída de 10 m →', pred['label'], {k: round(v, 3) for k, v in pred['probabilities'].items()})

## 3. Lógica difusa — control (RF-05, RF-06, CA-05)
Distancia (cerca/media/lejos), batería (crítica/baja/alta), pendiente (plana/media/fuerte) y temperatura; 12 reglas Mamdani min–max,
desfusificación por centroide. Sin reglas activas → salida documentada 0 (parar).

In [ ]:
fz = FuzzyController()
for rule in fz.rules:
    print(f"R{rule['id']:>2}: {rule['description']}")
for distance in (10, 1):
    out = fz.infer(distance, 60, 5, 12)
    active = ', '.join(f"R{r['id']}={r['strength']:.2f}" for r in out['activated_rules'])
    print(f"distancia {distance:>2} m → velocidad {out['speed']:.3f} · reglas activas: {active}")
fig, axes = plt.subplots(1, 2, figsize=(10, 2.8))
xs = np.linspace(0, 20, 201)
for label, spec in fz.membership_sets['distance'].items():
    axes[0].plot(xs, [fz.membership(x, spec) for x in xs], label=label)
axes[0].axvline(10, ls='--', c='k'); axes[0].axvline(1, ls=':', c='r'); axes[0].set_title('Distancia a riesgo (m)'); axes[0].legend()
for distance, style in ((10, '-'), (1, ':')):
    out = fz.infer(distance, 60, 5, 12)
    axes[1].plot(out['universe'], out['aggregated_curve'], style, label=f"{distance} m → centroide {out['centroid']:.2f}")
axes[1].set_title('Agregación y centroide (velocidad)'); axes[1].legend(); plt.show()

## 4. Algoritmo genético — planificación (RF-07, CA-04)
Cromosoma: K=3 puntos de paso decodificados a una ruta contigua con reparación. Población 30, ≥30 generaciones, torneo, cruce, mutación,
elitismo 2, semilla controlada y parada temprana tras 10 generaciones sin mejora. Familias integradas: **laberinto** (factibilidad de ruta)
y **viajero** (orden de visita de sitios científicos).

In [ ]:
mission = HybridMission(seed=42)
mission.step()
ga = mission.ga
print(f"Generaciones: {ga['generations_run']} · parada temprana: {ga['stopped_early']} · pesos: {ga['weights']}")
for cand in ga['candidates']:
    print(f"#{cand['index']} fitness {cand['fitness']:+.3f} factible={cand['feasible']} componentes={cand['components']} orden={cand['science_order']}")
gen5 = mission.ga_generation(5, run=0)
print(f"Generación 5: torneos {gen5['tournaments']}, cruces {gen5['crossovers']}, genes mutados {gen5['mutations']}")
for row in gen5['rows'][:3]:
    print('  ', row['chromosome'], 'fitness', row['fitness'])
plt.figure(figsize=(6, 2.6))
plt.plot([h['best'] for h in ga['history']], label='mejor'); plt.plot([h['best_feasible'] for h in ga['history']], '--', label='mejor factible')
plt.xlabel('generación'); plt.ylabel('fitness'); plt.legend(); plt.title('Convergencia del AG'); plt.show()

## 5. Centro de control de misión (una sola pantalla)
Panel A mapa y rutas · B telemetría · C decisión explicable por algoritmo · D traza filtrable/exportable · E controles y eventos.
Pestañas: Laboratorio RNA, Difuso, Genético y Comparación de políticas. Iniciar, pausar, paso a paso, reiniciar, velocidad,
inyectar eventos y editar el terreno real (clic en el mapa).

In [ ]:
dashboard = Dashboard(seed=42, planet='Marte', height=880)
dashboard.show()

## 6. Experimentos: 20 semillas, ablaciones y baselines (RF-14, RF-15)
Mismas semillas y condiciones para cada política: control (ruta más corta sin riesgos), AG puro con riesgos fijos, híbrido completo y
ablaciones sin RNA / sin difuso / sin AG. Se reporta media, mediana y dispersión.

In [ ]:
import time, json
results = {}
for scenario in ('estandar', 'eventos'):
    t0 = time.time()
    frame = run_experiments(seeds=20, scenario=scenario)
    results[scenario] = frame
    print(f"Escenario {scenario}: {len(frame)} misiones en {time.time() - t0:.1f} s")
    display(summarize(frame).round(3))
frame = pd.concat(results.values())
frame.to_csv('ares_x_experimentos.csv', index=False)
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, metric in zip(axes, ('success', 'collisions', 'battery_left')):
    data = results['eventos'].groupby('policy')[metric].apply(list)
    ax.boxplot([data[p] for p in POLICIES], tick_labels=list(POLICIES)) if metric != 'success' else ax.bar(list(POLICIES), [np.mean(data[p]) for p in POLICIES])
    ax.set_title(metric + ' (eventos)'); ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# Exportación de una misión completa (CSV de pasos + JSON con configuración, motivos y trazas) (RF-12)
demo = HybridMission(seed=42)
demo.step(); demo.inject('barranco'); demo.run()
open('ares_x_mision_semilla42.csv', 'w', encoding='utf-8').write(demo.export_csv())
json.dump(demo.export_json(), open('ares_x_mision_semilla42.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print(demo.summary())
try:
    from google.colab import files  # descarga opcional en Colab
    print('Use files.download("ares_x_mision_semilla42.json") para descargar.')
except ImportError:
    pass

## 7. Pruebas automáticas
Casos de borde (sección 12) y casos de aceptación CA-01…CA-08.

In [ ]:
%%writefile ares_x/tests/test_f3_random_events.py
"""v1.1 F3: random events drawn per step from a seed-derived generator."""
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

from ares.api import Session  # noqa: E402
from ares.mission import HybridMission, validate_config  # noqa: E402


def run(steps=200, **config):
    config.setdefault('seed', 42)
    mission = HybridMission(**config).prepare()
    for _ in range(steps):
        if mission.finished:
            break
        mission.step()
    return mission


def random_events(mission):
    return [e for e in mission.event_log if e.get('origin') == 'aleatorio']


def test_event_rate_defaults_to_zero_and_is_validated():
    assert validate_config({})['event_rate'] == 0.
    assert validate_config({'event_rate': .05})['event_rate'] == .05
    for bad in (-.01, .11, 'x', float('nan')):
        with pytest.raises(ValueError, match='event_rate'):
            validate_config({'event_rate': bad})


def test_rate_zero_keeps_the_trace_identical_to_the_default():
    base, zero = run(60), run(60, event_rate=0)
    assert base.export_csv() == zero.export_csv()
    assert not random_events(zero)


def test_same_seed_and_rate_reproduce_the_same_events_and_trace():
    a, b = run(200, event_rate=.05), run(200, event_rate=.05)
    assert [(e['step'], e['kind']) for e in random_events(a)] == [(e['step'], e['kind']) for e in random_events(b)]
    assert a.export_csv() == b.export_csv()


def test_two_hundred_steps_at_five_percent_give_between_2_and_25_events():
    counts = []
    for seed in (1, 7, 42):
        mission = HybridMission(seed=seed, event_rate=.05).prepare()
        drawn = 0
        for _ in range(200):
            before = len(random_events(mission))
            mission.step()
            drawn += len(random_events(mission)) - before
            if mission.finished:  # keep drawing on a fresh copy so the window is 200 steps
                mission = HybridMission(seed=seed + 1000 + mission.step_index, event_rate=.05).prepare()
        counts.append(drawn)
    assert all(2 <= c <= 25 for c in counts), counts


def test_random_events_are_valid_for_the_mode_and_marked_as_random():
    probe = run(200, planet='Saturno', mode='sonda', event_rate=.1)
    kinds = {e['kind'] for e in random_events(probe)}
    assert kinds and 'barranco' not in kinds
    assert all(e['label'] and e['message'] for e in random_events(probe))
    rover = run(80, event_rate=.1)
    operator = rover.inject('sismo') if not rover.finished else None
    assert operator is None or operator['origin'] == 'operador'


def test_no_events_after_the_mission_has_finished():
    mission = run(400, event_rate=.1)
    assert mission.finished
    count = len(mission.event_log)
    for _ in range(20):
        mission.step()
    assert len(mission.event_log) == count


def test_random_events_do_not_count_as_queued_operator_orders():
    mission = HybridMission(seed=42, event_rate=.1).prepare()
    mission.step()
    mission.inject('perdida_enlace')
    queued = mission.queued_orders
    for _ in range(4):
        mission.step()
    assert mission.queued_orders == queued


def test_session_accepts_event_rate_in_configure():
    session = Session()
    snap = session.dispatch('configure', {'seed': 3, 'event_rate': .1})
    assert 'error' not in snap, snap
    assert snap['config']['event_rate'] == .1


def test_panel_entry_points_start_at_three_percent():
    from ares.dashboard import Dashboard
    assert Dashboard().session.config['event_rate'] == .03
    root = Path(__file__).resolve().parents[1]
    if not (root / 'server.py').exists():
        pytest.skip('server.py and build_site.py are not bundled in the notebook')
    assert 'Session(event_rate=.03)' in (root / 'server.py').read_text()
    assert 'Session(event_rate=0.03)' in (root / 'build_site.py').read_text()


def test_feed_and_log_tell_random_events_apart():
    session = Session(seed=42, event_rate=.1)
    snap = None
    for _ in range(60):
        snap = session.dispatch('step', {'n': 1})
        if any(e['origin'] == 'aleatorio' for e in snap['event_feed']['recent']):
            break
    assert any(e['origin'] == 'aleatorio' for e in snap['event_feed']['recent'])
    assert any('(aleatorio)' in label for row in snap['log'] for label in row['events'])
    snap = session.dispatch('inject', {'kind': 'sismo'})
    assert snap['event_feed']['recent'][-1]['origin'] == 'operador'


In [ ]:
%%writefile ares_x/tests/test_h1_physics.py
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from ares import PlanetProfile, World, SensorSuite, Mission


def test_empty_battery():
    mission = Mission(World(seed=3), battery=0)
    start = mission.position
    for _ in range(3):
        mission.step((0, 1))
    assert mission.position == start
    assert mission.battery == 0
    assert mission.state.value == 'FRACASO'


@pytest.mark.parametrize('distance', [float('nan'), float('inf'), -float('inf'), 0, None])
def test_invalid_distance(distance):
    assert not SensorSuite.validate({'distance': distance})


def test_new_cliff_stops():
    mission = Mission(World(seed=2))
    mission.inject_event('barranco', position=(0, 1))
    mission.step((0, 1))
    assert mission.position == (0, 0)
    assert mission.state.value == 'REPLANIFICACION'
    assert (0, 1) in mission.world.cliffs


def test_seed_trace():
    def run():
        mission = Mission(World(seed=42))
        for target in [(0, 1), (1, 0), (0, 0)]:
            mission.step(target)
        return mission.logger.trace
    assert run() == run()


def test_jupiter_wheels():
    with pytest.raises(ValueError, match='sonda'):
        PlanetProfile.for_body('Júpiter', mode='ruedas')


def test_no_science():
    mission = Mission(World(seed=4, science_count=0))
    assert mission.science_progress == 1.0
    mission.step()
    assert mission.state.value == 'EXITO'


@pytest.mark.parametrize('body', ['Mercurio', 'Venus', 'Tierra', 'Marte', 'Júpiter', 'Saturno', 'Urano', 'Neptuno'])
def test_planet_modes(body):
    assert PlanetProfile.for_body(body, mode='sonda').mode == 'probe'
    if body in ['Júpiter', 'Saturno', 'Urano', 'Neptuno']:
        with pytest.raises(ValueError, match='suelo sólido'):
            PlanetProfile.for_body(body)
    else:
        assert PlanetProfile.for_body(body).rocky


def test_map_generation_and_manual_obstacle():
    import numpy as np
    a, b = World(seed=19), World(seed=19)
    assert (a.height, a.width) == (12, 16)
    assert np.array_equal(a.elevation, b.elevation)
    assert a.science_sites == b.science_sites
    assert not (a.obstacles & a.science_sites)
    a.add_obstacle((1, 1))
    assert not a.traversable((1, 1))
    with pytest.raises(ValueError):
        a.add_obstacle((-1, 0))


@pytest.mark.parametrize('failure', ['nan', 'saturation', 'missing'])
def test_sensor_failure_stops_and_logs(failure):
    mission = Mission(World(seed=3))
    mission.world.obstacles.discard((0, 1))
    mission.inject_event('falla_sensor', failure=failure)
    row = mission.step((0, 1))
    assert mission.position == (0, 0)
    assert row['state'] == 'MODO_SEGURO'
    assert row['sensors']['distance'] is None


def test_link_loss_local_operation_and_csv():
    import csv
    import io
    mission = Mission(World(seed=3))
    mission.world.obstacles.discard((0, 1))
    mission.inject_event('perdida_enlace')
    row = mission.step((0, 1))
    assert mission.position == (0, 1)
    assert not row['sensors']['link_available']
    rows = list(csv.DictReader(io.StringIO(mission.logger.export_csv())))
    assert len(rows) == 1
    assert rows[0]['action'] == 'avanzar'
    row['battery'] = -100
    assert mission.logger.trace[0]['battery'] >= 0


def test_events_and_narrative():
    from ares import EVENTS
    mission = Mission(World(seed=5), profile=PlanetProfile.for_body('Júpiter', 'probe'))
    for kind in EVENTS:
        mission.inject_event(kind, position=(1, 1))
    assert len(mission.pending_events) == len(EVENTS)
    assert mission.world.cliffs == {(1, 1)}
    assert mission.health == 80
    assert not mission.daylight
    assert mission.visibility == .2
    assert 'no confirma vida' in next(e['message'] for e in mission.pending_events if e['kind'] == 'biosenal_anomala')
    assert 'ficticio' in next(e['message'] for e in mission.pending_events if e['kind'] == 'alienigena')


def test_invalid_configuration():
    for kwargs in [{'battery': float('nan')}, {'battery': -1}, {'step_duration': 0}]:
        with pytest.raises(ValueError):
            Mission(World(), **kwargs)
    for kwargs in [{'width': 0}, {'difficulty': 'impossible'}, {'science_count': -1}]:
        with pytest.raises(ValueError):
            World(**kwargs)


def test_cliff_altimetry_and_insufficient_energy():
    mission = Mission(World(seed=1), battery=.5)
    mission.world.obstacles.discard((0, 1))
    mission.world.elevation[0, 1] = mission.world.elevation[0, 0] - 10
    assert mission.step((0, 1))['reason'] == 'BARRANCO_CONFIRMADO'
    mission.world.cliffs.clear()
    mission.world.elevation[0, 1] = 0
    assert mission.step((0, 1))['reason'] == 'ENERGIA_INSUFICIENTE'
    assert mission.battery == .5


def test_navigation_invalid_altimetry():
    mission = Mission(World(seed=1))
    mission.world.obstacles.discard((0, 1))
    mission.inject_event('falla_sensor', sensor='elevation_delta')
    assert mission.step((0, 1))['state'] == 'MODO_SEGURO'
    assert mission.position == (0, 0)


def test_thermal_limit():
    mission = Mission(World(seed=1), profile=PlanetProfile.for_body('Venus'), thermal_limit=100)
    assert mission.step()['reason'] == 'LIMITE_TERMICO'


@pytest.mark.parametrize('body', ['Júpiter', 'Saturno', 'Urano', 'Neptuno'])
def test_all_giants_require_probe(body):
    with pytest.raises(ValueError, match='suelo sólido.*sonda'):
        PlanetProfile.for_body(body, 'ruedas')
    assert PlanetProfile.for_body(body, 'sonda').mode == 'probe'


def test_invalid_altimeter_stops_before_movement():
    world = World(seed=1)
    world.obstacles.clear()
    sensors = SensorSuite(noise=0)
    sensors.fail('elevation_delta', 'nan')
    mission = Mission(world, sensors=sensors)
    row = mission.step((0, 1))
    assert row['action'] == 'detener'
    assert mission.position == (0, 0)
    assert row['state'] == 'MODO_SEGURO'


def test_empty_mission_does_not_override_cliff_stop():
    mission = Mission(World(science_count=0))
    mission.inject_event('barranco', position=(0, 1))
    assert mission.step((0, 1))['state'] == 'REPLANIFICACION'


def test_low_energy_allows_reachable_return():
    world = World(seed=0)
    world.obstacles.clear()
    mission = Mission(world, battery=5)
    mission.position = (0, 1)
    row = mission.step((0, 0))
    assert row['action'] == 'avanzar'
    assert mission.position == (0, 0)
    assert mission.battery == 4


def test_dust_reduces_solar_power():
    mission = Mission(World())
    before = mission.step()['sensors']['solar_power']
    mission.inject_event('tormenta_polvo')
    assert mission.step()['sensors']['solar_power'] < before


def test_invalid_target_is_safe():
    mission = Mission(World())
    row = mission.step((float('nan'), 1))
    assert row['action'] == 'detener'
    assert row['state'] == 'REPLANIFICACION'


def test_csv_and_snapshots():
    import csv
    import io
    from ares import MissionLogger
    logger = MissionLogger()
    readings = {'distance': float('nan')}
    row = logger.record(sensors=readings, reason='ESPERA')
    readings['distance'] = 10
    row['reason'] = 'changed'
    assert logger.trace[0]['sensors']['distance'] is None
    assert list(csv.DictReader(io.StringIO(logger.export_csv())))[0]['reason'] == 'ESPERA'


def test_energy_threshold_preserves_reachable_return():
    world = World(width=30, height=1)
    world.obstacles.clear()
    world.elevation[:] = 0
    mission = Mission(world, battery=20)
    for column in range(1, 11):
        mission.step((0, column))
    assert mission.battery == 10
    row = mission.step((0, 11))
    assert row['action'] == 'detener'
    assert mission.position == (0, 10)
    assert mission.battery == 10
    assert mission.step((0, 9))['action'] == 'avanzar'


In [ ]:
%%writefile ares_x/tests/test_h2_fuzzy.py
"""Standalone Mamdani inference and diagnostic contract."""
import sys
from pathlib import Path

import numpy as np
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from ares.fuzzy import FuzzyController, triangular, trapezoidal


def test_sets_and_documented_rules():
    controller = FuzzyController()
    assert len(controller.rules) >= 8
    assert all(len(sets) >= 3 for sets in controller.membership_sets.values())
    assert all(rule['description'] for rule in controller.rules)


def test_distance_changes_crisp_and_centroid():
    controller = FuzzyController()
    near = controller.infer(distance=1, battery=80, slope=0, temperature=20)
    medium = controller.infer(distance=10, battery=80, slope=0, temperature=20)
    assert near['speed'] < medium['speed']
    assert near['caution'] > medium['caution']
    assert len(medium['activated_rules']) > 1
    x = np.asarray(medium['universe'])
    curve = np.asarray(medium['aggregated_curve'])
    assert medium['centroid'] == pytest.approx(np.trapezoid(x * curve, x) / np.trapezoid(curve, x))
    assert medium['speed'] == medium['centroid']


def test_no_rule_fallback():
    result = FuzzyController(rules=[]).infer(10, 80, 0, 20)
    assert result['speed'] == 0
    assert result['reason'] == 'SIN_REGLAS_ACTIVAS'
    assert not any(result['aggregated_curve'])


@pytest.mark.parametrize('values', [(float('nan'),80,0,20), (float('inf'),80,0,20),
                                   (-1,80,0,20), (10,101,0,20), (10,80,46,20)])
def test_invalid_inputs_safe(values):
    result = FuzzyController().infer(*values)
    assert result['speed'] == 0
    assert result['reason'] == 'ENTRADA_INVALIDA'


def test_membership_shoulders_and_triangle():
    assert trapezoidal(0, 0, 0, 2, 4) == 1
    assert trapezoidal(4, 0, 2, 4, 4) == 1
    assert triangular(2, 0, 2, 4) == 1
    assert triangular(1, 0, 2, 4) == .5


def test_temperature_and_boundaries():
    controller = FuzzyController()
    normal = controller.infer(20, 100, 0, 20)
    hot = controller.infer(20, 100, 0, 450)
    assert hot['speed'] < normal['speed']
    for distance in (0, 20):
        result = controller.infer(distance, 0, 45, -200)
        assert 0 <= result['speed'] <= 1
        assert 0 <= result['caution'] <= 1


In [ ]:
%%writefile ares_x/tests/test_h2_genetic.py
"""Public genetic-planner behavior and observable evolutionary operations."""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

import numpy as np
import pytest

from ares.genetic import GeneticPlanner
from ares.world import World


def flat(width=6, height=5):
    world = World(width=width, height=height, science_count=0)
    world.obstacles.clear()
    world.cliffs.clear()
    world.science_sites = {(0, 3), (3, 3)}
    return world


def test_population_candidates_history_and_determinism():
    world = flat()
    first = GeneticPlanner(seed=7).plan(world, (0, 0), (4, 5))
    second = GeneticPlanner(seed=7).plan(world, (0, 0), (4, 5))
    assert first == second
    assert first['feasible'] and len(first['candidates']) >= 3
    assert len({tuple(c['route']) for c in first['candidates']}) >= 3
    assert len(first['history'][0]['chromosomes']) >= 30
    for candidate in first['candidates']:
        assert set(candidate['components']) == {'science', 'distance', 'energy', 'predicted_risk', 'time'}
        assert all(0 <= value <= 1 for value in candidate['components'].values())
        assert all(abs(a[0]-b[0]) + abs(a[1]-b[1]) == 1 for a, b in zip(candidate['route'], candidate['route'][1:]))
    assert any(h['crossovers'] > 0 and h['mutations'] > 0 for h in first['history'][1:])
    assert any(h['tournaments'] > 0 for h in first['history'][1:])


def test_elitism_preserves_best_chromosome():
    result = GeneticPlanner(seed=4, early_stop=False).plan(flat(), (0, 0), (4, 5))
    history = result['history']
    assert len(history) == 31
    for previous, current in zip(history, history[1:]):
        assert previous['best_chromosome'] in current['chromosomes']
        assert current['best_fitness'] >= previous['best_fitness']


def test_early_stop_retains_best_feasible():
    result = GeneticPlanner(seed=4, mutation_rate=0, crossover_rate=0).plan(flat(), (0, 0), (4, 5))
    assert result['stopped_early'] and result['generations_run'] == 10
    assert result['feasible']
    assert result['fitness'] == max(h['best_feasible_fitness'] for h in result['history'])


def test_invalid_mutation_is_repaired_and_penalty_is_inspectable():
    world = flat()
    planner = GeneticPlanner(seed=1, waypoints=3)
    invalid = [(900, -40), (float('nan'), 0), None]
    repaired = planner.repair(invalid, world)
    assert len(repaired) == 3 and all(world.traversable(p) for p in repaired)
    mutated = planner.mutate(invalid, world)
    assert all(world.traversable(p) for p in mutated)
    candidate = planner.evaluate(invalid, world, (0, 0), (4, 5))
    assert candidate['repaired'] and candidate['feasible']


@pytest.mark.parametrize('kind', ['obstacle', 'cliff'])
def test_maze_avoids_hard_barriers(kind):
    world = flat()
    barrier = {(r, 2) for r in range(4)}
    getattr(world, 'obstacles' if kind == 'obstacle' else 'cliffs').update(barrier)
    result = GeneticPlanner(seed=2).plan(world, (0, 0), (0, 5))
    assert result['feasible'] and not (set(result['route']) & barrier)
    assert (4, 2) in result['route']


@pytest.mark.parametrize('kwargs', [{'battery': 0}, {'battery': 1}, {'temperature': 81}, {'thermal_limit': 10}])
def test_energy_thermal_and_safe_return(kwargs):
    result = GeneticPlanner(seed=3).plan(flat(), (0, 0), (4, 5), **kwargs)
    assert not result['feasible'] and result['route'] == []
    assert 'sin solución' in result['message'].lower()
    assert all(c['penalty'] > 0 for c in result['candidates'])


def test_safe_return_is_included_not_only_outbound_energy():
    world = flat()
    planner = GeneticPlanner(seed=1, waypoints=1)
    candidate = planner.evaluate([(0, 0)], world, (0, 0), (0, 5), battery=7)
    assert candidate['energy_required'] == 10
    assert not candidate['feasible'] and 'battery' in candidate['violations']


def test_unreachable_and_bounds_are_explicit():
    world = flat()
    world.obstacles.update({(r, 2) for r in range(5)})
    planner = GeneticPlanner(seed=1)
    result = planner.plan(world, (0, 0), (0, 5))
    assert not result['feasible'] and 'sin solución' in result['message'].lower()
    assert not planner.plan(world, (-1, 0), (0, 5))['feasible']


def test_risk_weights_and_scientific_visit_order():
    world = flat()
    planner = GeneticPlanner(waypoints=2, weights={'science': 5, 'distance': 1, 'energy': 1, 'predicted_risk': 10, 'time': 1})
    world.science_sites = {(3, 0), (0, 3)}
    risk = np.zeros((5, 6))
    risk[0, 3] = 1
    candidate = planner.evaluate([(3, 0), (0, 3)], world, (0, 0), (0, 5), estimated_risk=risk)
    assert candidate['science_order'] == [(3, 0), (0, 3)]
    assert candidate['components']['science'] == 1
    without_risk = planner.evaluate([(3, 0), (0, 3)], world, (0, 0), (0, 5))
    assert candidate['fitness'] < without_risk['fitness']
    world.science_sites.clear()
    assert planner.evaluate([(0, 0)] * 2, world, (0, 0), (0, 5))['components']['science'] == 0


@pytest.mark.parametrize('kwargs', [{'population_size': 29}, {'generations': 29}, {'mutation_rate': float('nan')}, {'weights': {'science': -1}}])
def test_invalid_configuration(kwargs):
    with pytest.raises(ValueError):
        GeneticPlanner(**kwargs)


def test_disconnected_waypoint_is_repaired_without_losing_reachable_goal():
    world = flat()
    world.obstacles.update({(r, 2) for r in range(5)})
    candidate = GeneticPlanner(waypoints=1).evaluate([(0, 5)], world, (0, 0), (4, 0))
    assert candidate['feasible'] and candidate['repaired']
    assert candidate['chromosome'] == [(0, 0)]
    assert candidate['route'][-1] == (4, 0)


def test_physical_cliff_edges_rejected_outbound_and_on_safe_return():
    world = flat(width=3, height=1)
    world.science_sites.clear()
    world.elevation[:] = [[0, -10, 0]]
    result = GeneticPlanner(seed=2).plan(world, (0, 0), (0, 2))
    assert not result['feasible'] and result['route'] == []
    assert 'sin solución' in result['message'].lower()
    assert all(not c['feasible'] and c['penalty'] > 0 for c in result['candidates'])

    # Uphill travel is permitted, but its return edge is a forbidden descent.
    world.elevation[:] = [[0, 10, 0]]
    candidate = GeneticPlanner(waypoints=1).evaluate([(0, 0)], world, (0, 0), (0, 1))
    assert candidate['route'] == [(0, 0), (0, 1)]
    assert not candidate['feasible'] and candidate['energy_required'] == float('inf')
    assert 'battery' in candidate['violations']


In [ ]:
%%writefile ares_x/tests/test_h2_neural.py
"""Leakage, reproducibility and inspectable neural perception contracts."""
import numpy as np
import pytest
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

from ares.neural import FEATURES, LABELS, NeuralPerception, generate_training_data


@pytest.fixture(scope='module')
def fitted():
    model = NeuralPerception(seed=17)
    model.train(map_count=12, samples_per_map=100)
    return model


def test_synthetic_truth_and_group_separation(fitted):
    data = generate_training_data(seed=17, map_count=12, samples_per_map=100)
    assert set(data['y']) == set(LABELS)
    assert data['X'].shape == (1200, len(FEATURES))
    assert np.isfinite(data['X']).all()
    assert 'cliff' not in FEATURES
    assert len(np.unique(data['X'][:, 0])) > 100
    splits = fitted.split_indices_
    groups = [set(data['groups'][splits[key]]) for key in ('train', 'validation', 'test')]
    assert not groups[0] & groups[1] and not groups[0] & groups[2] and not groups[1] & groups[2]
    assert set.union(*groups) == set(data['groups'])
    np.testing.assert_allclose(fitted.scaler.mean_, data['X'][splits['train']].mean(axis=0))
    assert fitted.scaler.n_samples_seen_ == len(splits['train'])
    assert not np.allclose(fitted.scaler.mean_, data['X'].mean(axis=0))
    assert fitted.model.hidden_layer_sizes == (32, 16)
    assert fitted.model.activation == 'relu'
    assert fitted.model.out_activation_ == 'softmax'
    assert fitted.model.early_stopping is False


def test_probability_trace_confidence_and_actual_network(fitted):
    row = dict(distance=9., slope=4., elevation_delta=1., texture=.2, visibility=.8, vibration=.1)
    result = fitted.predict(row)
    probabilities = list(result['probabilities'].values())
    assert list(result['probabilities']) == list(LABELS)
    assert np.isfinite(probabilities).all() and sum(probabilities) == pytest.approx(1.)
    assert result['confidence'] == max(probabilities)
    assert result['low_confidence'] == (result['confidence'] < fitted.confidence_threshold)
    assert 0 <= result['risk'] <= 1
    transformed = fitted.scaler.transform([[row[key] for key in FEATURES]])
    np.testing.assert_allclose(result['normalized_input'], transformed[0])
    direct = fitted.model.predict_proba(transformed)[0]
    for label, probability in zip(fitted.model.classes_, direct):
        assert result['probabilities'][label] == pytest.approx(probability)
    assert [layer['neurons'] for layer in result['activation_summaries']] == [32, 16]
    assert all(0 <= layer['active_fraction'] <= 1 for layer in result['activation_summaries'])
    old_threshold = fitted.confidence_threshold
    fitted.confidence_threshold = 1.
    try:
        assert fitted.predict(row)['low_confidence']
    finally:
        fitted.confidence_threshold = old_threshold


def test_metrics_are_heldout_and_have_calibration(fitted):
    for split in ('validation', 'test'):
        metrics = fitted.report_[split]
        assert np.asarray(metrics['confusion_matrix']).shape == (3, 3)
        assert np.asarray(metrics['confusion_matrix']).sum() == metrics['samples']
        for key in ('accuracy', 'majority_baseline', 'macro_f1', 'barranco_recall', 'calibration_error'):
            assert 0 <= metrics[key] <= 1
        assert metrics['macro_f1'] > .65
        assert metrics['accuracy'] > metrics['majority_baseline']
    assert fitted.operational_label == 'supuesto de simulación'


def test_deterministic_data_training_and_predictions(fitted):
    other = NeuralPerception(seed=17)
    other.train(map_count=12, samples_per_map=100)
    row = [8., 3., -1., .4, .8, .05]
    assert other.predict(row) == fitted.predict(row)
    assert other.report_ == fitted.report_


@pytest.mark.parametrize('value', [None, [], [1] * 5, [np.nan] * 6,
    [np.inf] * 6, [0., 3., 0., .5, 1., 0.], [21., 3., 0., .5, 1., 0.],
    [1e308] * 6, {}, {'distance': 4.}, ['x'] * 6])
def test_invalid_input_is_rejected(fitted, value):
    with pytest.raises(ValueError, match='Entrada|sensor'):
        fitted.predict(value)


def test_untrained_and_invalid_configuration():
    with pytest.raises(ValueError, match='entren'):
        NeuralPerception().predict([8., 3., -1., .4, .8, .05])
    for threshold in (np.nan, 0, 1.1, 'x'):
        with pytest.raises(ValueError, match='confianza'):
            NeuralPerception(confidence_threshold=threshold)
    with pytest.raises(ValueError, match='mapas'):
        generate_training_data(map_count=2)


def test_invalid_training_data_rejected():
    data = generate_training_data(seed=3, map_count=8, samples_per_map=40)
    data['X'][0, 0] = np.nan
    with pytest.raises(ValueError, match='Entrada|datos'):
        NeuralPerception().train(data)


In [ ]:
%%writefile ares_x/tests/test_h3_hybrid.py
"""H3: hybrid integration, acceptance cases CA-01..CA-08, causality and edge cases."""
import json
import math
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

from ares.api import Session, handle_json  # noqa: E402
from ares.mission import HybridMission, validate_config  # noqa: E402


def first_move_mission(**kw):
    mission = HybridMission(seed=42, **kw)
    mission.step()
    return mission


# ------------------------------------------------------------------ CA-01
def test_ca01_mars_reproducible_at_least_ten_steps_consistent_telemetry():
    mission = HybridMission(planet='Marte', seed=42)
    records = [mission.step() for _ in range(12)]
    assert [r['step'] for r in records] == list(range(1, 13))
    for prev, cur in zip(records, records[1:]):
        assert 0 <= cur['battery'] <= 100
        for a, b in zip([prev['position']] + cur['moved'], cur['moved']):
            assert abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1  # contiguous moves only
    assert len(mission.path) > 10


def test_same_seed_and_configuration_gives_identical_trace():
    a = HybridMission(seed=7).run()
    b = HybridMission(seed=7).run()
    ta = HybridMission(seed=7)
    tb = HybridMission(seed=7)
    ta.run(), tb.run()
    assert a['steps'] == b['steps']
    assert json.dumps(ta.logger.trace, sort_keys=True) == json.dumps(tb.logger.trace, sort_keys=True)


# ------------------------------------------------------------------ CA-02
def test_ca02_cliff_in_front_stops_before_danger_and_replans():
    mission = first_move_mission()
    runs_before = len(mission.ga_runs)
    entry = mission.inject('barranco')
    cliff = tuple(entry['position'])
    record = mission.step()
    assert record['reason'] == 'R04_VETO_BARRANCO_CONFIRMADO'
    assert tuple(record['position']) != cliff
    assert record['rna'] is not None and 'barranco' in record['rna']['probabilities']
    assert record['fuzzy']['rules'], 'fuzzy rules must be visible'
    assert len(mission.ga_runs) == runs_before + 1
    assert cliff not in [tuple(c) for c in mission.route]
    mission.run()
    assert cliff not in mission.path
    assert mission.metrics['safety_violations'] == 0


# ------------------------------------------------------------------ CA-03
def test_ca03_low_battery_prioritizes_refuge_or_explains():
    mission = HybridMission(seed=42, battery=12)
    mission.run()
    reasons = {r['reason'] for r in mission.logger.trace}
    assert mission.finished
    assert min(r['battery'] for r in mission.logger.trace) >= 0
    assert reasons & {'R07_RETORNO_ENERGIA', 'R06_REFUGIO_INALCANZABLE', 'R15_MISION_COMPLETA'}
    assert mission.metrics['safety_violations'] == 0


def test_battery_one_percent_far_refuge_declared_unreachable():
    mission = HybridMission(seed=42, battery=100)
    for _ in range(12):
        mission.step()
    mission.battery = 1.
    record = mission.step()
    assert record['reason'] in ('R06_REFUGIO_INALCANZABLE', 'R07_RETORNO_ENERGIA', 'R09_RECARGA', 'R15_MISION_COMPLETA')
    if record['reason'] == 'R06_REFUGIO_INALCANZABLE':
        assert mission.finished and mission.state == 'MODO_SEGURO'


def test_battery_zero_never_moves_and_never_negative():
    mission = HybridMission(seed=3, battery=0)
    record = mission.step()
    assert record['moved'] == [] and record['battery'] == 0
    assert mission.state == 'FRACASO'


# ------------------------------------------------------------------ CA-04
def test_ca04_generation_five_shows_three_chromosomes_and_operators():
    session = Session(seed=42)
    session.dispatch('step', {'n': 1})
    out = session.dispatch('ga_generation', {'generation': 5, 'run': 0})
    assert 'error' not in out
    assert len(out['rows']) >= 3
    assert all('fitness' in row for row in out['rows'])
    assert out['crossovers'] + out['mutations'] > 0


# ------------------------------------------------------------------ CA-05
def test_ca05_distance_10_to_1_changes_fuzzy_output():
    session = Session()
    far = session.dispatch('fuzzy_eval', {'distance': 10, 'battery': 60, 'slope': 5})
    near = session.dispatch('fuzzy_eval', {'distance': 1, 'battery': 60, 'slope': 5})
    assert near['speed'] < far['speed']
    assert {r['id'] for r in near['rules']} != {r['id'] for r in far['rules']}
    assert near['degrees']['distance']['cerca'] == 1.0


# ------------------------------------------------------------------ CA-06
def test_ca06_changing_rna_features_updates_probabilities():
    session = Session()
    base = dict(distance=10, slope=2, elevation_delta=0, texture=.25, visibility=1, vibration=0)
    free = session.dispatch('rna_eval', base)
    drop = session.dispatch('rna_eval', dict(base, distance=.05, elevation_delta=-10, slope=30, vibration=1.5))
    assert free['label'] == 'libre'
    assert drop['probabilities']['barranco'] > free['probabilities']['barranco']
    state = session.dispatch('state')
    assert 0 <= state['nn']['test']['barranco_recall'] <= 1
    assert state['nn']['test']['confusion_matrix']


# ------------------------------------------------------------------ CA-07
def test_ca07_jupiter_wheels_blocked_with_explanation():
    session = Session()
    out = session.dispatch('configure', {'planet': 'Júpiter', 'mode': 'rover'})
    assert 'error' in out and 'sonda' in out['error']
    ok = session.dispatch('configure', {'planet': 'Júpiter', 'mode': 'sonda'})
    assert ok['planet']['mode'] == 'probe'
    session.dispatch('step', {'n': 5})


# ------------------------------------------------------------------ CA-08
def test_ca08_same_seed_full_vs_ablation_and_export():
    session = Session(seed=11)
    out = session.dispatch('compare', {'seeds': 2, 'scenario': 'eventos', 'policies': ['hibrido', 'sin_rna']})
    assert {row['policy'] for row in out['summary']} == {'hibrido', 'sin_rna'}
    for row in out['summary']:
        assert set(row['success']) == {'media', 'mediana', 'desv'}
    session.dispatch('step', {'n': 5})
    csv_text = session.dispatch('export', {'format': 'csv'})['content']
    assert csv_text.count('\n') >= 6
    exported = json.loads(session.dispatch('export', {'format': 'json'})['content'])
    assert exported['config']['seed'] == 11 and len(exported['trace']) == 5


# -------------------------------------------------------------- causality
def test_causality_rna_output_changes_decision(monkeypatch):
    normal = first_move_mission()
    assert normal.step()['reason'] == 'R14_AVANCE'
    forced = first_move_mission()
    real_predict = forced.perception.predict

    def fake(features):
        out = real_predict(features)
        out.update(label='barranco', confidence=.95, probabilities={'libre': .02, 'obstaculo': .03, 'barranco': .95})
        return out
    monkeypatch.setattr(forced.perception, 'predict', fake)
    assert forced.step()['reason'] == 'R05_RNA_PELIGRO'


def test_causality_fuzzy_output_changes_movement(monkeypatch):
    normal = first_move_mission()
    assert normal.step()['moved']
    forced = first_move_mission()
    real_infer = forced.fuzzy.infer
    monkeypatch.setattr(forced.fuzzy, 'infer', lambda *a: {**real_infer(*a), 'speed': .02})
    record = forced.step()
    assert record['reason'] == 'R12_DIFUSO_PARAR' and record['moved'] == []


def test_causality_ga_output_changes_route():
    default = HybridMission(seed=42)
    default.step()
    no_science = HybridMission(seed=42, weights={'science': 0, 'distance': 1, 'energy': 1, 'predicted_risk': 2, 'time': 1})
    no_science.step()
    assert default.planned_history[0] != no_science.planned_history[0]


# ---------------------------------------------------------------- edges
@pytest.mark.parametrize('bad', [{'battery': math.nan}, {'battery': 150}, {'seed': -1}, {'policy': 'magia'},
                                 {'weights': {'science': math.inf}}])
def test_impossible_configuration_rejected_with_readable_message(bad):
    with pytest.raises(ValueError):
        validate_config(bad)
    assert 'error' in json.loads(handle_json(Session(), 'configure', json.dumps(bad).replace('NaN', '"NaN"')))


def test_all_navigation_sensors_failing_never_moves_blind():
    mission = first_move_mission()
    position = mission.position
    mission.inject('falla_sensor', sensor='todos')
    record = mission.step()
    assert record['reason'] == 'R02_SENSORES_NAVEGACION_FALLIDOS'
    assert tuple(record['position']) == position


def test_single_sensor_failure_degrades_without_exception():
    mission = first_move_mission()
    mission.inject('falla_sensor', sensor='distance')
    record = mission.step()
    assert record['reason'] in ('R13_AVANCE_DEGRADADO', 'R12_DIFUSO_PARAR', 'R11_REPLANIFICAR')
    assert record['rna'] is None and record['rna_error']


def test_simultaneous_meteorite_and_low_battery_resolved_deterministically():
    outcomes = []
    for _ in range(2):
        mission = first_move_mission()
        mission.inject('meteorito')
        mission.inject('bateria_baja')
        outcomes.append([mission.step()['reason'] for _ in range(4)])
    assert outcomes[0] == outcomes[1]


def test_map_without_science_sites_closes_without_division_by_zero():
    mission = HybridMission(seed=5, science_count=0)
    summary = mission.run()
    assert summary['science'] == 1.0 and mission.finished


def test_no_route_reports_no_solution():
    mission = HybridMission(seed=42)
    for cell in ((0, 1), (1, 0), (1, 1)):
        mission.world.obstacles.add(cell)
        mission.belief.obstacles.add(cell)
    mission.run()
    assert mission.finished
    assert any(r['reason'] == 'R10_SIN_RUTA' for r in mission.logger.trace)
    assert mission.metrics['distance'] == 0


def test_ga_route_crossing_confirmed_cliff_is_vetoed():
    mission = HybridMission(seed=42)
    accepted, rejected = mission.supervisor.filter_routes(
        [{'route': [(0, 0), (0, 1)], 'feasible': True}, {'route': [(0, 0), (1, 0)], 'feasible': True}],
        known_cliffs={(0, 1)}, known_obstacles=set())
    assert accepted == [1] and rejected[0]['reason'] == 'cruza peligro confirmado'


def test_events_cover_all_kinds_without_exception():
    for kind in ('bateria_baja', 'dia', 'noche', 'sismo', 'meteorito', 'tormenta_polvo', 'corrosion',
                 'biosenal_anomala', 'alienigena', 'falla_sensor', 'perdida_enlace', 'viento', 'barranco'):
        mission = first_move_mission()
        mission.inject(kind)
        mission.step()
        mission.step()


def test_biosignal_never_claims_life():
    mission = first_move_mission()
    entry = mission.inject('biosenal_anomala')
    assert 'no confirma vida' in entry['message']


# ------------------------------------------------- RF-11, RF-03, instruments
def test_initial_route_visible_before_first_step_and_after_reset():
    session = Session(seed=42)
    assert len(session.dispatch('state')['map']['route']) > 1
    session.dispatch('step', {'n': 6})
    snapshot = session.dispatch('reset')
    assert snapshot['status']['step'] == 0 and len(snapshot['map']['route']) > 1


def test_prepare_does_not_change_the_trace():
    plain, prepared = HybridMission(seed=9), HybridMission(seed=9).prepare()
    plain.run(), prepared.run()
    strip = lambda trace: [{k: v for k, v in row.items() if k != 'ga'} for row in trace]
    assert json.dumps(strip(plain.logger.trace), sort_keys=True) == json.dumps(strip(prepared.logger.trace), sort_keys=True)


def test_rf11_training_history_has_loss_and_accuracy_per_iteration():
    history = Session().dispatch('meta')['training']
    assert len(history) >= 10 and history[0]['iteration'] == 1
    assert history[-1]['loss'] < history[0]['loss']
    assert history[-1]['validation_accuracy'] > history[0]['validation_accuracy'] - 1e-9
    assert all(0 <= row['train_accuracy'] <= 1 for row in history)


def test_rf03_step_duration_scales_time_and_energy():
    short, long = HybridMission(seed=42, step_duration=1.), HybridMission(seed=42, step_duration=2.)
    a, b = short.step(), long.step()
    assert b['t'] == 2 * a['t']
    # Energy per cell scales with the step duration (routes may differ, so compare the cost model).
    assert long.cell_cost(.5, 3.) == pytest.approx(2 * short.cell_cost(.5, 3.))
    assert 'error' in Session().dispatch('configure', {'step_duration': 0})


def test_instruments_are_recovered_and_reported():
    mission = HybridMission(seed=42)
    summary = mission.run()
    assert summary['instruments_total'] == 3
    assert summary['instruments'] == 3 and summary['success']
    assert any(e['kind'] == 'instrumento' for e in mission.event_log)
    assert mission.recovered <= set(mission.path)


# --------------------------------------------- unconfirmed RNA beliefs
def test_probe_mission_recovers_from_false_rna_mark_blocking_the_refuge():
    mission = HybridMission(planet='Júpiter', mode='sonda', seed=21)
    summary = mission.run()
    assert summary['success'], summary['outcome']
    assert summary['safety_violations'] == 0


def test_unconfirmed_rna_marks_are_discarded_once_then_become_firm():
    mission = first_move_mission()
    row, col = mission.position
    ring = [(row + dr, col + dc) for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1))]
    ring = [c for c in ring if mission.world.traversable(c)]
    for cell in ring:
        mission._mark_rna(cell, 'obstaculo')
    mission._plan('PRUEBA')
    assert len(mission.route) > 1
    assert not mission.rna_marks and set(ring) <= mission.rna_retried
    assert not set(ring) & mission.belief.obstacles
    mission._mark_rna(ring[0], 'obstaculo')      # second independent RNA flag
    assert ring[0] in mission.belief.obstacles and ring[0] not in mission.rna_marks


def test_confirmed_hazards_are_never_discarded():
    mission = first_move_mission()
    entry = mission.inject('barranco')
    cliff = tuple(entry['position'])
    mission.step()                                # edge sensor confirms it
    mission._mark_rna((5, 5), 'obstaculo')
    mission.rna_marks.add((5, 5))
    for cell in list(mission.rna_marks):
        mission.belief.obstacles.discard(cell)
    mission._plan('PRUEBA')
    assert cliff in mission.belief.cliffs


# ------------------------------------------------------------ event guards
def test_hazard_event_never_defaults_onto_the_rover():
    for seed in range(12):
        mission = HybridMission(seed=seed).prepare()
        mission.step(), mission.step()
        try:
            entry = mission.inject('alienigena')
        except ValueError as error:
            assert 'adyacente' in str(error)       # only when no free neighbour exists
            continue
        assert tuple(entry['position']) != mission.position
        assert 'ficticio' in entry['message']


def test_events_are_rejected_after_the_mission_ends():
    mission = HybridMission(seed=42)
    mission.run()
    with pytest.raises(ValueError, match='ya terminó'):
        mission.inject('alienigena')


def test_alien_is_avoided_without_collision():
    mission = HybridMission(seed=42).prepare()
    mission.step(), mission.step()
    alien = tuple(mission.inject('alienigena')['position'])
    summary = mission.run()
    assert alien not in mission.path and summary['collisions'] == 0 and summary['success']


# ------------------------------------------------------------ event bugs
@pytest.mark.parametrize('planet,mode,seed', [('Marte', 'rover', 3), ('Marte', 'rover', 16),
                                              ('Júpiter', 'sonda', 9), ('Júpiter', 'sonda', 18)])
def test_low_battery_event_returns_to_a_reachable_refuge(planet, mode, seed):
    mission = HybridMission(planet=planet, mode=mode, seed=seed).prepare()
    for _ in range(8):
        mission.step()
    mission.inject('bateria_baja')
    summary = mission.run()
    assert summary['outcome'] != 'Sin solución: no existe ruta factible'
    assert summary['state'] in ('EXITO', 'MODO_SEGURO') and summary['battery_left'] >= 0


def test_critical_damage_ends_the_mission_in_safe_mode():
    mission = HybridMission(seed=42).prepare()
    for _ in range(6):
        mission.step()
    for _ in range(4):
        mission.inject('corrosion')
    summary = mission.run()
    assert summary['state'] == 'MODO_SEGURO' and summary['steps'] <= 8
    assert mission.logger.trace[-1]['reason'] == 'R03_DANO_CRITICO'


def test_wind_in_probe_mode_is_not_reported_as_a_quake():
    probe = HybridMission(planet='Júpiter', mode='sonda', seed=42).prepare()
    probe.step()
    probe.inject('viento')
    record = probe.step()
    assert record['reason'] == 'R16_ESPERA_VIENTO' and record['moved'] == []
    assert probe.wait_steps == 0
    assert probe.step()['reason'] != 'R16_ESPERA_VIENTO'
    rover = HybridMission(seed=42).prepare()
    rover.step()
    rover.inject('viento')
    assert rover.step()['moved']


def test_link_sensor_recovers_after_the_outage():
    mission = HybridMission(seed=42).prepare()
    mission.step()
    mission.inject('perdida_enlace')
    assert mission.step()['sensors']['link_available'] is False
    for _ in range(12):
        record = mission.step()
    assert record['sensors']['link_available'] is True


def test_clearing_a_cliff_restores_the_terrain():
    session = Session(seed=42)
    session.dispatch('step', {'n': 2})
    mission = session.mission
    cell = tuple(session.dispatch('inject', {'kind': 'barranco'})['map']['events'][-1]['position'])
    before = float(mission.belief.elevation[cell])
    session.dispatch('place', {'position': list(cell), 'kind': 'limpiar'})
    assert cell not in mission.world.cliffs
    assert abs(float(mission.world.elevation[cell]) - before) < 1e-6
    assert mission.step()['reason'] != 'R04_VETO_BARRANCO_CONFIRMADO'


In [ ]:
%%writefile ares_x/tests/test_panel_c_fuzzy.py
"""Real mission controls -> Mamdani payload and panel E browser checks.

Verification only: failures document defects; application code is not patched.
Run: .venv/bin/python -m pytest tests/test_panel_c_fuzzy.py -q
"""
import json
import shutil
import socket
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import pytest

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / 'src'))
from ares.api import Session  # noqa: E402
from ares.mission import EVENT_LABELS, POLICIES  # noqa: E402
from ares.planet import PROFILES  # noqa: E402


def command(session, name, **args):
    result = session.dispatch(name, args)
    assert 'error' not in result, result
    return result


def assert_fuzzy(session, snapshot):
    f = snapshot['fuzzy']
    assert (f is not None) == POLICIES[snapshot['config']['policy']]['fuzzy']
    if f is None:
        return
    assert f['speed'] == f['centroid']
    assert f['caution'] == pytest.approx(1 - f['speed'], abs=.0001)
    assert len(f['curve']) == 51 and all(0 <= v <= 1 for v in f['curve'])
    assert len(f['all_rules']) == 12
    active = {r['id']: r['strength'] for r in f['rules']}
    for rule in f['all_rules']:
        strength = min(f['degrees'][var][label] for var, label in rule['antecedents'])
        assert active.get(rule['id'], 0) == pytest.approx(strength, abs=.001)
    manual = command(session, 'fuzzy_eval', **f['inputs'])
    assert manual['reason'] == f['reason']
    assert manual['speed'] == pytest.approx(f['speed'], abs=.003)


CONFIGS = [
    *[{'planet': p[0], 'mode': 'rover' if p[1] else 'sonda'} for p in PROFILES.values()],
    {'mode': 'sonda'}, *[{'policy': p} for p in POLICIES],
    *[{'seed': s} for s in (0, 7, 42)],
    *[{'difficulty': d} for d in ('low', 'medium', 'high')],
    *[{'battery': b} for b in (0, 10, 50, 100)],
    *[{'step_duration': d} for d in (.5, 1, 1.5, 2, 2.5, 3)],
    {'drone': True}, {'drone': False},
]


@pytest.mark.parametrize('config', CONFIGS, ids=str)
def test_configuration_resets_fuzzy_then_uses_selected_policy(config):
    s = Session(seed=42)
    before = command(s, 'step', n=1)
    reset = command(s, 'configure', **config)
    assert reset['fuzzy'] is None and reset['status']['step'] == 0
    for key, value in config.items():
        assert reset['config'][key] == value
    after = command(s, 'step', n=1)
    assert_fuzzy(s, after)
    if after['fuzzy']:
        assert after['fuzzy']['all_rules'] == before['fuzzy']['all_rules']


@pytest.mark.parametrize('kind', list(EVENT_LABELS) + ['navigation_total'])
def test_events_update_fuzzy_on_next_step_not_injection(kind):
    s = Session(seed=42)
    before = command(s, 'step', n=1)
    args = {'kind': kind} if kind != 'navigation_total' else {'kind': 'falla_sensor', 'sensor': 'todos'}
    injected = command(s, 'inject', **args)
    assert injected['fuzzy'] == before['fuzzy']
    assert injected['event_feed']['total'] == before['event_feed']['total'] + 1
    after = command(s, 'step', n=1)
    assert_fuzzy(s, after)
    f = after['fuzzy']
    if kind == 'bateria_baja':
        assert f['inputs']['battery'] < before['fuzzy']['inputs']['battery']
    if kind == 'ola_calor':
        assert f['inputs']['temperature'] > before['fuzzy']['inputs']['temperature']
    if kind == 'navigation_total':
        assert f['inputs']['distance'] == 2


@pytest.mark.parametrize('action,args', [('reset', {}), ('ghosts', {'enabled': True}),
                                      ('ghosts', {'enabled': False})])
def test_restart_controls_clear_fuzzy_and_reproduce_first_inference(action, args):
    s = Session(seed=42)
    before = command(s, 'step', n=1)
    assert command(s, action, **args)['fuzzy'] is None
    assert command(s, 'step', n=1)['fuzzy'] == before['fuzzy']


@pytest.mark.parametrize('kind', ['obstaculo', 'barranco', 'limpiar'])
def test_terrain_and_destination_actions_do_not_recompute_until_step(kind):
    s = Session(seed=42)
    before = command(s, 'step', n=1)
    cell = before['rna']['cell']
    for action, args in [('destination', {'position': cell}),
                         ('destination', {'position': cell}),
                         ('destination', {'position': cell}), ('destination', {}),
                         ('place', {'position': cell, 'kind': kind})]:
        assert command(s, action, **args)['fuzzy'] == before['fuzzy']
    assert_fuzzy(s, command(s, 'step', n=1))


@pytest.mark.parametrize('format', ['csv', 'json', 'comparison_csv'])
def test_exports_preserve_fuzzy_and_mission_state(format):
    s = Session(seed=42)
    before = command(s, 'step', n=1)
    result = s.dispatch('export', {'format': format})
    assert ('error' in result) == (format == 'comparison_csv')
    assert command(s, 'state') == before


def test_zero_battery_input_activates_critical_battery_rule():
    f = command(Session(), 'fuzzy_eval', distance=20, battery=0, slope=45, temperature=20)
    assert f['reason'] == 'INFERENCIA_MAMDANI'
    assert any(r['id'] == 2 and r['strength'] == 1 for r in f['rules'])
    assert 0 < f['speed'] < 1


@pytest.fixture(scope='module')
def browser_results(tmp_path_factory):
    if not (ROOT / 'server.py').exists():
        pytest.skip('server.py and the browser script are not bundled in the notebook')
    assert shutil.which('chromium') and shutil.which('node'), 'Chromium and Node are required'
    scratch = tmp_path_factory.mktemp('panel-c-e')
    with socket.socket() as reservation:
        reservation.bind(('127.0.0.1', 0))
        app_port = reservation.getsockname()[1]
    processes = []
    with (scratch / 'server.log').open('w') as log:
        try:
            server = subprocess.Popen([sys.executable, str(ROOT / 'server.py'), '--port', str(app_port)],
                                      stdout=log, stderr=log, cwd=ROOT)
            processes.append(server)
            def wait_for(predicate, process, timeout):
                deadline = time.monotonic() + timeout
                while not predicate():
                    assert process.poll() is None, (scratch / 'server.log').read_text()
                    assert time.monotonic() < deadline, 'Process startup timed out'
                    time.sleep(.1)
            def listening():
                try:
                    urllib.request.urlopen(f'http://127.0.0.1:{app_port}/', timeout=1).close()
                    return True
                except OSError:
                    return False
            wait_for(listening, server, 90)
            chrome = subprocess.Popen(['chromium', '--headless', '--disable-gpu', '--no-first-run',
                '--no-default-browser-check', '--remote-debugging-port=0',
                f'--user-data-dir={scratch / "profile"}', 'about:blank'], stdout=log, stderr=log)
            processes.append(chrome)
            port_file = scratch / 'profile' / 'DevToolsActivePort'
            wait_for(port_file.exists, chrome, 20)
            run = subprocess.run(['node', str(ROOT / 'tests/panel_c_fuzzy_browser.mjs'),
                port_file.read_text().splitlines()[0], str(app_port)],
                capture_output=True, text=True, timeout=180)
            assert run.returncode == 0, run.stderr + run.stdout
            (scratch / 'browser-results.json').write_text(run.stdout)
            return json.loads(run.stdout)
        finally:
            for process in reversed(processes):
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()


def test_panel_c_to_e_real_browser(browser_results):
    failures = [r for r in browser_results if not r['passed']]
    assert not failures, json.dumps(failures, ensure_ascii=False, indent=2)


In [ ]:
%%writefile ares_x/tests/test_panel_c_genetic.py
"""Mission panel C -> genetic planner F using the real public engine API.

Verification only. Run: .venv/bin/python -m pytest tests/test_panel_c_genetic.py -q
Browser checks require Chromium and Node with built-in WebSocket.
"""
import json
import shutil
import socket
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import pytest

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / 'src'))
from ares.api import Session  # noqa: E402
from ares.mission import EVENT_LABELS, POLICIES  # noqa: E402
from ares.planet import PROFILES  # noqa: E402


def command(session, name, **args):
    result = session.dispatch(name, args)
    assert 'error' not in result, result
    return result


def assert_planner(session, snapshot):
    ga, runs = snapshot['ga'], snapshot['ga_runs']
    if not POLICIES[snapshot['config']['policy']]['ga']:
        assert ga['mode'] == 'voraz' and not runs and not ga['candidates']
        assert 'error' in session.dispatch('ga_generation', {'generation': 0})
        return
    assert runs and ga['mode'] in ('ag', 'retorno')
    for index, run in enumerate(runs):
        assert run['index'] == index and run['reason']
    if ga['mode'] == 'ag':
        assert ga['history'] and ga['generations_run'] == len(ga['history']) - 1
        if ga['chosen'] is not None:
            chosen = next(c for c in ga['candidates'] if c['index'] == ga['chosen'])
            assert chosen['feasible']
            assert not any(r['index'] == ga['chosen'] for r in ga['rejected'])
        for candidate in ga['candidates']:
            assert set(candidate['components']) == {'science', 'distance', 'energy', 'predicted_risk', 'time'}
        for route in snapshot['map']['candidates']:
            assert all(abs(a[0]-b[0])+abs(a[1]-b[1]) == 1 for a, b in zip(route, route[1:]))
            assert not {tuple(c) for c in route} & {tuple(c) for c in snapshot['map']['obstacles']}
    latest = command(session, 'ga_run', run=len(runs)-1)
    generation = command(session, 'ga_generation', run=len(runs)-1, generation=5)
    assert latest['run'] == generation['run'] == len(runs)-1
    assert generation['rows'] and generation['total'] == len(latest['ga']['history'])
    assert all({'chromosome', 'fitness', 'feasible', 'violations', 'components'} <= row.keys()
               for row in generation['rows'])


CONFIGS = [
    *[{'planet': p[0], 'mode': 'rover' if p[1] else 'sonda'} for p in PROFILES.values()],
    {'mode': 'sonda'}, *[{'policy': p} for p in POLICIES],
    *[{'seed': s} for s in (0, 7, 42)],
    *[{'difficulty': d} for d in ('low', 'medium', 'high')],
    *[{'battery': b} for b in (0, 10, 50, 100)],
    *[{'step_duration': d} for d in (.5, 1, 1.5, 2, 2.5, 3)],
    {'drone': True}, {'drone': False},
]


@pytest.mark.parametrize('config', CONFIGS, ids=str)
def test_configuration_prepares_new_plan_before_first_step(config):
    s = Session(seed=42)
    command(s, 'step', n=1)
    applied = command(s, 'configure', **config)
    assert applied['status']['step'] == 0
    for key, value in config.items():
        assert applied['config'][key] == value
    assert_planner(s, applied)
    if POLICIES[applied['config']['policy']]['ga']:
        assert len(applied['ga_runs']) == 1 and applied['ga_runs'][0]['step'] == 0


@pytest.mark.parametrize('kind', list(EVENT_LABELS) + ['navigation_total'])
def test_events_preserve_inspectable_plans_and_replan_when_required(kind):
    s = Session(seed=42)
    before = command(s, 'state')
    args = {'kind': kind} if kind != 'navigation_total' else {'kind': 'falla_sensor', 'sensor': 'todos'}
    injected = command(s, 'inject', **args)
    assert injected['event_feed']['total'] == before['event_feed']['total'] + 1
    if kind in ('meteorito', 'sismo', 'viento', 'bateria_baja', 'ola_calor', 'tormenta_polvo'):
        assert (len(injected['ga_runs']) > len(before['ga_runs'])
                or injected['ga']['mode'] == 'retorno')
    assert_planner(s, injected)
    assert_planner(s, command(s, 'step', n=1))


@pytest.mark.parametrize('action,args', [('reset', {}), ('ghosts', {'enabled': True}),
                                      ('ghosts', {'enabled': False})])
def test_restart_replaces_history_with_seeded_initial_plan(action, args):
    s = Session(seed=42)
    before = command(s, 'state')
    command(s, 'inject', kind='tormenta_polvo')
    reset = command(s, action, **args)
    assert reset['ga'] == before['ga'] and reset['ga_runs'] == before['ga_runs']


def test_in_place_drone_toggle_preserves_plan_and_execution_history():
    s = Session(seed=42)
    before = command(s, 'step', n=1)
    for enabled in (True, False):
        toggled = command(s, 'drone', enabled=enabled)
        assert toggled['config']['drone'] is enabled
        assert toggled['status']['step'] == before['status']['step']
        assert toggled['map']['path'] == before['map']['path']
        assert toggled['ga'] == before['ga'] and toggled['ga_runs'] == before['ga_runs']
        before = command(s, 'step', n=1)
        assert_planner(s, before)


@pytest.mark.parametrize('kind', ['obstaculo', 'barranco', 'limpiar'])
def test_destinations_and_terrain_preserve_public_generation_inspection(kind):
    s = Session(seed=42)
    before = command(s, 'state')
    cell = before['map']['route'][2]
    for action, args in [('destination', {'position': cell}), ('destination', {'position': cell}),
                         ('destination', {'position': cell}), ('destination', {}),
                         ('place', {'position': cell, 'kind': kind})]:
        assert_planner(s, command(s, action, **args))
    assert_planner(s, command(s, 'step', n=1))


@pytest.mark.parametrize('format', ['csv', 'json', 'comparison_csv'])
def test_exports_do_not_change_plans_or_execution_history(format):
    s = Session(seed=42)
    before = command(s, 'state')
    result = s.dispatch('export', {'format': format})
    assert ('error' in result) == (format == 'comparison_csv')
    assert command(s, 'state') == before


def test_generation_inspection_does_not_change_state_or_next_seeded_step():
    first, second = Session(seed=42), Session(seed=42)
    before = command(first, 'state')
    for generation in (0, 5, 999):
        command(first, 'ga_generation', run=0, generation=generation)
        command(first, 'ga_run', run=0)
    assert command(first, 'state') == before
    a, b = command(first, 'step', n=1), command(second, 'step', n=1)
    assert a['ga'] == b['ga'] and a['map'] == b['map']


def test_historical_generation_keeps_original_rows_after_mission_progress():
    s = Session(seed=42)
    before = command(s, 'ga_generation', run=0, generation=5)
    command(s, 'step', n=20)
    after = command(s, 'ga_generation', run=0, generation=5)
    assert after['rows'] == before['rows']
    assert after['best'] == before['best']


@pytest.fixture(scope='module')
def browser_results(tmp_path_factory):
    if not (ROOT / 'server.py').exists():
        pytest.skip('server.py and the browser script are not bundled in the notebook')
    assert shutil.which('chromium') and shutil.which('node'), 'Chromium and Node are required'
    scratch = tmp_path_factory.mktemp('panel-c-f')
    with socket.socket() as reservation:
        reservation.bind(('127.0.0.1', 0))
        app_port = reservation.getsockname()[1]
    processes = []
    with (scratch / 'server.log').open('w') as log:
        try:
            server = subprocess.Popen([sys.executable, str(ROOT / 'server.py'), '--port', str(app_port)],
                                      stdout=log, stderr=log, cwd=ROOT)
            processes.append(server)
            def wait_for(predicate, process, timeout):
                deadline = time.monotonic() + timeout
                while not predicate():
                    assert process.poll() is None, (scratch / 'server.log').read_text()
                    assert time.monotonic() < deadline, 'Process startup timed out'
                    time.sleep(.1)
            def listening():
                try:
                    urllib.request.urlopen(f'http://127.0.0.1:{app_port}/', timeout=1).close()
                    return True
                except OSError:
                    return False
            wait_for(listening, server, 90)
            chrome = subprocess.Popen(['chromium', '--headless', '--disable-gpu', '--no-first-run',
                '--no-default-browser-check', '--remote-debugging-port=0',
                f'--user-data-dir={scratch / "profile"}', 'about:blank'], stdout=log, stderr=log)
            processes.append(chrome)
            port_file = scratch / 'profile/DevToolsActivePort'
            wait_for(port_file.exists, chrome, 20)
            run = subprocess.run(['node', str(ROOT / 'tests/panel_c_genetic_browser.mjs'),
                port_file.read_text().splitlines()[0], str(app_port)], capture_output=True, text=True, timeout=240)
            assert run.returncode == 0, run.stderr + run.stdout
            (scratch / 'browser-results.json').write_text(run.stdout)
            return json.loads(run.stdout)
        finally:
            for process in reversed(processes):
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()


def test_panel_c_to_f_real_browser(browser_results):
    failures = [r for r in browser_results if not r['passed']]
    assert not failures, json.dumps(failures, ensure_ascii=False, indent=2)


In [ ]:
%%writefile ares_x/tests/test_panel_c_neural.py
"""Panel C -> D integration checks using the real public engine interface.

Run: .venv/bin/python -m pytest tests/test_panel_c_neural.py -q
Browser checks additionally require Chromium and Node with built-in WebSocket.
"""
import json
import shutil
import socket
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import pytest

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / 'src'))

from ares.api import Session  # noqa: E402
from ares.mission import EVENT_LABELS, POLICIES  # noqa: E402
from ares.planet import PROFILES  # noqa: E402


@pytest.fixture
def session():
    return Session(seed=42)


def command(session, name, **args):
    result = session.dispatch(name, args)
    assert 'error' not in result, result
    return result


def assert_neural(snapshot):
    prediction = snapshot['rna']
    if prediction is None:
        assert (not POLICIES[snapshot['config']['policy']]['rna']
                or snapshot['rna_error'] or snapshot['status']['finished']
                or snapshot['decision']), snapshot
        return
    assert len(prediction['features']) == len(prediction['normalized']) == 6
    assert set(prediction['probabilities']) == {'libre', 'obstaculo', 'barranco'}
    assert sum(prediction['probabilities'].values()) == pytest.approx(1, abs=.0002)
    assert prediction['confidence'] == max(prediction['probabilities'].values())
    assert prediction['label'] in prediction['probabilities']
    assert len(prediction['cell']) == 2
    assert prediction['layers']


CONFIGS = [
    *[{'planet': profile[0], 'mode': 'rover' if profile[1] else 'sonda'}
      for profile in PROFILES.values()],
    {'mode': 'sonda'},
    *[{'policy': policy} for policy in POLICIES],
    *[{'seed': seed} for seed in (0, 7, 42)],
    *[{'difficulty': level} for level in ('low', 'medium', 'high')],
    *[{'battery': battery} for battery in (0, 50, 100)],
    *[{'step_duration': duration} for duration in (.5, 1, 1.5, 2, 2.5, 3)],
    {'drone': True}, {'drone': False},
]


@pytest.mark.parametrize('config', CONFIGS, ids=lambda c: str(c))
def test_apply_config_clears_prediction_without_retraining(session, config):
    initial = command(session, 'step', n=1)
    assert initial['rna'] is not None
    network = command(session, 'meta')['network']
    applied = command(session, 'configure', **config)
    assert applied['status']['step'] == 0 and applied['rna'] is None
    for key, value in config.items():
        assert applied['config'][key] == value
    after = command(session, 'step', n=1)
    assert_neural(after)
    assert after['nn'] == initial['nn']
    assert command(session, 'meta')['network'] == network
    if 'policy' in config:
        assert (after['rna'] is not None) == POLICIES[config['policy']]['rna']


@pytest.mark.parametrize('kind', list(EVENT_LABELS) + ['navigation_total'])
def test_each_event_retains_last_prediction_until_next_step(session, kind):
    before = command(session, 'step', n=1)
    args = {'kind': kind} if kind != 'navigation_total' else {'kind': 'falla_sensor', 'sensor': 'todos'}
    injected = command(session, 'inject', **args)
    assert injected['rna'] == before['rna']
    assert injected['event_feed']['total'] == before['event_feed']['total'] + 1
    after = command(session, 'step', n=1)
    assert after['status']['step'] == before['status']['step'] + 1
    assert after['nn'] == before['nn']
    assert_neural(after)
    if kind == 'navigation_total':
        assert after['rna'] is None and after['rna_error']


@pytest.mark.parametrize('operation,args', [
    ('reset', {}), ('ghosts', {'enabled': True}), ('ghosts', {'enabled': False}),
])
def test_reset_and_ghost_controls_clear_live_inference(session, operation, args):
    before = command(session, 'step', n=1)
    reset = command(session, operation, **args)
    assert reset['rna'] is None and reset['status']['step'] == 0
    after = command(session, 'step', n=1)
    assert after['rna'] == before['rna'] and after['nn'] == before['nn']


@pytest.mark.parametrize('kind', ['obstaculo', 'barranco', 'limpiar'])
def test_terrain_edit_is_observed_on_next_step(session, kind):
    before = command(session, 'step', n=1)
    cell = before['rna']['cell']
    edited = command(session, 'place', position=cell, kind=kind)
    assert edited['rna'] == before['rna']
    if kind == 'limpiar':
        assert cell not in edited['map']['cliffs'] + edited['map']['obstacles']
    else:
        key = 'hidden_cliffs' if kind == 'barranco' else 'hidden_obstacles'
        assert cell in edited['map'][key]
    after = command(session, 'step', n=1)
    assert_neural(after)
    assert after['nn'] == before['nn']


def test_destination_add_remove_and_clear_do_not_retrain(session):
    before = command(session, 'step', n=1)
    cell = before['rna']['cell']
    added = command(session, 'destination', position=cell)
    assert cell in added['map']['destinations'] and added['rna'] == before['rna']
    removed = command(session, 'destination', position=cell)
    assert cell not in removed['map']['destinations']
    command(session, 'destination', position=cell)
    cleared = command(session, 'destination')
    assert cleared['map']['destinations'] == []
    assert_neural(command(session, 'step', n=1))
    assert command(session, 'state')['nn'] == before['nn']


@pytest.mark.parametrize('format', ['csv', 'json', 'comparison_csv'])
def test_export_does_not_change_panel_d(session, format):
    before = command(session, 'step', n=1)
    result = session.dispatch('export', {'format': format})
    if format == 'comparison_csv':
        assert 'Primero' in result['error']
    else:
        assert result['content'] and result['filename'].endswith('.' + format)
    assert command(session, 'state') == before


def test_live_inference_matches_panel_d_manual_evaluation(session):
    live = command(session, 'step', n=1)['rna']
    manual = command(session, 'rna_eval', **live['features'])
    assert manual['label'] == live['label']
    for label, probability in live['probabilities'].items():
        assert manual['probabilities'][label] == pytest.approx(probability, abs=.002)


@pytest.fixture(scope='module')
def browser_results(tmp_path_factory):
    if not (ROOT / 'server.py').exists():
        pytest.skip('server.py and the browser script are not bundled in the notebook')
    assert shutil.which('chromium') and shutil.which('node'), 'Chromium and Node are required'
    scratch = tmp_path_factory.mktemp('panel-c-d')
    with socket.socket() as reservation:
        reservation.bind(('127.0.0.1', 0))
        app_port = reservation.getsockname()[1]
    processes = []
    with (scratch / 'server.log').open('w') as log:
        try:
            server = subprocess.Popen([sys.executable, str(ROOT / 'server.py'), '--port', str(app_port)],
                                      stdout=log, stderr=log, cwd=ROOT)
            processes.append(server)
            deadline = time.monotonic() + 90
            while True:
                try:
                    urllib.request.urlopen(f'http://127.0.0.1:{app_port}/', timeout=1).close()
                    break
                except OSError:
                    assert server.poll() is None, (scratch / 'server.log').read_text()
                    assert time.monotonic() < deadline, 'Server startup timed out'
                    time.sleep(.2)
            chrome = subprocess.Popen(['chromium', '--headless', '--disable-gpu',
                '--no-first-run', '--no-default-browser-check', '--remote-debugging-port=0',
                f'--user-data-dir={scratch / "profile"}', 'about:blank'],
                stdout=log, stderr=log)
            processes.append(chrome)
            port_file = scratch / 'profile' / 'DevToolsActivePort'
            deadline = time.monotonic() + 20
            while not port_file.exists():
                assert chrome.poll() is None, (scratch / 'server.log').read_text()
                assert time.monotonic() < deadline, 'Chromium startup timed out'
                time.sleep(.1)
            port = port_file.read_text().splitlines()[0]
            run = subprocess.run(['node', str(ROOT / 'tests' / 'panel_c_neural_browser.mjs'), port, str(app_port)],
                                 capture_output=True, text=True, timeout=180)
            assert run.returncode == 0, run.stderr + run.stdout
            (scratch / 'browser-results.json').write_text(run.stdout)
            return json.loads(run.stdout)
        finally:
            for process in reversed(processes):
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()


def test_panel_c_to_d_real_browser(browser_results):
    failures = [result for result in browser_results if not result['passed']]
    assert not failures, json.dumps(failures, ensure_ascii=False, indent=2)


In [ ]:
%%writefile ares_x/tests/test_panel_c_telemetry.py
"""Real mission controls -> telemetry API and panel A rendering.

Verification only. Run: .venv/bin/python -m pytest tests/test_panel_c_telemetry.py -q
Chromium and Node with built-in WebSocket are required for the browser test.
"""
import json
import math
import shutil
import socket
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

import pytest

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / 'src'))
from ares.api import Session  # noqa: E402
from ares.mission import EVENT_LABELS, POLICIES  # noqa: E402
from ares.planet import PROFILES  # noqa: E402


def command(session, name, **args):
    result = session.dispatch(name, args)
    assert 'error' not in result, result
    return result


def assert_readings(snapshot):
    t, st, planet = snapshot['telemetry'], snapshot['status'], snapshot['planet']
    assert 0 <= st['battery'] <= 100 and 0 <= st['health'] <= 100
    for key in ('distance', 'elevation_delta', 'slope', 'temperature_external',
                'radiation', 'pressure', 'oxygen_ambient', 'oxygen_stored', 'solar_power'):
        assert t[key] is None or math.isfinite(t[key]), (key, t[key])
    assert t['pressure'] == pytest.approx(planet['pressure'])
    assert t['oxygen_ambient'] == pytest.approx(planet['oxygen'])
    assert t['solar_power'] >= 0
    assert set(t['nav_valid']) == {'proximidad', 'lidar'}


CONFIGS = [
    *[{'planet': p[0], 'mode': 'rover' if p[1] else 'sonda'} for p in PROFILES.values()],
    {'mode': 'sonda'}, *[{'policy': p} for p in POLICIES],
    *[{'seed': s} for s in (0, 7, 42)],
    *[{'difficulty': d} for d in ('low', 'medium', 'high')],
    *[{'battery': b} for b in (0, 10, 50, 100)],
    *[{'step_duration': d} for d in (.5, 1, 1.5, 2, 2.5, 3)],
    {'drone': True}, {'drone': False},
]


@pytest.mark.parametrize('config', CONFIGS, ids=str)
def test_configuration_resets_readings_then_samples_selected_environment(config):
    s = Session(seed=42)
    command(s, 'step', n=1)
    applied = command(s, 'configure', **config)
    assert applied['status']['step'] == 0
    assert applied['status']['battery'] == applied['config']['battery']
    assert applied['status']['events_total'] == 0
    assert applied['telemetry']['distance'] is None
    assert applied['telemetry']['pressure'] is None
    for key, value in config.items():
        assert applied['config'][key] == value
    assert_readings(command(s, 'step', n=1))


@pytest.mark.parametrize('kind', list(EVENT_LABELS) + ['navigation_total'])
def test_events_update_current_status_and_sample_readings_on_next_step(kind):
    s = Session(seed=42)
    before = command(s, 'step', n=1)
    args = {'kind': kind} if kind != 'navigation_total' else {'kind': 'falla_sensor', 'sensor': 'todos'}
    injected = command(s, 'inject', **args)
    assert injected['telemetry'] == before['telemetry']
    assert injected['status']['events_total'] == before['status']['events_total'] + 1
    st, old = injected['status'], before['status']
    if kind == 'bateria_baja':
        assert st['battery'] == min(old['battery'], 10)
    if kind == 'corrosion':
        assert st['health'] == pytest.approx(old['health'] - 20)
    if kind == 'sismo':
        assert st['health'] == pytest.approx(old['health'] - 3) and st['vibration'] > 0
    if kind == 'ola_calor':
        assert st['temperature_internal'] == pytest.approx(old['temperature_internal'] + 35)
    if kind == 'tormenta_polvo':
        assert st['visibility'] == .2 and st['storm'] > 0
    if kind in ('dia', 'noche'):
        assert st['daylight'] is (kind == 'dia')
    if kind == 'perdida_enlace':
        assert not st['link']
    if kind == 'biosenal_anomala':
        assert st['anomalies'] == old['anomalies'] + 1
    if kind == 'alienigena':
        assert st['contacts'] == old['contacts'] + 1
    after = command(s, 'step', n=1)
    assert_readings(after)
    if kind in ('falla_sensor', 'navigation_total'):
        assert after['telemetry']['nav_valid']['proximidad'] is False
        assert after['telemetry']['distance'] is None
    if kind == 'navigation_total':
        assert after['telemetry']['nav_valid']['lidar'] is False


@pytest.mark.parametrize('action,args', [('reset', {}), ('ghosts', {'enabled': True}),
                                      ('ghosts', {'enabled': False})])
def test_restart_controls_clear_measurements_and_main_rover_counters(action, args):
    s = Session(seed=42)
    command(s, 'step', n=2)
    command(s, 'inject', kind='corrosion')
    reset = command(s, action, **args)
    assert reset['status']['step'] == reset['status']['events_total'] == 0
    assert reset['status']['health'] == 100
    assert reset['telemetry']['temperature_external'] is None
    assert_readings(command(s, 'step', n=1))


def test_drone_toggle_preserves_measurements_and_mission_progress():
    s = Session(seed=42)
    before = command(s, 'step', n=1)
    for enabled in (True, False):
        toggled = command(s, 'drone', enabled=enabled)
        assert toggled['telemetry'] == before['telemetry']
        assert toggled['status']['step'] == before['status']['step']
        assert bool(toggled['drone']) is enabled
        before = command(s, 'step', n=1)
        assert_readings(before)


@pytest.mark.parametrize('duration', [.5, 1, 1.5, 2, 2.5, 3])
def test_step_duration_updates_elapsed_time_and_disconnected_link_latency(duration):
    s = Session(seed=42, step_duration=duration)
    command(s, 'inject', kind='perdida_enlace')
    after = command(s, 'step', n=2)
    assert after['status']['t'] == pytest.approx(2 * duration)
    assert after['status']['link_latency'] == pytest.approx(2 * duration)


@pytest.mark.parametrize('format', ['csv', 'json', 'comparison_csv'])
def test_exports_do_not_mutate_telemetry(format):
    s = Session(seed=42)
    before = command(s, 'step', n=1)
    result = s.dispatch('export', {'format': format})
    assert ('error' in result) == (format == 'comparison_csv')
    assert command(s, 'state') == before


def test_same_seed_and_logical_steps_reproduce_telemetry():
    sessions = [Session(seed=42), Session(seed=42)]
    for operation, args in [('step', {'n': 1}), ('inject', {'kind': 'noche'}),
                            ('step', {'n': 2})]:
        a, b = [command(s, operation, **args) for s in sessions]
        assert a['telemetry'] == b['telemetry'] and a['status'] == b['status']


@pytest.fixture(scope='module')
def browser_results(tmp_path_factory):
    if not (ROOT / 'server.py').exists():
        pytest.skip('server.py and the browser script are not bundled in the notebook')
    assert shutil.which('chromium') and shutil.which('node'), 'Chromium and Node are required'
    scratch = tmp_path_factory.mktemp('panel-c-a')
    with socket.socket() as reservation:
        reservation.bind(('127.0.0.1', 0))
        app_port = reservation.getsockname()[1]
    processes = []
    with (scratch / 'server.log').open('w') as log:
        try:
            server = subprocess.Popen([sys.executable, str(ROOT / 'server.py'), '--port', str(app_port)],
                                      stdout=log, stderr=log, cwd=ROOT)
            processes.append(server)
            def wait_for(predicate, process, timeout):
                deadline = time.monotonic() + timeout
                while not predicate():
                    assert process.poll() is None, (scratch / 'server.log').read_text()
                    assert time.monotonic() < deadline, 'Process startup timed out'
                    time.sleep(.1)
            def listening():
                try:
                    urllib.request.urlopen(f'http://127.0.0.1:{app_port}/', timeout=1).close()
                    return True
                except OSError:
                    return False
            wait_for(listening, server, 90)
            chrome = subprocess.Popen(['chromium', '--headless', '--disable-gpu', '--no-first-run',
                '--no-default-browser-check', '--remote-debugging-port=0',
                f'--user-data-dir={scratch / "profile"}', 'about:blank'], stdout=log, stderr=log)
            processes.append(chrome)
            port_file = scratch / 'profile/DevToolsActivePort'
            wait_for(port_file.exists, chrome, 20)
            run = subprocess.run(['node', str(ROOT / 'tests/panel_c_telemetry_browser.mjs'),
                port_file.read_text().splitlines()[0], str(app_port)], capture_output=True, text=True, timeout=360)
            assert run.returncode == 0, run.stderr + run.stdout
            (scratch / 'browser-results.json').write_text(run.stdout)
            (scratch / 'browser-diagnostics.json').write_text(run.stderr)
            return json.loads(run.stdout)
        finally:
            for process in reversed(processes):
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()


def test_panel_c_to_a_real_browser(browser_results):
    failures = [r for r in browser_results if not r['passed']]
    assert not failures, json.dumps(failures, ensure_ascii=False, indent=2)


In [ ]:
%%writefile ares_x/tests/test_panel_e_states.py
"""Panel E must tell 'no inference yet' apart from 'policy without fuzzy logic'."""
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

from ares.api import Session  # noqa: E402
from ares.mission import POLICIES  # noqa: E402


@pytest.mark.parametrize('policy', list(POLICIES))
def test_snapshot_says_which_components_the_policy_uses(policy):
    snapshot = Session(seed=42, policy=policy).dispatch('state')
    flags = POLICIES[policy]
    assert snapshot['policy_uses'] == {'rna': flags['rna'], 'fuzzy': flags['fuzzy'], 'ga': flags['ga']}


def test_a_hybrid_mission_at_step_zero_has_fuzzy_enabled_but_no_inference_yet():
    session = Session(seed=42)
    session.dispatch('step', {})
    snapshot = session.dispatch('reset')
    assert snapshot['fuzzy'] is None and snapshot['policy_uses']['fuzzy'] is True
    assert session.dispatch('step', {})['fuzzy'] is not None


def test_a_historical_ga_run_keeps_its_candidate_routes_for_the_minimap():
    session = Session(seed=42)
    first = session.dispatch('step', {})
    session.dispatch('inject', {'kind': 'tormenta_polvo'})          # forces a new GA run
    run = session.dispatch('ga_run', {'run': 0})
    routes = run['ga']['routes']
    assert len(routes) == len(run['ga']['candidates']) and all(len(r) >= 2 for r in routes)
    assert routes[run['ga']['chosen']] == first['map']['candidates'][first['map']['chosen']]


In [ ]:
%%writefile ares_x/tests/test_v11_destination.py
"""v1.1 F6: operator-defined destination."""
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

from ares.api import Session  # noqa: E402
from ares.mission import HybridMission  # noqa: E402


def started(steps=2, **config):
    config.setdefault('seed', 42)
    mission = HybridMission(**config).prepare()
    for _ in range(steps):
        mission.step()
    return mission


def free_cell_off_route(mission):
    route = {tuple(c) for c in mission.route}
    return next(c for c in sorted(mission._free_cells(), key=lambda c: -(c[0] + c[1]))
                if c not in route and c not in mission.world.cliffs and c not in mission.world.obstacles
                and mission._bfs(mission.position, c))


def test_valid_destination_replans_at_once_and_the_rover_reaches_it():
    mission = started()
    target = free_cell_off_route(mission)
    step = mission.step_index
    entry = mission.set_destination(list(target))
    assert mission.step_index == step and entry['kind'] == 'destino'
    assert tuple(mission.route[-1]) == target and mission.destination == target
    summary = mission.run()
    assert target in mission.path
    assert any(e['kind'] == 'destino_alcanzado' for e in mission.event_log)
    assert mission.destination is None and summary['safety_violations'] == 0


def test_mission_continues_after_reaching_the_destination():
    mission = started()
    target = free_cell_off_route(mission)
    mission.set_destination(list(target))
    while mission.destination is not None and not mission.finished:
        mission.step()
    assert not mission.finished                      # arriving is not the end of the mission
    summary = mission.run()
    assert summary['state'] in ('EXITO', 'MODO_SEGURO')


def test_destination_on_a_known_hazard_is_rejected_without_changing_the_route():
    mission = started()
    cliff = tuple(mission.inject('barranco')['position'])
    mission.step()                                   # edge sensor confirms it
    before = [tuple(c) for c in mission.route]
    with pytest.raises(ValueError, match='obstáculo o barranco'):
        mission.set_destination(list(cliff))
    assert [tuple(c) for c in mission.route] == before and mission.destination is None


def test_destination_without_energy_for_a_safe_return_is_rejected():
    mission = started()
    target = free_cell_off_route(mission)
    mission.battery = 3.
    with pytest.raises(ValueError, match='retorno seguro'):
        mission.set_destination(list(target))
    assert mission.destination is None


@pytest.mark.parametrize('cell', [[-1, 0], [0, 99], [12, 3]])
def test_destination_outside_the_map_is_rejected(cell):
    with pytest.raises(ValueError, match='fuera del mapa'):
        started().set_destination(cell)


def test_destination_can_be_cancelled():
    mission = started()
    mission.set_destination(list(free_cell_off_route(mission)))
    mission.set_destination(None)
    assert mission.destination is None and mission.run()['state'] in ('EXITO', 'MODO_SEGURO')


def test_destination_never_overrides_a_safety_veto():
    mission = started()
    target = free_cell_off_route(mission)
    mission.set_destination(list(target))
    cliff = tuple(mission.inject('barranco')['position'])
    record = mission.step()
    assert record['reason'] == 'R04_VETO_BARRANCO_CONFIRMADO' and mission.position != cliff
    assert mission.run()['safety_violations'] == 0


def test_destination_through_the_panel_api():
    session = Session(seed=42)
    session.dispatch('step', {'n': 2})
    target = list(free_cell_off_route(session.mission))
    snapshot = session.dispatch('destination', {'position': target})
    assert snapshot['map']['destination'] == target and snapshot['status']['destination_distance'] > 0
    assert 'error' in session.dispatch('destination', {'position': [99, 99]})
    assert session.dispatch('destination', {'position': None})['map']['destination'] is None


In [ ]:
%%writefile ares_x/tests/test_v11_events.py
"""v1.1: heat wave (F8), per-event metrics (F4) and immediate replanning (F9)."""
import json
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

from ares.api import Session  # noqa: E402
from ares.mission import HybridMission  # noqa: E402


def pair(steps=3, **config):
    """Two identical prepared missions advanced the same number of steps."""
    config.setdefault('seed', 42)
    missions = [HybridMission(**config).prepare() for _ in range(2)]
    for mission in missions:
        for _ in range(steps):
            mission.step()
    return missions


# ------------------------------------------------------------------ F8
def test_heat_wave_on_mars_raises_temperatures_and_slows_the_rover():
    base, hot = pair()
    cool_cost = hot.cell_cost(.5)
    hot.inject('ola_calor')
    assert hot.temperature_internal == pytest.approx(base.temperature_internal + 35.)
    assert hot.cell_cost(.5) == pytest.approx(1.2 * cool_cost)
    a, b = base.step(), hot.step()
    assert b['sensors']['temperature_external'] == pytest.approx(a['sensors']['temperature_external'] + 40., abs=1.)
    assert b['crisp_speed'] < a['crisp_speed']
    assert b['reason'] != 'R01_LIMITE_TERMICO' and not hot.finished


def test_heat_wave_ends_after_twelve_steps():
    base, hot = pair()
    hot.inject('ola_calor')
    for _ in range(12):
        assert hot.timers['heat'] > 0
        hot.step()
    assert hot.timers['heat'] == 0
    assert hot.temperature_internal == pytest.approx(base.temperature_internal)
    assert hot.cell_cost(.5) == pytest.approx(base.cell_cost(.5))


def test_heat_wave_on_venus_exceeds_the_thermal_limit():
    mission = HybridMission(planet='Venus', seed=42).prepare()
    mission.step()
    mission.inject('ola_calor')
    record = mission.step()
    assert record['reason'] == 'R01_LIMITE_TERMICO'
    assert mission.finished and mission.state == 'MODO_SEGURO'


def test_heat_wave_is_available_from_the_panel_api():
    session = Session(seed=42)
    assert 'ola_calor' in session.dispatch('meta')['events']
    session.dispatch('step', {'n': 2})
    snapshot = session.dispatch('inject', {'kind': 'ola_calor'})
    assert snapshot['status']['heat'] == 12


# ------------------------------------------------------------------ F4
def test_quake_shakes_and_damages():
    base, shaken = pair()
    shaken.inject('sismo')
    assert shaken.health == pytest.approx(base.health - 3.)
    assert shaken.vibration_level > base.vibration_level == 0


def test_meteorite_damages_only_when_it_lands_next_to_the_rover():
    base, near = pair()
    row, col = near.position
    adjacent = next(c for c in ((row, col + 1), (row + 1, col), (row, col - 1), (row - 1, col))
                    if near.world.in_bounds(c) and c not in near.world.refuges)
    near.inject('meteorito', position=list(adjacent))
    assert near.health == pytest.approx(base.health - 5.)
    assert near.vibration_level > 0
    _, far = pair()
    far.inject('meteorito', position=[11, 15] if far.position != (11, 15) else [11, 14])
    assert far.health == pytest.approx(base.health)


def test_biosignal_costs_analysis_energy_and_never_claims_life():
    base, probe = pair()
    entry = probe.inject('biosenal_anomala')
    assert probe.battery == pytest.approx(base.battery - 1.)
    assert probe.summary()['anomalies'] == 1
    assert 'no confirma vida' in entry['message']


def test_alien_contact_is_counted():
    _, mission = pair()
    mission.inject('alienigena')
    assert mission.summary()['contacts'] == 1


def test_link_loss_reports_latency_and_queued_orders():
    _, mission = pair()
    assert mission.link_latency == 0 and mission.queued_orders == 0
    mission.inject('perdida_enlace')
    mission.step(), mission.step()
    assert mission.link_latency > 0
    mission.inject('dia')                    # operator order issued while the link is down
    assert mission.queued_orders == 1
    for _ in range(12):
        mission.step()
    assert mission.link_latency == 0 and mission.queued_orders == 0
    assert mission.summary()['link_outage_steps'] == 10


@pytest.mark.parametrize('planet,mode', [('Marte', 'rover'), ('Júpiter', 'sonda')])
def test_wind_raises_consumption_for_five_steps(planet, mode):
    base, windy = pair(planet=planet, mode=mode)
    windy.inject('viento')
    assert windy.wind_speed > base.wind_speed == 0
    assert windy.cell_cost(.5) == pytest.approx(1.3 * base.cell_cost(.5))
    for _ in range(6):
        windy.step()
    assert windy.wind_speed == 0 and windy.cell_cost(.5) == pytest.approx(base.cell_cost(.5))


def test_sensor_failure_counts_degraded_steps():
    _, mission = pair()
    mission.inject('falla_sensor', sensor='distance')
    for _ in range(4):
        mission.step()
    assert mission.summary()['degraded_steps'] >= 3


def test_summary_and_exports_count_events_by_kind():
    _, mission = pair()
    for kind in ('sismo', 'sismo', 'tormenta_polvo'):
        mission.inject(kind)
    for _ in range(4):
        mission.step()
    summary = mission.summary()
    assert summary['events']['sismo'] == 2 and summary['events']['tormenta_polvo'] == 1
    assert summary['events_total'] == 3 and summary['wait_steps'] >= 2
    exported = mission.export_json()
    assert exported['summary']['events_total'] == 3
    assert json.dumps(exported, allow_nan=False)
    assert 'event_count' in mission.export_csv().splitlines()[0]


def test_panel_status_exposes_event_metrics():
    session = Session(seed=42)
    session.dispatch('step', {'n': 2})
    status = session.dispatch('inject', {'kind': 'sismo'})['status']
    assert status['vibration'] > 0 and status['events_total'] == 1
    assert {'wind', 'link_latency', 'queued_orders', 'heat'} <= set(status)


# ------------------------------------------------------------------ F9
def test_perceivable_event_replans_immediately_without_a_step():
    _, mission = pair(steps=2)
    step, runs = mission.step_index, len(mission.ga_runs)
    target = tuple(mission.route[3])
    mission.inject('meteorito')
    assert mission.step_index == step
    assert target not in [tuple(c) for c in mission.route]
    assert len(mission.ga_runs) == runs + 1
    record = mission.step()
    assert len(mission.ga_runs) == runs + 1          # the AG does not run twice for one event
    assert record['ga'] is not None and record['ga']['reason'].startswith('METEORITO')


@pytest.mark.parametrize('kind', ['sismo', 'viento', 'bateria_baja', 'ola_calor', 'tormenta_polvo'])
def test_other_perceivable_events_also_replan_at_once(kind):
    _, mission = pair(steps=2)
    replans = mission.metrics['replans']
    mission.inject(kind)
    assert mission.metrics['replans'] > replans and mission.replan_reason is None


def test_hidden_hazard_does_not_replan_until_sensed():
    _, mission = pair(steps=2)
    before, replans = [tuple(c) for c in mission.route], mission.metrics['replans']
    cliff = tuple(mission.inject('barranco')['position'])
    assert [tuple(c) for c in mission.route] == before and mission.metrics['replans'] == replans
    assert mission.step()['reason'] == 'R04_VETO_BARRANCO_CONFIRMADO'
    assert cliff not in [tuple(c) for c in mission.route]


def test_event_sequences_remain_reproducible():
    traces = []
    for _ in range(2):
        _, mission = pair(steps=2)
        for kind in ('meteorito', 'ola_calor', 'viento'):
            mission.inject(kind)
            mission.step()
        mission.run()
        traces.append(json.dumps(mission.logger.trace, sort_keys=True))
    assert traces[0] == traces[1]


In [ ]:
%%writefile ares_x/tests/test_v11_multi_destination.py
"""v1.1 F13: several operator destinations visited in the best order."""
import itertools
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

from ares.api import Session  # noqa: E402
from ares.genetic import optimize_visit_order  # noqa: E402
from ares.mission import MAX_DESTINATIONS, HybridMission  # noqa: E402


def started(steps=2, **config):
    config.setdefault('seed', 42)
    mission = HybridMission(**config).prepare()
    for _ in range(steps):
        mission.step()
    return mission


def spread_cells(mission, count):
    """Reachable free cells far apart from each other, in a deliberately bad order."""
    cells = [c for c in sorted(mission._free_cells())
             if c != mission.position and c not in mission.world.cliffs and c not in mission.world.obstacles
             and mission._bfs(mission.position, c)]
    picked = [max(cells, key=lambda c: c[0] + c[1])]
    while len(picked) < count:
        picked.append(max((c for c in cells if c not in picked),
                          key=lambda c: min(abs(c[0] - p[0]) + abs(c[1] - p[1]) for p in picked)))
    return picked


def brute_force(leg, closing):
    stops = range(1, len(leg))
    cost = lambda order: (leg[0][order[0]] + sum(leg[a][b] for a, b in zip(order, order[1:])) + closing[order[-1]])
    return min(cost(order) for order in itertools.permutations(stops))


@pytest.mark.parametrize('seed', range(5))
def test_the_genetic_order_matches_the_exact_optimum(seed):
    import random
    rng = random.Random(seed)
    points = [(rng.randrange(12), rng.randrange(16)) for _ in range(8)]   # start + 7 stops
    leg = [[abs(a[0] - b[0]) + abs(a[1] - b[1]) for b in points] for a in points]
    closing = [p[0] + p[1] for p in points]
    result = optimize_visit_order(leg, closing, seed=seed)
    assert sorted(result['order']) == list(range(7))
    assert result['cost'] == brute_force(leg, closing)
    assert result['history'][-1] <= result['history'][0]


def test_the_genetic_order_is_reproducible_and_handles_trivial_cases():
    leg = [[0, 4, 9], [4, 0, 3], [9, 3, 0]]
    assert optimize_visit_order(leg, [4, 9, 0], seed=1) == optimize_visit_order(leg, [4, 9, 0], seed=1)
    assert optimize_visit_order([[0, 5], [5, 0]], [0, 5])['order'] == [0]
    assert optimize_visit_order([[0]], [0])['order'] == []


def test_several_destinations_are_kept_and_all_are_visited():
    mission = started()
    targets = spread_cells(mission, 3)
    for cell in targets:
        mission.set_destination(list(cell))
    assert sorted(mission.destinations) == sorted(targets)
    assert mission.destination == mission.destinations[0] == tuple(mission.route[-1])
    summary = mission.run()
    assert all(cell in mission.path for cell in targets)
    assert mission.destinations == [] and summary['safety_violations'] == 0
    assert sum(e['kind'] == 'destino_alcanzado' for e in mission.event_log) == 3


def test_the_visit_order_is_the_shortest_tour_not_the_order_of_entry():
    mission = started()
    targets = spread_cells(mission, 4)
    for cell in targets:
        mission.set_destination(list(cell))
    nodes = [mission.position] + targets
    dist = lambda a, b: len(mission._bfs(a, b)) - 1
    leg = [[dist(a, b) for b in nodes] for a in nodes]
    closing = [len(mission._bfs(n, goals=set(mission.world.refuges))) - 1 for n in nodes]
    order = [nodes.index(cell) for cell in mission.destinations]
    cost = leg[0][order[0]] + sum(leg[a][b] for a, b in zip(order, order[1:])) + closing[order[-1]]
    assert cost == brute_force(leg, closing)
    assert any(e['kind'] == 'orden_destinos' for e in mission.event_log)


def test_a_destination_crossed_on_the_way_counts_as_visited():
    mission = started()
    far = spread_cells(mission, 1)[0]
    mission.set_destination(list(far))
    on_the_way = tuple(mission.route[len(mission.route) // 2])
    mission.set_destination(list(on_the_way))
    assert mission.destinations[0] == on_the_way
    mission.run()
    assert mission.destinations == [] and on_the_way in mission.path and far in mission.path


def test_one_destination_can_be_removed_and_none_clears_all():
    mission = started()
    a, b, c = spread_cells(mission, 3)
    for cell in (a, b, c):
        mission.set_destination(list(cell))
    mission.remove_destination(list(b))
    assert sorted(mission.destinations) == sorted([a, c])
    with pytest.raises(ValueError, match='No hay un destino'):
        mission.remove_destination(list(b))
    mission.set_destination(None)
    assert mission.destinations == [] and mission.destination is None


def test_duplicates_and_too_many_destinations_are_rejected():
    mission = started()
    cells = spread_cells(mission, MAX_DESTINATIONS + 1)
    mission.battery = 1e6                                # energy is not the subject here
    mission.set_destination(list(cells[0]))
    with pytest.raises(ValueError, match='ya está fijado'):
        mission.set_destination(list(cells[0]))
    for cell in cells[1:MAX_DESTINATIONS]:
        mission.set_destination(list(cell))
    with pytest.raises(ValueError, match='Máximo'):
        mission.set_destination(list(cells[-1]))
    assert len(mission.destinations) == MAX_DESTINATIONS


def test_a_destination_that_makes_the_whole_tour_too_expensive_is_rejected():
    mission = started()
    a, b = spread_cells(mission, 2)
    out = len(mission._bfs(mission.position, a)) - 1
    back = len(mission._bfs(a, goals=set(mission.world.refuges))) - 1
    from ares.mission import PLAN_SPEED
    mission.battery = (out + back) * mission.cell_cost(PLAN_SPEED) + .01   # enough for a alone
    mission.set_destination(list(a))
    with pytest.raises(ValueError, match='retorno seguro'):
        mission.set_destination(list(b))
    assert mission.destinations == [a]


def test_same_seed_gives_the_same_tour():
    def tour():
        mission = started()
        for cell in spread_cells(mission, 4):
            mission.set_destination(list(cell))
        return list(mission.destinations), mission.run()['distance']
    assert tour() == tour()


def test_panel_api_adds_toggles_and_reports_the_tour():
    session = Session(seed=42)
    session.dispatch('step', {})
    a, b = (list(c) for c in spread_cells(session.mission, 2))
    session.dispatch('destination', {'position': a})
    snapshot = session.dispatch('destination', {'position': b})
    assert sorted(snapshot['map']['destinations']) == sorted([a, b])
    assert snapshot['map']['destination'] == snapshot['map']['destinations'][0]
    assert snapshot['map']['tour'][0] == snapshot['map']['destinations'][0]
    assert snapshot['map']['tour'][-1] == snapshot['map']['destinations'][-1]
    assert snapshot['status']['destinations_left'] == 2 and snapshot['status']['tour_cells'] > 0
    snapshot = session.dispatch('destination', {'position': a})          # same cell again removes it
    assert snapshot['map']['destinations'] == [b]
    assert session.dispatch('destination', {'position': None})['map']['destinations'] == []


In [ ]:
%%writefile ares_x/tests/test_v12_drone.py
"""v1.2 F16: scout drone that flies ahead and shares hazards with the rover."""
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

from ares.api import Session  # noqa: E402
from ares.mission import DRONE_AHEAD, DRONE_ENERGY, HybridMission, validate_config  # noqa: E402


def started(steps=2, **config):
    config.setdefault('seed', 42)
    mission = HybridMission(**config).prepare()
    for _ in range(steps):
        mission.step()
    return mission


def test_drone_is_off_by_default_and_leaves_the_trace_untouched():
    assert validate_config({})['drone'] is False
    plain, explicit = started(steps=0), started(steps=0, drone=False)
    timing = ('decision_ms', 'planning_s')          # wall-clock, not part of the trace
    a, b = plain.run(), explicit.run()
    assert {k: v for k, v in a.items() if k not in timing} == {k: v for k, v in b.items() if k not in timing}
    assert plain.drone is None and plain.metrics['drone_discoveries'] == 0


def test_drone_finds_a_hidden_cliff_on_the_route_before_the_edge_sensor():
    mission = started(drone=True)
    cliff = tuple(mission.inject('barranco')['position'])
    assert cliff not in mission.belief.cliffs            # still hidden from the rover
    record = mission.step()
    assert cliff in mission.belief.cliffs and cliff not in [tuple(c) for c in mission.route]
    assert record['reason'] != 'R04_VETO_BARRANCO_CONFIRMADO'
    assert mission.metrics['drone_discoveries'] >= 1
    found = [e for e in mission.event_log if e['kind'] == 'dron']
    assert found and list(cliff) in found[-1]['cells']


def test_drone_flies_ahead_on_the_route_and_its_energy_comes_from_the_rover():
    mission = started(drone=True, steps=3)
    assert mission.drone['flying'] and mission.drone['reason'] is None
    assert len(mission.drone['scanned']) >= 3
    assert mission.metrics['drone_steps'] == 3
    assert mission.metrics['drone_energy'] == pytest.approx(3 * DRONE_ENERGY)
    assert DRONE_AHEAD >= 2


@pytest.mark.parametrize('event', ['tormenta_polvo', 'viento'])
def test_drone_stays_grounded_in_bad_weather(event):
    mission = started(drone=True)
    flown = mission.metrics['drone_steps']
    mission.inject(event)
    mission.step()
    assert not mission.drone['flying'] and mission.drone['reason']
    assert mission.metrics['drone_steps'] == flown


def test_drone_is_grounded_below_the_battery_reserve_and_absent_in_probe_mode():
    mission = started(drone=True)
    mission.battery = 10.
    mission.step()
    assert not mission.drone['flying'] and 'reserva' in mission.drone['reason']
    probe = started(drone=True, planet='Júpiter', mode='sonda')
    assert not probe.drone['flying'] and probe.metrics['drone_steps'] == 0


def test_drone_never_costs_safety_and_helps_over_several_seeds():
    discoveries = 0
    for seed in range(6):
        summary = HybridMission(seed=seed, drone=True).prepare().run()
        assert summary['safety_violations'] == 0 and summary['collisions'] == 0
        discoveries += summary['drone_discoveries']
    assert discoveries > 0


def test_drone_through_the_panel_api():
    session = Session(seed=42)
    assert session.dispatch('state')['drone'] is None
    snapshot = session.dispatch('configure', {'drone': True})
    assert snapshot['config']['drone'] is True
    snapshot = session.dispatch('step', {})
    assert snapshot['drone']['flying'] and len(snapshot['drone']['position']) == 2
    assert 'error' in session.dispatch('configure', {'drone': 'quizás'})


# ---- toggling the drone during a mission (no restart) ----

def test_drone_can_be_turned_on_mid_mission_without_restarting():
    mission = started(steps=4)
    step, position, path = mission.step_index, mission.position, list(mission.path)
    entry = mission.set_drone(True)
    assert (mission.step_index, mission.position, mission.path) == (step, position, path)
    assert mission.config['drone'] is True and entry['kind'] == 'dron_activado' and entry['step'] == step
    assert mission.metrics['drone_steps'] == 0
    mission.step()
    assert mission.drone['flying'] and mission.metrics['drone_steps'] == 1


def test_drone_can_be_turned_off_mid_mission_and_stops_flying():
    mission = started(steps=2, drone=True)
    flown = mission.metrics['drone_steps']
    entry = mission.set_drone(False)
    assert entry['kind'] == 'dron_desactivado' and mission.drone is None and mission.config['drone'] is False
    mission.step()
    assert mission.metrics['drone_steps'] == flown and mission.drone is None


def test_the_toggle_is_recorded_in_the_export():
    mission = started(steps=3)
    mission.set_drone(True)
    exported = mission.export_json()
    assert any(e['kind'] == 'dron_activado' and e['step'] == 3 for e in exported['events'])


def test_toggle_rejects_same_state_finished_mission_and_bad_values():
    mission = started()
    with pytest.raises(ValueError, match='ya está apagado'):
        mission.set_drone(False)
    with pytest.raises(ValueError, match='verdadero o falso'):
        mission.set_drone('sí')
    mission.run()
    with pytest.raises(ValueError, match='terminó'):
        mission.set_drone(True)


def test_drone_toggle_through_the_api_keeps_the_mission_and_reaches_the_ghosts():
    session = Session(seed=42)
    session.dispatch('ghosts', {'enabled': True})
    before = session.dispatch('step', {'n': 3})
    snapshot = session.dispatch('drone', {'enabled': True})
    assert snapshot['status']['step'] == before['status']['step'] == 3
    assert snapshot['map']['path'] == before['map']['path'] and snapshot['config']['drone'] is True
    assert all(g.config['drone'] for g in session.ghosts.values())
    assert session.dispatch('reset')['config']['drone'] is True        # the choice survives a restart
    assert session.dispatch('drone', {'enabled': False})['config']['drone'] is False
    assert 'error' in session.dispatch('drone', {'enabled': False})


In [ ]:
%%writefile ares_x/tests/test_v12_ghosts.py
"""v1.2 F14: ghost race, the other policies run the same map and seed alongside."""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

from ares.api import Session  # noqa: E402


def test_snapshot_feeds_new_events_to_the_alert_strip():
    session = Session(seed=42)
    before = session.dispatch('step', {})['event_feed']['total']
    feed = session.dispatch('inject', {'kind': 'tormenta_polvo'})['event_feed']
    assert feed['total'] == before + 1
    assert feed['recent'][-1]['kind'] == 'tormenta_polvo' and feed['recent'][-1]['label'] and feed['recent'][-1]['message']


def test_ghost_mode_is_off_by_default():
    assert Session(seed=42).dispatch('state')['ghosts'] == []


def test_enabling_ghosts_restarts_the_mission_with_the_three_ablations():
    session = Session(seed=42)
    session.dispatch('step', {'n': 3})
    snapshot = session.dispatch('ghosts', {'enabled': True})
    assert snapshot['status']['step'] == 0
    assert [g['policy'] for g in snapshot['ghosts']] == ['sin_rna', 'sin_difuso', 'sin_ag']
    assert all(g['position'] == snapshot['status']['position'] and g['step'] == 0 for g in snapshot['ghosts'])
    assert all(g['label'] and g['path'] for g in snapshot['ghosts'])


def test_ghosts_advance_with_the_main_rover_on_the_same_world():
    session = Session(seed=42)
    session.dispatch('ghosts', {'enabled': True})
    snapshot = session.dispatch('step', {'n': 5})
    assert all(g['step'] == snapshot['status']['step'] == 5 for g in snapshot['ghosts'])
    main = session.mission
    for ghost in session.ghosts.values():
        assert ghost.world.obstacles == main.world.obstacles and ghost.seed == main.seed


def test_the_main_policy_is_replaced_by_the_hybrid_among_the_ghosts():
    session = Session(seed=42, policy='sin_ag')
    snapshot = session.dispatch('ghosts', {'enabled': True})
    assert [g['policy'] for g in snapshot['ghosts']] == ['hibrido', 'sin_rna', 'sin_difuso']


def test_events_and_terrain_edits_reach_every_ghost():
    session = Session(seed=42)
    session.dispatch('ghosts', {'enabled': True})
    session.dispatch('step', {})
    session.dispatch('inject', {'kind': 'tormenta_polvo'})
    session.dispatch('place', {'position': [6, 6], 'kind': 'obstaculo'})
    for ghost in session.ghosts.values():
        assert ghost.timers['storm'] > 0 and (6, 6) in ghost.world.obstacles


def test_ghosts_survive_reset_and_configure_and_can_be_turned_off():
    session = Session(seed=42)
    session.dispatch('ghosts', {'enabled': True})
    session.dispatch('step', {'n': 2})
    assert all(g['step'] == 0 for g in session.dispatch('reset')['ghosts'])
    snapshot = session.dispatch('configure', {'seed': 7})
    assert len(snapshot['ghosts']) == 3 and all(g.seed == 7 for g in session.ghosts.values())
    assert session.dispatch('ghosts', {'enabled': False})['ghosts'] == []


def test_the_race_reports_a_result_for_every_runner_when_it_ends():
    session = Session(seed=42)
    session.dispatch('ghosts', {'enabled': True})
    for _ in range(8):
        snapshot = session.dispatch('step', {'n': 50})
    assert snapshot['status']['finished'] and all(g['finished'] and g['outcome'] for g in snapshot['ghosts'])
    assert all({'battery', 'science', 'distance', 'collisions', 'state'} <= set(g) for g in snapshot['ghosts'])


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'])
result = subprocess.run([sys.executable, '-m', 'pytest', '-q', '--color=no', '-p', 'no:cacheprovider', 'ares_x/tests'], capture_output=True, text=True)
print(result.stdout[-2500:] or result.stderr[-2500:])

## 8. Conclusiones, límites y supuestos
- **Integración causal:** las pruebas de causalidad muestran que alterar solo la salida de la RNA, del difuso o del AG cambia la decisión o el
  movimiento; el supervisor nunca decide con una "función secreta".
- **Seguridad primero:** prohibiciones físicas (energía, sensores, barranco confirmado) preceden a las propuestas de IA; cada decisión tiene
  un código de motivo reproducible (R00–R15).
- **No omnisciencia:** el rover planifica sobre un mapa creído; barrancos y parte de los obstáculos están ocultos y se descubren con sensores.
- **Límites:** física simplificada en cuadrícula 16×12; consumos, temperaturas internas, umbrales y daños son *supuestos de simulación*,
  no especificaciones de ingeniería. La señal biológica es anómala y simulada: nunca confirma vida.
- **Gigantes gaseosos:** Júpiter, Saturno, Urano y Neptuno bloquean el modo ruedas y proponen la sonda atmosférica virtual.